# SolarCrow 10 — final competition notebook

A consolidation of **SolarCrow 9's architecture** and the measured November 7 experiments. This is a complete top-to-bottom notebook, not the research sweep notebook.

## The final decision

- **Load** the known ResNet34 coarse and EfficientNet-B3 fine checkpoints. No neural training by default.
- Fit or reuse the **eight-feature, unweighted best-annotator hurdle scorer**, historically selected at **tau=0.325**. This is *not* the annotator-consensus hurdle discussed in research proposals.
- Four-view mean for ordinary fine crops; **top 20% uncertain primary crops get eight-view midpoint median**, including the measured float16 rounding.
- Compare a short final shortlist on the existing development split: N1, N4, residual rescue, and **N4+rescue**. The last combination is a deliberate new challenger, not a previously measured improvement. Highest measured PQ wins; no large architecture or parameter search.
- The selected recipe, scorer and tau flow directly into **the same test-generation path**. The research export accidentally used the old scorer/recipe; that handoff is repaired here.
- No antichain DP, high-seed hysteresis, radial correction or new loss is enabled: their measured trade-offs do not justify replacing the final recipe now.

**Historical evidence, not promised new results:** November 7 G0 PQ 0.430837; N1 hurdle 0.442570; N4 midpoint 0.444568. Public LB 0.35 belongs to a separate artifact/evaluation. This newly assembled notebook has not been run on Kaggle. Exact historical score/CSV reproduction is not claimed.

## Execution and patterns

| Chapter | Purpose | Default work |
|---|---|---|
| 1–4 | Settings, environment, split, metric and geometry | Lightweight setup; reuse saved physical split |
| 5 | EDA/annotator audit | Off |
| 6–7 | Factory + Template Method model lifecycle; cached coarse Proxy | Strict checkpoint loading; optional training only by explicit mode |
| 8 | Candidate Builder, TTA/selection Strategies, sparse value objects, Pipeline Facade | Shared inference and final recipe orchestration |
| 9 | Scorer Builder/Provider and plain-data persistence | Reuse matching bundle; otherwise CPU scorer fit on training-side candidates |
| 10 | Validation shortlist, shared metric, explicit recipe selection | One shared primary pass, bounded extra crops |
| 11–12 | Submission Builder/export, serialized audit, provenance manifest | Test uses selected recipe; audit the saved CSV |
| Appendix | Existing SolarCrow 9 diagnostics | Off; enable selectively |

Progress callbacks are Observers; settings/data records are immutable where practical. Input/model ownership remains explicit—no later hydration cell replaces a loaded model.

### Before Save & Run All

Attach the competition dataset and `fanzzum/solarcrow8d` checkpoint dataset. Enable a T4 GPU (two GPUs are used through DataParallel when available). Internet is needed only if missing packages must be installed; attached wheels are supported. Edit the one Settings cell or use `FILAMENT_OVERRIDES`.

Every execution creates a **new run directory** below `/kaggle/working/solarcrow10_runs/`. Its `submission.csv` is the file to submit. Old checkpoints, CSVs, splits and runs are not overwritten. Attach a previous run as input to reuse its versioned scorer without regenerating the training bank.

The 141-image validation split has been used repeatedly, and this shortlist again uses it for selection. Report resulting scores as **development/validation-selected**, not unbiased OOF or a hidden-test forecast. Unknown external checkpoint training membership stays unknown.

In [ ]:
# ============================================================
# 1. SETUP — bootstrap (runs BEFORE torch is imported)
# ============================================================
import glob
import importlib
import os
import subprocess
import sys

# Required by torch.use_deterministic_algorithms() on CUDA >= 10.2; must exist before CUDA initialises.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", "0")

INPUT_ROOT = os.environ.get("FILAMENT_INPUT_DIR", "/kaggle/input")


def ensure_package(import_name, pip_name, offline_wheel_glob):
    """Import a package, installing it online (pip) or from an attached wheel dataset if missing.

    Replaces the `!pip install` shell-magic, which only works inside IPython and fails when the
    kernel has no internet; this version tries every available source and fails with a clear message.
    """
    try:
        importlib.import_module(import_name)
        return
    except ImportError:
        pass

    attempts = [[sys.executable, "-m", "pip", "install", "-q", pip_name]]
    wheel_dirs = sorted({os.path.dirname(p) for p in glob.glob(offline_wheel_glob, recursive=True)})
    attempts += [
        [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", d, pip_name]
        for d in wheel_dirs
    ]
    last_error = ""
    for command in attempts:
        result = subprocess.run(command, capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            importlib.invalidate_caches()
            try:
                importlib.import_module(import_name)
                return
            except ImportError as exc:
                last_error = str(exc)
        else:
            last_error = (result.stderr or result.stdout)[-400:]
    raise RuntimeError(
        f"Could not import or install {pip_name!r}. Enable internet for this notebook, or attach a "
        f"dataset containing the wheel (searched: {offline_wheel_glob}). Last error: {last_error}"
    )


ensure_package(
    "segmentation_models_pytorch",
    "segmentation-models-pytorch==0.6.0",
    os.path.join(INPUT_ROOT, "**", "segmentation_models_pytorch*.whl"),
)
ensure_package("pycocotools", "pycocotools", os.path.join(INPUT_ROOT, "**", "pycocotools*.whl"))
print("Bootstrap complete; existing installed packages are not forcibly upgraded.")

In [ ]:
# All imports in one place.
import abc
import contextlib
import dataclasses
import fnmatch
import functools
import gc
import hashlib
import itertools
import json
import math
import random
import re
import shutil
import statistics
import time
import traceback
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import cv2
import joblib
import matplotlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Headless-safe plotting: inline backend inside a kernel, Agg for plain `python` / nbconvert-less runs.
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Polygon

import segmentation_models_pytorch as smp
from pycocotools import mask as mask_util
from scipy.optimize import linear_sum_assignment
from scipy.stats import median_abs_deviation
from sklearn.ensemble import HistGradientBoostingRegressor

try:
    from IPython.display import display
except ImportError:  # plain-python fallback
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# ============================================================
# 1.1 EXPERIMENT SETTINGS  (single immutable source of truth)
# ============================================================
# Optional headless parameter injection (papermill-style): set FILAMENT_OVERRIDES='{"fine_epochs": 3}'.

_DEFAULT_TAU_GRID = (0.0,) + tuple(round(0.10 + 0.02 * i, 2) for i in range(31))   # 0.00, 0.10 … 0.70


@dataclass(frozen=True)
class Settings:
    # --- identity & lifecycle ------------------------------------------------------------
    experiment_id: str = "solarcrow10_final_repro"
    coarse_mode: str = "load"            # "train" | "load"
    fine_mode: str = "load"             # "train" | "load"
    split_mode: str = "reuse_if_available"   # "reuse_if_available" | "new"
    scorer_mode: str = "auto"            # "auto"   : reuse only a scorer whose fingerprint matches the active models, else re-fit
                                         # "retrain": always re-fit   |  "load": trust the file found (explicit override)
    allow_unverified_scorer: bool = False  # accept a legacy scorer with no fingerprint (only when fine_mode == "load")
    seed: int = 42
    coarse_epochs: int = 15
    fine_epochs: int = 20

    # --- external checkpoints (explicit hints; InputLocator also scans /kaggle/input/**) ------
    coarse_load_path: str = "/kaggle/input/datasets/fanzzum/solarcrow8d/coarse_model_3.pth"
    fine_load_path: str = "/kaggle/input/datasets/fanzzum/solarcrow8d/EXP_4_EfficientNetB3_BCE_Dice.pth"
    scorer_load_paths: Tuple[str, ...] = (
        "/kaggle/input/datasets/fanzzum/solarcrow10/candidate_scorer_solarcrow10.pkl",
        "/kaggle/input/solarcrow10/candidate_scorer_solarcrow10.pkl",
    )

    # --- final recipe: one explicit configuration for validation AND submission ----------
    coarse_encoder: str = "resnet34"
    fine_encoder: str = "efficientnet-b3"
    use_data_parallel: bool = True
    coarse_tta_chunk: int = 1   # matches the measured per-view coarse evaluation
    split_load_path: str = "/kaggle/input/datasets/fanzzum/solarcrow8d/results8/split.json"
    allow_new_split: bool = True
    width_buckets: Tuple[int, ...] = (160, 288, 576, 928)
    height_buckets: Tuple[int, ...] = (160, 256, 448, 800)
    min_candidate_area: int = 20   # generation floor; final cleanup remains 75
    fine_base_tta: str = "4view"
    final_recipe: str = "n4_midpoint"
    select_recipe_on_validation: bool = True
    enable_residual_challenger: bool = False
    routed_fraction: float = 0.20
    routed_aggregation: str = "midpoint_median"
    routed_probability_dtype: str = "float16"
    residual_threshold: float = 0.22
    residual_iou_limit: float = 0.30
    residual_max_added: int = 4
    scorer_target: str = "hurdle_max"  # measured best-annotator hurdle, not consensus
    scorer_rasterizer: str = "opencv_round"  # measured bank; "coco" is a separate opt-in condition
    run_synthetic_tests: bool = True
    run_coarse_tta_self_test: bool = True

    # --- geometry --------------------------------------------------------------------------
    image_height: int = 2048
    image_width: int = 2048
    coarse_resize: int = 1024
    expansion: int = 32
    dilation_size: int = 64
    bucket_percentiles: Tuple[int, ...] = (50, 90, 99, 100)

    # --- training ----------------------------------------------------------------------------
    coarse_batch_size: int = 4
    fine_batch_size: int = 8
    coarse_pos_weight: float = 5.0
    num_workers: int = 2

    # --- inference knobs ---------------------------------------------------------------------
    coarse_threshold: float = 0.3
    coarse_min_area: int = 20
    coarse_closing_kernel: int = 45
    coarse_use_tta: bool = True          # 4-view batched TTA
    fine_threshold: float = 0.5
    min_prediction_area: int = 75
    fine_infer_batch_size: int = 4
    empty_cache_every: int = 40          # release cached GPU blocks every N images

    # --- candidate scorer / tau ----------------------------------------------------------------
    use_scorer: bool = True
    default_tau: float = 0.25            # measured hurdle operating point; optional sweep overrides it
    auto_tune_tau: bool = False
    tau_grid: Tuple[float, ...] = (0.20, 0.225, 0.25, 0.275, 0.30, 0.325, 0.35)
    tau_refine_step: float = 0.0       # 0 disables the local refinement pass
    tau_smoothing_window: int = 1        # 1 = plain arg-max; 3/5 = arg-max of a moving average (less noisy)
    scorer_max_train_images: Optional[int] = None   # None = all training physical images

    # --- optional work (all failure-isolated) --------------------------------------------------
    run_eda: bool = True
    run_annotator_ceiling: bool = True
    run_fine_input_inspection: bool = True
    run_validation_report: bool = True
    run_diagnostics: bool = True         # appendix: coarse-only, Q4 audits, autopsy, metric audit …
    run_q4_diagnostics: bool = True
    preview_test_images: int = 10
    q4_raster_annotation_sample: int = 250
    q4_disk_image_sample: int = 20
    q4_support_image_sample: int = 15
    autopsy_random_samples: int = 3
    autopsy_max_worst: int = 5
    optional_budget_hours: float = 8.0   # optional stages are skipped once this wall-clock budget is used

    def __post_init__(self):
        for name in ("coarse_mode", "fine_mode"):
            if getattr(self, name) not in {"train", "load", "fine_tune"}:
                raise ValueError(f"{name} must be train, load or fine_tune")
        if self.split_mode not in {"reuse_if_available", "new"}:
            raise ValueError("split_mode must be 'reuse_if_available' or 'new'")
        if self.scorer_mode not in {"auto", "retrain", "load"}:
            raise ValueError("scorer_mode must be 'auto', 'retrain' or 'load'")
        if self.tau_smoothing_window < 1 or self.tau_smoothing_window % 2 == 0:
            raise ValueError("tau_smoothing_window must be a positive odd integer")

        if self.final_recipe not in {"n1_standard", "n4_midpoint", "n3_rescue", "n4_rescue"}:
            raise ValueError("Unknown final_recipe")
        if self.scorer_target not in {"hurdle_max", "mean_eligible", "max_iou"}:
            raise ValueError("Unknown scorer_target")
        if self.scorer_rasterizer not in {"opencv_round", "coco"}:
            raise ValueError("Unknown scorer_rasterizer")
        if not 0 <= self.routed_fraction <= 1 or self.coarse_tta_chunk not in (1, 2, 4):
            raise ValueError("Invalid routing fraction or coarse TTA chunk")
        if min(self.fine_infer_batch_size, self.min_candidate_area, self.min_prediction_area) < 1:
            raise ValueError("Batch sizes and area floors must be positive")
        for buckets in (self.width_buckets, self.height_buckets):
            if not buckets or list(buckets) != sorted(set(buckets)) or any(v % 32 for v in buckets):
                raise ValueError("Fixed buckets must be sorted positive multiples of 32")

    @property
    def image_shape(self):
        return (self.image_height, self.image_width)

    @classmethod
    def from_environment(cls):
        """Factory Method: defaults + optional JSON overrides from $FILAMENT_OVERRIDES."""
        base = cls()
        raw = os.environ.get("FILAMENT_OVERRIDES", "").strip()
        if not raw:
            return base
        overrides = json.loads(raw)
        unknown = set(overrides) - {f.name for f in dataclasses.fields(cls)}
        if unknown:
            raise ValueError(f"Unknown FILAMENT_OVERRIDES keys: {sorted(unknown)}")
        for key, value in list(overrides.items()):
            if isinstance(getattr(base, key), tuple) and isinstance(value, list):
                overrides[key] = tuple(value)
        return dataclasses.replace(base, **overrides)


settings = Settings.from_environment()
print(f"Experiment: {settings.experiment_id} | coarse={settings.coarse_mode} fine={settings.fine_mode} "
      f"scorer={settings.scorer_mode} | auto_tune_tau={settings.auto_tune_tau}")

In [ ]:
# ============================================================
# 1.2 ENVIRONMENT — writable workspace, read-only input locator, device
# ============================================================
class Workspace:
    """Facade over the ONLY writable location (/kaggle/working).

    Every output (checkpoints, scorers, CSVs, plots, manifests) goes through this class, which
    (a) refuses any path outside its root, so a stray write can never hit the read-only
    /kaggle/input mount, and (b) writes atomically (tmp file + os.replace).
    """

    def __init__(self, root):
        self.root = Path(root).expanduser().resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        probe = self.root / ".write_probe"
        try:
            probe.write_text("ok")
            probe.unlink()
        except OSError as exc:
            raise PermissionError(f"Workspace {self.root} is not writable: {exc}") from exc

    @classmethod
    def discover(cls):
        """Factory Method: $FILAMENT_WORK_DIR > /kaggle/working > ./outputs."""
        override = os.environ.get("FILAMENT_WORK_DIR")
        import tempfile
        base = Path(override) if override else (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("./outputs"))
        runs = base.expanduser().resolve() / "solarcrow10_runs"
        runs.mkdir(parents=True, exist_ok=True)
        safe_id = re.sub(r"[^A-Za-z0-9_.-]", "_", settings.experiment_id)
        instance = cls(tempfile.mkdtemp(prefix=safe_id + "__", dir=str(runs)))
        instance.input_search_root = str(runs)
        return instance

    def path(self, *parts):
        target = self.root.joinpath(*parts).resolve()
        if target != self.root and self.root not in target.parents:
            raise ValueError(f"Refusing to write outside the workspace: {target}")
        target.parent.mkdir(parents=True, exist_ok=True)
        return str(target)

    def unique_path(self, *parts):
        """Like path(), but never reuses an existing file: appends __01, __02, …"""
        path = self.path(*parts)
        if not os.path.exists(path):
            return path
        stem, suffix = os.path.splitext(path)
        index = 1
        while os.path.exists(f"{stem}__{index:02d}{suffix}"):
            index += 1
        return f"{stem}__{index:02d}{suffix}"

    def _atomic(self, final_path, writer):
        tmp_path = final_path + ".tmp"
        try:
            writer(tmp_path)
            os.replace(tmp_path, final_path)
        finally:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
        return final_path

    def write_csv(self, frame, *parts, index=False):
        return self._atomic(self.path(*parts), lambda p: frame.to_csv(p, index=index))

    def write_pickle(self, frame, *parts):
        return self._atomic(self.path(*parts), lambda p: frame.to_pickle(p))

    def write_json(self, obj, *parts, unique=False):
        final = self.unique_path(*parts) if unique else self.path(*parts)

        def _write(p):
            with open(p, "w", encoding="utf-8") as fh:
                json.dump(obj, fh, indent=2, default=str)
        return self._atomic(final, _write)

    def save_state_dict(self, model, *parts):
        """Checkpoints are never overwritten: a numbered sibling is created when the name exists."""
        final = self.unique_path(*parts)
        return self._atomic(final, lambda p: torch.save(model.state_dict(), p))

    def dump_joblib(self, obj, *parts):
        return self._atomic(self.path(*parts), lambda p: joblib.dump(obj, p))

    def save_figure(self, fig, *parts):
        return self._atomic(self.path(*parts), lambda p: fig.savefig(p, dpi=110, bbox_inches="tight", format="png"))


class InputLocator:
    """Resolves read-only inputs across every plausible /kaggle/input layout (never writes)."""

    SKIP_DIRS = {"train_images", "test_images", "__pycache__", ".git"}
    NUMBERED = re.compile(r"__(\d+)(?=\.[^.]+$)")

    def __init__(self, roots, max_depth=7):
        self.roots = [Path(r) for r in roots if r and Path(r).is_dir()]
        self.max_depth = max_depth

    def _directories(self):
        for root in self.roots:
            stack = [(root, 0)]
            while stack:
                current, depth = stack.pop()
                yield current
                if depth >= self.max_depth:
                    continue
                try:
                    children = sorted(p for p in current.iterdir() if p.is_dir() and p.name not in self.SKIP_DIRS)
                except OSError:
                    continue
                stack.extend((child, depth + 1) for child in reversed(children))

    @classmethod
    def _order_key(cls, path):
        match = cls.NUMBERED.search(str(path))
        return (int(match.group(1)) if match else -1, str(path))

    def find_files(self, patterns):
        """All files whose basename matches any glob pattern; highest `__NN` checkpoint suffix first."""
        found = []
        for directory in self._directories():
            try:
                names = os.listdir(directory)
            except OSError:
                continue
            for name in names:
                if any(fnmatch.fnmatch(name, pattern) for pattern in patterns):
                    candidate = directory / name
                    if candidate.is_file():
                        found.append(candidate)
        return sorted(set(found), key=self._order_key, reverse=True)

    def resolve_file(self, explicit_paths=(), patterns=()):
        """First existing explicit path, else the best pattern match. Returns str or None."""
        for path in explicit_paths:
            if path and os.path.isfile(path):
                return os.path.abspath(path)
        matches = self.find_files(patterns) if patterns else []
        return str(matches[0]) if matches else None

    def find_dataset_root(self, hints, marker_dirname="MAGFiLO_1.0_Kaggle_2026"):
        """Directory that contains train/<annotation json> — explicit hints first, then a bounded scan."""
        def is_root(path):
            train_dir = Path(path) / "train"
            return train_dir.is_dir() and any(train_dir.glob("*Annotations*.json"))

        for hint in hints:
            if hint and is_root(hint):
                return str(Path(hint))
        for directory in self._directories():
            if directory.name == marker_dirname and is_root(directory):
                return str(directory)
        for directory in self._directories():
            if is_root(directory):
                return str(directory)
        return None


@dataclass(frozen=True)
class DatasetPaths:
    root: str
    train_images: str
    test_images: str
    annotations: str


def resolve_dataset_paths(locator):
    hints = [
        os.path.join(INPUT_ROOT, "competitions", "filament-segmentation-2026", "MAGFiLO_1.0_Kaggle_2026"),
        os.path.join(INPUT_ROOT, "filament-segmentation-2026", "MAGFiLO_1.0_Kaggle_2026"),
        "./MAGFiLO_1.0_Kaggle_2026",
        ".",
    ]
    root = locator.find_dataset_root(hints)
    if root is None:
        listing = sorted(os.listdir(INPUT_ROOT))[:20] if os.path.isdir(INPUT_ROOT) else "(input root missing)"
        raise FileNotFoundError(
            "Could not locate the competition data (a folder containing train/*Annotations*.json). "
            f"Searched hints {hints} and scanned {INPUT_ROOT}. Top-level input entries: {listing}"
        )
    annotation_files = sorted(Path(root, "train").glob("*Annotations*.json"))
    preferred = [p for p in annotation_files if p.name == "MAGFiLO_1.0_Annotations_kaggle2026_train.json"]
    return DatasetPaths(
        root=root,
        train_images=os.path.join(root, "train", "train_images"),
        test_images=os.path.join(root, "test", "test_images"),
        annotations=str((preferred or annotation_files)[0]),
    )


def select_device():
    """CUDA (Kaggle GPU) -> Apple MPS -> CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


workspace = Workspace.discover()
locator = InputLocator([INPUT_ROOT, workspace.input_search_root])
dataset_paths = resolve_dataset_paths(locator)
device = select_device()

print(f"Active device    : {device}")
print(f"Dataset root     : {dataset_paths.root}")
print(f"Annotations      : {dataset_paths.annotations}")
print(f"Writable workspace: {workspace.root}")

In [ ]:
# ============================================================
# 1.3 RUNTIME UTILITIES — determinism, logging, failure isolation, progress
# ============================================================
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff")


def seed_everything(seed):
    """Re-seed every RNG. Called before each stochastic stage so results never depend on which
    exploratory cells ran earlier (the old notebook consumed the global RNG in EDA cells)."""
    random.seed(seed)
    np.random.seed(seed % (2 ** 32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_determinism():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:  # very old torch
        pass


def make_generator(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def seed_worker(_worker_id):
    """DataLoader worker seeding: derived from the loader generator, hence reproducible."""
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def sha256_file(path, chunk_size=1024 * 1024):
    if not path or not os.path.isfile(path):
        return None
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def list_images(directory):
    if not os.path.isdir(directory):
        raise FileNotFoundError(f"Image directory missing: {directory}")
    return sorted(f for f in os.listdir(directory) if f.lower().endswith(IMAGE_EXTENSIONS))


class RunLog:
    """Collects stage timings and failures of optional stages for the final manifest."""

    def __init__(self, optional_budget_hours):
        self.start = time.time()
        self.stage_seconds = {}
        self.optional_failures = {}
        self.optional_skipped = []
        self.optional_budget_seconds = optional_budget_hours * 3600.0

    @contextlib.contextmanager
    def stage(self, name):
        began = time.time()
        print(f"\n▶ {name}")
        try:
            yield
        finally:
            self.stage_seconds[name] = round(time.time() - began, 1)
            print(f"■ {name} finished in {self.stage_seconds[name]:.1f}s")

    def elapsed(self):
        return time.time() - self.start

    def within_budget(self):
        return self.elapsed() < self.optional_budget_seconds


configure_determinism()
seed_everything(settings.seed)
run_log = RunLog(settings.optional_budget_hours)


def optional_stage(name):
    """Decorator: run a non-critical stage (EDA, plots, diagnostics) in isolation.

    A failure is logged with a traceback and recorded, but never aborts the commit — the critical
    path (training → scoring → validation → submission) keeps its fail-fast behaviour.
    """
    def decorator(function):
        @functools.wraps(function)
        def wrapper(*args, **kwargs):
            if not run_log.within_budget():
                run_log.optional_skipped.append(name)
                print(f"[optional stage '{name}' skipped: wall-clock budget used up]")
                return None
            began = time.time()
            try:
                return function(*args, **kwargs)
            except Exception as exc:  # noqa: BLE001 - deliberate isolation boundary
                run_log.optional_failures[name] = f"{type(exc).__name__}: {exc}"
                print(f"[optional stage '{name}' failed and was skipped] {type(exc).__name__}: {exc}")
                traceback.print_exc(limit=4)
                return None
            finally:
                run_log.stage_seconds[f"optional:{name}"] = round(time.time() - began, 1)
                plt.close("all")
        return wrapper
    return decorator


class ProgressReporter:
    """Observer-style progress callback used by every long loop."""

    def __init__(self, label, total, every=25):
        self.label, self.total, self.every, self.count = label, total, every, 0
        self.began = time.time()

    def step(self, extra=""):
        self.count += 1
        if self.count % self.every == 0 or self.count == self.total:
            print(f"{self.label}: {self.count}/{self.total} | {time.time() - self.began:.0f}s {extra}".rstrip(), flush=True)


def show_figure(fig=None):
    """Display and always close the figure (prevents figure accumulation in long headless runs)."""
    plt.show()
    plt.close(fig if fig is not None else "all")

## 2. Strict one-to-one PQ metric and synthetic checks

Every headline and detailed evaluator now uses the same eligible-edge matching, **IoU strictly >0.5**. This replaces SolarCrow 9's headline all-hit shortcut. Empty records remain in evaluation. Synthetic checks run on small arrays only; official implementation parity is not claimed.

In [ ]:
def threshold_aware_hungarian(iou_matrix, iou_threshold=0.5):
    """Return (row_index, col_index, IoU) matches with IoU strictly > threshold.

    Matching is lexicographic: maximize the number of eligible edges first,
    then maximize the total IoU among matchings with that cardinality.
    Ineligible edges have zero weight and are removed from the returned pairs.
    """
    matrix = np.asarray(iou_matrix, dtype=float)
    if matrix.ndim != 2:
        raise ValueError(f"IoU matrix must be 2-D; got shape {matrix.shape}")
    n_rows, n_cols = matrix.shape
    if n_rows == 0 or n_cols == 0:
        return []
    if not np.isfinite(matrix).all():
        raise ValueError("IoU matrix contains non-finite values")

    eligible = matrix > float(iou_threshold)
    # The bonus exceeds the largest possible total-IoU difference, so one
    # additional eligible match always beats any smaller-cardinality matching.
    cardinality_bonus = min(n_rows, n_cols) + 1.0
    weights = np.where(eligible, cardinality_bonus + matrix, 0.0)
    row_indices, col_indices = linear_sum_assignment(weights, maximize=True)

    return [
        (int(row), int(col), float(matrix[row, col]))
        for row, col in zip(row_indices, col_indices)
        if eligible[row, col]
    ]


def run_metric_helper_self_tests():
    """Fast synthetic tests for strict thresholding and eligible-edge matching."""
    assert len(threshold_aware_hungarian(np.array([[1.0]]), 0.5)) == 1
    assert threshold_aware_hungarian(np.zeros((1, 0)), 0.5) == []
    assert threshold_aware_hungarian(np.zeros((0, 1)), 0.5) == []
    assert threshold_aware_hungarian(np.array([[0.5]]), 0.5) == []  # strict >

    # Counterexample: raw-IoU assignment prefers the two ineligible cross-pairs,
    # then threshold filtering loses the valid 0.5025 match.
    counterexample = np.array([[0.5025, 0.3311], [0.3311, 0.0]])
    matches = threshold_aware_hungarian(counterexample, 0.5)
    assert len(matches) == 1 and matches[0][:2] == (0, 0)
    assert abs(matches[0][2] - 0.5025) < 1e-12

    # Split-like and overlapping-candidate cases remain one-to-one.
    assert len(threshold_aware_hungarian(np.array([[0.8, 0.6]]), 0.5)) == 1
    overlap_case = np.array([[0.8, 0.7], [0.7, 0.0]])
    overlap_matches = threshold_aware_hungarian(overlap_case, 0.5)
    assert len(overlap_matches) == 2
    return "All threshold-aware matching self-tests passed."


def _as_numpy(matrix):
    return matrix.detach().cpu().numpy() if hasattr(matrix, "detach") else np.asarray(matrix)


def pq_counts(overlap_df, iou_threshold=0.5):
    """One source of truth: eligible-edge one-to-one counts for ALL metric callers."""
    total_iou, tp, fp, fn = 0.0, 0, 0, 0
    for row in overlap_df.itertuples(index=False):
        n_gt, n_pred = int(row.n_gt), int(row.n_pred)
        matched = threshold_aware_hungarian(_as_numpy(row.iou_matrix), iou_threshold) if n_gt and n_pred else []
        total_iou += sum(iou for _, _, iou in matched)
        tp += len(matched)
        fp += n_pred - len(matched)
        fn += n_gt - len(matched)
    return total_iou, tp, fp, fn



def get_pq_score(overlap_df, iou_threshold=0.5):
    tp_iou_sum, tp, fp, fn = pq_counts(overlap_df, iou_threshold)
    denominator = tp + 0.5 * fp + 0.5 * fn
    return tp_iou_sum / denominator if denominator > 0 else 0.0


def compute_detailed_metrics(overlap_df, iou_threshold=0.5):
    """PQ decomposed into SQ, RQ, TP, FP, FN, Precision, Recall.

    Returns (pooled_metrics dict, per-station DataFrame, per-physical-image DataFrame).
    """
    rows = []
    for row in overlap_df.itertuples(index=False):
        annotator_img = row.annotator_image
        # physical image ID and 2-letter station code, e.g. '010401-20160920230134Lh' -> 'Lh'
        phys_id = annotator_img.split("-", maxsplit=1)[1].split(".")[0]
        station = phys_id[-2:]
        n_gt, n_pred = int(row.n_gt), int(row.n_pred)

        if n_gt == 0:
            tp_list, fp, fn = [], n_pred, 0
        elif n_pred == 0:
            tp_list, fp, fn = [], 0, n_gt
        else:
            # iou_matrix: ground truths on rows, predictions on columns
            accepted = threshold_aware_hungarian(_as_numpy(row.iou_matrix), iou_threshold=iou_threshold)
            tp_list = [iou for _, _, iou in accepted]
            fp, fn = n_pred - len(tp_list), n_gt - len(tp_list)

        rows.append({
            "annotator_image": annotator_img, "physical_image": phys_id, "station": station,
            "n_gt": n_gt, "n_pred": n_pred, "tp_ious": tp_list, "tp": len(tp_list), "fp": fp, "fn": fn,
        })

    records_df = pd.DataFrame(rows)

    def aggregate(df):
        all_tp_ious = [iou for sublist in df["tp_ious"] for iou in sublist]
        total_tp, total_fp, total_fn = len(all_tp_ious), df["fp"].sum(), df["fn"].sum()
        denom = total_tp + 0.5 * total_fp + 0.5 * total_fn
        return {
            "PQ": (sum(all_tp_ious) / denom) if denom > 0 else 0.0,
            "SQ": (sum(all_tp_ious) / total_tp) if total_tp > 0 else 0.0,
            "RQ": (total_tp / denom) if denom > 0 else 0.0,
            "TP": total_tp, "FP": total_fp, "FN": total_fn,
            "Precision": (total_tp / (total_tp + total_fp)) if (total_tp + total_fp) > 0 else 0.0,
            "Recall": (total_tp / (total_tp + total_fn)) if (total_tp + total_fn) > 0 else 0.0,
        }

    pooled = aggregate(records_df)
    station_df = pd.DataFrame([
        {"Station": st, "Images": group["physical_image"].nunique(), **aggregate(group)}
        for st, group in records_df.groupby("station")
    ]).sort_values("PQ", ascending=False).reset_index(drop=True)
    per_image_df = pd.DataFrame([
        {"physical_image": p_id, "station": group["station"].iloc[0], **aggregate(group)}
        for p_id, group in records_df.groupby("physical_image")
    ]).sort_values("PQ").reset_index(drop=True)
    return pooled, station_df, per_image_df


print(run_metric_helper_self_tests())
if settings.run_synthetic_tests:
    print(run_metric_helper_self_tests())

## 3. Fixed physical-image split

Load the known `results8/split.json`; validate unique/disjoint physical membership and complete dataset coverage. All records of a physical image stay together, including empty records. Missing split files fail clearly unless creation is explicitly enabled. A saved split does not establish externally trained checkpoint membership.

In [ ]:
@dataclass
class AnnotationIndex:
    """Read-only lookup tables over the COCO-style annotation file."""
    raw: dict
    images_by_id: Dict[str, dict]
    annotations_by_id: Dict[int, dict]
    annotation_ids_by_image: Dict[str, List[int]]
    file_to_image_ids: Dict[str, List[str]]

    @classmethod
    def from_file(cls, path):
        with open(path, "r") as handle:
            raw = json.load(handle)
        annotation_ids_by_image = defaultdict(list)
        for annotation in raw["annotations"]:
            annotation_ids_by_image[annotation["image_id"]].append(annotation["id"])
        file_to_image_ids = defaultdict(list)
        for image in raw["images"]:
            file_to_image_ids[image["id"].split("-", 1)[1]].append(image["id"])
        return cls(
            raw=raw,
            images_by_id={image["id"]: image for image in raw["images"]},
            annotations_by_id={annotation["id"]: annotation for annotation in raw["annotations"]},
            annotation_ids_by_image=dict(annotation_ids_by_image),
            file_to_image_ids=dict(file_to_image_ids),
        )

    @property
    def physical_file_ids(self):
        return set(self.file_to_image_ids)

    def file_name_of_physical(self, file_id):
        """File name of the first annotator record (all records of one physical image share the file)."""
        return self.images_by_id[self.file_to_image_ids[file_id][0]]["file_name"]


@dataclass
class DataSplit:
    train_file_ids: List[str]
    validation_file_ids: List[str]
    train_split_dict: Dict[str, List[int]]        # annotator-record id -> annotation ids
    validation_split_dict: Dict[str, List[int]]
    path: str
    sha256: str
    origin: str


class SplitRepository:
    """Loads a compatible saved split or creates (and persists) a deterministic new one."""

    FILENAME = "split.json"

    def __init__(self, settings, workspace, locator):
        self.settings, self.workspace, self.locator = settings, workspace, locator

    @staticmethod
    def _validate(index, train_ids, validation_ids):
        train_set, validation_set = set(train_ids), set(validation_ids)
        if len(train_ids) != len(train_set) or len(validation_ids) != len(validation_set):
            raise ValueError("Saved split contains duplicate physical IDs")
        if train_set & validation_set:
            raise ValueError("Saved split leaks physical images between train and validation.")
        everything = index.physical_file_ids
        if train_set | validation_set != everything:
            raise ValueError(
                "Saved split does not exactly match this dataset. "
                f"Missing={len(everything - (train_set | validation_set))}, "
                f"unknown={len((train_set | validation_set) - everything)}."
            )

    def _try_load(self, path, index, strict):
        try:
            with open(path, "r", encoding="utf-8") as handle:
                saved = json.load(handle)
            train_ids, validation_ids = list(saved["train_file_ids"]), list(saved["validation_file_ids"])
            self._validate(index, train_ids, validation_ids)
            return train_ids, validation_ids
        except (OSError, KeyError, ValueError, json.JSONDecodeError) as exc:
            if strict:
                raise
            print(f"  ignoring incompatible split {path}: {exc}")
            return None

    def obtain(self, index):
        s = self.settings
        train_ids = validation_ids = None
        split_path, origin = None, None

        if s.split_mode == "reuse_if_available":
            explicit = self.locator.resolve_file([s.split_load_path], [Path(s.split_load_path).name])
            if explicit:
                train_ids, validation_ids = self._try_load(explicit, index, strict=True)
                split_path, origin = explicit, "explicit saved split (external checkpoint membership remains unknown)"
            working_split = self.workspace.path(self.FILENAME)
            if train_ids is None and os.path.isfile(working_split):
                train_ids, validation_ids = self._try_load(working_split, index, strict=True)
                split_path, origin = working_split, "working-dir split.json"
            elif train_ids is None:
                for candidate in self.locator.find_files([self.FILENAME]):
                    loaded = self._try_load(str(candidate), index, strict=False)
                    if loaded:
                        train_ids, validation_ids = loaded
                        split_path, origin = str(candidate), f"input dataset ({candidate})"
                        break

        if train_ids is None:
            if not s.allow_new_split:
                raise FileNotFoundError("No compatible saved split. Attach results8/split.json or explicitly set allow_new_split=True; never silently redraw the final split.")
            # Local RNG: stable even when exploratory cells consumed the global RNG.
            shuffled = random.Random(s.seed).sample(sorted(index.physical_file_ids), len(index.physical_file_ids))
            margin = math.ceil(len(shuffled) * 0.8)
            train_ids, validation_ids = shuffled[:margin], shuffled[margin:]
            origin = "newly drawn (seeded)"
            self._validate(index, train_ids, validation_ids)

        payload = {"train_file_ids": train_ids, "validation_file_ids": validation_ids}
        if split_path is None or not str(split_path).startswith(str(self.workspace.root)):
            name = self.FILENAME if s.split_mode == "reuse_if_available" else f"split_{s.experiment_id}.json"
            split_path = self.workspace.write_json(payload, name, unique=True)   # always keep a writable copy

        train_records = [rid for fid in train_ids for rid in index.file_to_image_ids[fid]]
        validation_records = [rid for fid in validation_ids for rid in index.file_to_image_ids[fid]]
        annotations = index.annotation_ids_by_image
        return DataSplit(
            train_file_ids=train_ids,
            validation_file_ids=validation_ids,
            train_split_dict={rid: annotations.get(rid, []) for rid in train_records},
            validation_split_dict={rid: annotations.get(rid, []) for rid in validation_records},
            path=split_path,
            sha256=sha256_file(split_path),
            origin=origin,
        )


with run_log.stage("Data index and split"):
    index = AnnotationIndex.from_file(dataset_paths.annotations)
    split = SplitRepository(settings, workspace, locator).obtain(index)

    first_record = index.raw["images"][0]
    settings = dataclasses.replace(
        settings,
        image_height=int(first_record.get("height", settings.image_height)),
        image_width=int(first_record.get("width", settings.image_width)),
    )

    print(f"Split origin               : {split.origin}")
    print(f"Split file                 : {split.path}")
    print(f"Split SHA-256              : {split.sha256}")
    print(f"Train physical images      : {len(split.train_file_ids)}")
    print(f"Validation physical images : {len(split.validation_file_ids)}")
    print(f"Train annotator records    : {len(split.train_split_dict)}")
    print(f"Validation annotator records: {len(split.validation_split_dict)}")
    print(f"Image geometry (from annotations): {settings.image_shape}")

In [ ]:
# ============================================================
# 3.1 PRE-FLIGHT — fail in seconds (not hours) if the environment cannot complete the run
# ============================================================
@optional_stage("environment banner")
def print_environment_banner():
    print(f"torch {torch.__version__} | smp {smp.__version__} | numpy {np.__version__} | cv2 {cv2.__version__}")
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name} | {props.total_memory / 2**30:.1f} GiB")


def run_preflight(settings, dataset_paths, workspace):
    problems = []
    train_images, test_images = list_images(dataset_paths.train_images), list_images(dataset_paths.test_images)
    if not train_images:
        problems.append(f"no images in {dataset_paths.train_images}")
    if not test_images:
        problems.append(f"no images in {dataset_paths.test_images}")

    for label, directory, names in (("train", dataset_paths.train_images, train_images),
                                    ("test", dataset_paths.test_images, test_images)):
        if names:
            shape = plt.imread(os.path.join(directory, names[0])).shape[:2]
            if tuple(shape) != tuple(settings.image_shape):
                problems.append(f"first {label} image has shape {tuple(shape)} but annotations declare "
                                f"{settings.image_shape}; RLE sizes would be wrong")

    free_gib = shutil.disk_usage(workspace.root).free / 2 ** 30
    if free_gib < 2:
        problems.append(f"only {free_gib:.1f} GiB free in {workspace.root}")
    if problems:
        raise RuntimeError("Pre-flight failed:\n  - " + "\n  - ".join(problems))
    print(f"Pre-flight OK | train images: {len(train_images)} | test images: {len(test_images)} | "
          f"free disk: {free_gib:.1f} GiB | workspace writable: {workspace.root}")
    return train_images, test_images


with run_log.stage("Pre-flight"):
    print_environment_banner()
    train_image_names, test_image_names = run_preflight(settings, dataset_paths, workspace)

## 4. Measured preprocessing, sparse geometry and fixed B3 buckets

The final recipe uses grayscale OpenCV input, largest-component Otsu disk, disk median/unscaled MAD, and native crop geometry from the successful research run. Fixed width buckets are 160/288/576/928 and heights 160/256/448/800. Oversize crops retain the measured center-clipping behavior; changing support now would be another experiment. The retained legacy training polygon rasterizer is not silently changed; scorer targets explicitly select their own recorded rasterizer.

In [ ]:
# ------------------------------------------------------------------
# Image I/O, normalisation and disk mask
# ------------------------------------------------------------------
def _otsu_input(img):
    """cv2's Otsu needs 8/16-bit input. uint8/uint16 pass through unchanged; floats in [0,1] are rescaled."""
    if img.dtype in (np.uint8, np.uint16):
        return img
    return np.clip(np.asarray(img, dtype=np.float64) * 255.0, 0, 255).astype(np.uint8)


def get_disk_mask(img):
    src = _otsu_input(img)
    _, binary = cv2.threshold(src, 0, 255 if src.dtype == np.uint8 else 65535, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, labels, stats, _ = cv2.connectedComponentsWithStats((binary > 0).astype(np.uint8), connectivity=8)
    return labels == (1 + int(stats[1:, cv2.CC_STAT_AREA].argmax())) if n > 1 else np.zeros(img.shape[:2], bool)


def normalise_array(current_img):
    """Match EXP_4 training normalization: Otsu-positive median and unscaled MAD."""
    src = _otsu_input(current_img)
    _, binary = cv2.threshold(src, 0, 255 if src.dtype == np.uint8 else 65535, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    selected = current_img[binary > 0].astype(np.float32)
    if selected.size == 0:
        raise ValueError("Empty Otsu support; refusing meaningless normalization")
    med = float(np.median(selected))
    mad = float(median_abs_deviation(selected, scale=1.0))
    return (current_img.astype(np.float32) - med) / (mad + 1e-6)



def normalise_image(image_path):
    return normalise_array(plt.imread(image_path))


def prepare_image(image_path):
    raw = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
    if raw is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    if raw.ndim == 3:
        raw = cv2.cvtColor(raw, cv2.COLOR_BGRA2GRAY if raw.shape[2] == 4 else cv2.COLOR_BGR2GRAY)
    if raw.shape != settings.image_shape:
        raise ValueError(f"Image {image_path} has {raw.shape}, expected {settings.image_shape}")
    disk = get_disk_mask(raw)
    return normalise_array(raw), disk



@dataclass(eq=False)
class PreparedImage:
    key: str                  # absolute path — cache key of the coarse stage
    name: str
    image: np.ndarray         # normalised, disk-masked
    disk_mask: np.ndarray

    @property
    def shape(self):
        return self.image.shape


def load_prepared(directory, name):
    path = os.path.join(directory, name)
    image, disk_mask = prepare_image(path)
    return PreparedImage(key=os.path.abspath(path), name=name, image=image, disk_mask=disk_mask)


# ------------------------------------------------------------------
# Rasterisation helpers (production rasteriser: first polygon only, as trained)
# ------------------------------------------------------------------
def polygon_to_mask(annotation, h_img, w_img):
    """Fill the annotation polygon into a full-size uint8 mask (255 inside)."""
    points = annotation["segmentation"][0]
    coordinates = np.array(list(zip(points[::2], points[1::2])), dtype=np.int32)   # flat → (x, y) pairs
    mask = np.zeros((h_img, w_img), dtype=np.uint8)
    cv2.fillPoly(mask, [coordinates], 255)
    return mask


def expanded_bbox(annotation, h_img, w_img, expansion):
    """bbox grown by `expansion` px on each side, clamped to the image."""
    x, y, w, h = annotation["bbox"]
    x = max(0, int(x - expansion))
    y = max(0, int(y - expansion))
    w = min(w_img - x, int(w + 2 * expansion))
    h = min(h_img - y, int(h + 2 * expansion))
    return x, y, w, h


def center_in_canvas(crop, bucket_w, bucket_h, fill_value=0):
    """Place a crop in the middle of a (bucket_h, bucket_w) canvas."""
    h, w = crop.shape
    canvas = np.full((bucket_h, bucket_w), fill_value, dtype=crop.dtype)
    y_offset = (bucket_h - h) // 2
    x_offset = (bucket_w - w) // 2
    canvas[y_offset:y_offset + h, x_offset:x_offset + w] = crop
    return canvas


# ------------------------------------------------------------------
# SparseMask — bbox-local mask value object
# ------------------------------------------------------------------
@dataclass(frozen=True, eq=False)
class SparseMask:
    y0: int
    x0: int
    data: np.ndarray            # uint8 {0,1}, shape (h, w)

    @functools.cached_property
    def area(self):
        return int(np.count_nonzero(self.data))

    @property
    def bbox_xywh(self):
        return (self.x0, self.y0, self.data.shape[1], self.data.shape[0])

    @classmethod
    def from_full(cls, mask):
        binary = (np.asarray(mask) > 0).astype(np.uint8)
        x, y, w, h = cv2.boundingRect(binary)
        if w == 0 or h == 0:
            return cls(0, 0, np.zeros((0, 0), np.uint8))
        return cls(y, x, np.ascontiguousarray(binary[y:y + h, x:x + w]))

    def to_full(self, shape):
        full = np.zeros(shape, dtype=np.uint8)
        h, w = self.data.shape
        full[self.y0:self.y0 + h, self.x0:self.x0 + w] = self.data
        return full

    def intersection_area(self, other):
        y0, x0 = max(self.y0, other.y0), max(self.x0, other.x0)
        y1 = min(self.y0 + self.data.shape[0], other.y0 + other.data.shape[0])
        x1 = min(self.x0 + self.data.shape[1], other.x0 + other.data.shape[1])
        if y1 <= y0 or x1 <= x0:
            return 0
        a = self.data[y0 - self.y0:y1 - self.y0, x0 - self.x0:x1 - self.x0]
        b = other.data[y0 - other.y0:y1 - other.y0, x0 - other.x0:x1 - other.x0]
        return int(np.count_nonzero(a & b))

    def iou(self, other):
        inter = self.intersection_area(other)
        union = self.area + other.area - inter
        return inter / union if union > 0 else 0.0


@dataclass(frozen=True, eq=False)
class CoarseComponent:
    """One connected component proposed by the coarse stage (native-resolution coordinates)."""
    bbox: Tuple[int, int, int, int]      # x, y, w, h
    mask: SparseMask
    area: int
    mean_probability: float = 0.0


# ------------------------------------------------------------------
# Crop-size buckets and fine-stage input construction
# ------------------------------------------------------------------
def round_up_32(x):
    return int(np.ceil(x / 32) * 32)


@dataclass(frozen=True)
class BucketScheme:
    width_buckets: Tuple[int, ...]
    height_buckets: Tuple[int, ...]
    expansion: int
    dilation_size: int

    @classmethod
    def from_annotations(cls, annotations, expansion, dilation_size, percentiles):
        """Factory Method: bucket sizes are percentiles of the expanded training-bbox sizes."""
        widths = np.array([int(a["bbox"][2] + 2 * expansion) for a in annotations])
        heights = np.array([int(a["bbox"][3] + 2 * expansion) for a in annotations])
        width_buckets = sorted({max(32, round_up_32(np.percentile(widths, p))) for p in percentiles})
        height_buckets = sorted({max(32, round_up_32(np.percentile(heights, p))) for p in percentiles})
        return cls(tuple(width_buckets), tuple(height_buckets), expansion, dilation_size)

    def choose(self, crop_w, crop_h):
        bucket_w = next((b for b in self.width_buckets if b >= crop_w), self.width_buckets[-1])
        bucket_h = next((b for b in self.height_buckets if b >= crop_h), self.height_buckets[-1])
        return bucket_w, bucket_h


@dataclass(frozen=True, eq=False)
class FineInput:
    image_canvas: np.ndarray                 # float, bucket-sized
    region_canvas: np.ndarray                # uint8 {0,1}, bucket-sized
    crop_coords: Tuple[int, int, int, int]   # x0, y0, x1, y1 in the full image
    bucket: Tuple[int, int]                  # bucket_w, bucket_h

    @property
    def offsets(self):
        x0, y0, x1, y1 = self.crop_coords
        return (self.bucket[1] - (y1 - y0)) // 2, (self.bucket[0] - (x1 - x0)) // 2   # offset_y, offset_x


class FineInputBuilder:
    """Builds the exact fine-U-Net input (image canvas + dilated coarse region) for one component."""

    def __init__(self, scheme):
        self.scheme = scheme

    def _dilated_window(self, mask, window, image_shape):
        """cv2.dilate(full_mask, ones(k,k))[window], computed on window ± k only (identical result)."""
        k = self.scheme.dilation_size
        wx0, wy0, wx1, wy1 = window
        h_img, w_img = image_shape
        X0, Y0, X1, Y1 = max(0, wx0 - k), max(0, wy0 - k), min(w_img, wx1 + k), min(h_img, wy1 + k)
        canvas = np.zeros((Y1 - Y0, X1 - X0), dtype=np.uint8)
        h, w = mask.data.shape
        ly0, lx0 = max(mask.y0, Y0), max(mask.x0, X0)
        ly1, lx1 = min(mask.y0 + h, Y1), min(mask.x0 + w, X1)
        if ly1 > ly0 and lx1 > lx0:
            canvas[ly0 - Y0:ly1 - Y0, lx0 - X0:lx1 - X0] = mask.data[ly0 - mask.y0:ly1 - mask.y0, lx0 - mask.x0:lx1 - mask.x0]
        dilated = cv2.dilate(canvas, np.ones((k, k), dtype=np.uint8), iterations=1)
        return dilated[wy0 - Y0:wy1 - Y0, wx0 - X0:wx1 - X0]

    def build(self, image, component):
        e = self.scheme.expansion
        h_img, w_img = image.shape
        x, y, w, h = component.bbox

        x0, y0 = max(0, int(x - e)), max(0, int(y - e))
        x1, y1 = min(w_img, int(x + w + e)), min(h_img, int(y + h + e))
        crop_w, crop_h = x1 - x0, y1 - y0
        bucket_w, bucket_h = self.scheme.choose(crop_w, crop_h)

        if crop_w > bucket_w:                      # component wider than the largest bucket: re-centre
            center_x = int(x + w / 2)
            x0 = max(0, min(w_img - bucket_w, center_x - bucket_w // 2))
            x1 = x0 + bucket_w
        if crop_h > bucket_h:
            center_y = int(y + h / 2)
            y0 = max(0, min(h_img - bucket_h, center_y - bucket_h // 2))
            y1 = y0 + bucket_h

        image_crop = image[y0:y1, x0:x1]
        region_crop = self._dilated_window(component.mask, (x0, y0, x1, y1), image.shape)
        pad_value = np.median(image_crop)          # the rest of the bucket is filled with the median intensity
        return FineInput(
            image_canvas=center_in_canvas(image_crop, bucket_w, bucket_h, pad_value),
            region_canvas=center_in_canvas(region_crop, bucket_w, bucket_h, 0),
            crop_coords=(x0, y0, x1, y1),
            bucket=(bucket_w, bucket_h),
        )


# Fixed geometry of the measured B3 inference recipe; no checkpoint-dependent re-estimation.
bucket_scheme = BucketScheme(settings.width_buckets, settings.height_buckets, settings.expansion, settings.dilation_size)
fine_input_builder = FineInputBuilder(bucket_scheme)
print("Width buckets :", list(bucket_scheme.width_buckets))
print("Height buckets:", list(bucket_scheme.height_buckets))

## 5. Data exploration *(optional — failure-isolated)*

Dataset inspection, area / bbox distributions, normalisation statistics, disk-mask inspection and the human-vs-human annotator ceiling (A3). These stages feed no downstream state: a failure here is logged and skipped, never fatal.

JSON structure (for reference): `info, licenses, categories, images, annotations`; each annotation is one filament instance with `segmentation` (flattened x,y polygon), `area`, `iscrowd`, `spine`, `image_id`, `bbox`, `category_id`, `id`. Categories: Left, Right, Unidentifiable, Ambiguous.

**Inspection findings (from the original analysis):** the area distribution is strongly right-skewed (most filaments are small, a few are huge); bbox heights/widths peak near 50 px; up to ≈250 px the width-vs-height scatter is dense and roughly uniform, above that there are extreme outliers (e.g. 1200×200). Image intensity histograms show a first peak for the black outside and a second for the bright disk, so an Otsu threshold followed by the largest connected component isolates the disk; normalisation uses median/MAD (not mean/std) because extreme pixels bias the mean.

In [ ]:
def plot_hist(values, title, xlabel, bins, xlim=None):
    fig = plt.figure(figsize=(6, 4))
    plt.hist(values, bins=bins, edgecolor="black")
    if xlim:
        plt.xlim(*xlim)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.grid(True, linestyle="--", alpha=0.5)
    show_figure(fig)


def plot_scatter(x, y, title, xlabel, ylabel):
    fig = plt.figure(figsize=(6, 4))
    plt.scatter(x, y, alpha=0.5)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, linestyle="--", alpha=0.5)
    show_figure(fig)


class DatasetExplorer:
    """Read-only exploratory report over the annotation file and a few raw images."""

    def __init__(self, index, settings, dataset_paths):
        self.index, self.settings, self.paths = index, settings, dataset_paths
        self.data = index.raw

    def counts_and_annotators(self):
        data = self.data
        print(len(os.listdir(self.paths.train_images)), "train files |", len(os.listdir(self.paths.test_images)), "test files")
        print(type(data), list(data.keys()))
        print(data["images"][:1])
        print(data["categories"])
        print(data["annotations"][:1])

        instance_num = defaultdict(int)
        by_category = defaultdict(int)
        for item in data["annotations"]:
            instance_num[item["image_id"]] += 1
            by_category[item["category_id"]] += 1

        annotator_split = defaultdict(list)
        for item in data["images"]:
            annotator_id, file_id = item["id"].split("-", 1)
            annotator_split[file_id].append(annotator_id)
        unique_annotators = {a for ids in annotator_split.values() for a in ids}
        multi = sum(1 for ids in annotator_split.values() if len(ids) > 1)
        print(f"image count : {len(annotator_split)}")
        print(f"annotator count : {len(unique_annotators)}")
        print(f"images with multiple annotators : {multi}")
        print(dict(by_category))
        values = list(instance_num.values())
        print(f"mean : {statistics.mean(values)}\nmin : {min(values)}\nmax : {max(values)}\n"
              f"median : {statistics.median(values)}\nmode : {statistics.mode(values)}\n"
              f"standard dev : {statistics.stdev(values)}\nvariance : {statistics.variance(values)}")
        for category_id, label in ((1, "left"), (2, "right"), (3, "unidentifiable"), (4, "ambiguous")):
            print(f"{label} category count : {by_category[category_id]}")

    def geometry_distributions(self):
        areas = [a["area"] for a in self.data["annotations"]]
        bboxes = np.array([a["bbox"] for a in self.data["annotations"]])
        widths, heights = bboxes[:, 2], bboxes[:, 3]
        plot_hist(areas, "Distribution of Areas", "Area (pixels)", bins=50, xlim=(0, 15000))
        plot_hist(heights, "Bounding Box Heights", "Height (pixels)", bins=30)
        plot_hist(widths, "Bounding Box Widths", "Width (pixels)", bins=30)
        plot_scatter(widths, heights, "BBox Width vs. Height", "Width (pixels)", "Height (pixels)")

    def bucket_summary(self, scheme):
        print("Width buckets :", list(scheme.width_buckets))
        print("Height buckets:", list(scheme.height_buckets))

    def normalisation_statistics(self, n_images=10):
        thresholds, medians, mads = [], [], []
        for image in self.data["images"][:n_images]:
            current = plt.imread(os.path.join(self.paths.train_images, image["file_name"]))
            threshold, binary = cv2.threshold(_otsu_input(current), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            masked = current[binary > 0]
            thresholds.append(threshold)
            medians.append(np.median(masked))
            mads.append(median_abs_deviation(masked))
        for series in (thresholds, medians, mads):
            plot_hist(series, "", "", bins=10)

    def disk_and_polygon_inspection(self, record_position=5):
        records = self.data["images"]
        record = records[min(record_position, len(records) - 1)]
        img = plt.imread(os.path.join(self.paths.train_images, record["file_name"])).copy()
        disk_mask = get_disk_mask(img)

        fig = plt.figure(figsize=(7, 7))
        plt.imshow(img, cmap="gray")
        for annotation in self.index.annotation_ids_by_image.get(record["id"], []):
            points = self.index.annotations_by_id[annotation]["segmentation"][0]
            plt.gca().add_patch(Polygon(list(zip(points[::2], points[1::2])), closed=True, fill=False, linewidth=0.25))
        plt.title(f"Ground-truth polygons — {record['id']}")
        show_figure(fig)

        fig = plt.figure(figsize=(5, 5)); plt.imshow(disk_mask, cmap="gray"); plt.title("Disk mask"); show_figure(fig)

        # Stage 2: threshold the dark filaments inside the disk
        threshold_masked, _ = cv2.threshold(_otsu_input(img)[disk_mask].reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        print(f"Threshold inside disk: {threshold_masked}")
        masked_img = (_otsu_input(img) < threshold_masked) & disk_mask
        fig = plt.figure(figsize=(5, 5)); plt.imshow(masked_img, cmap="gray"); plt.title("Dark pixels inside disk"); show_figure(fig)

        # Stage 3: morphological closing connects nearby gaps in the thresholded filaments
        closed = cv2.morphologyEx(masked_img.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
        fig = plt.figure(figsize=(5, 5)); plt.imshow(closed, cmap="gray"); plt.title("After closing (15×15)"); show_figure(fig)

        # Stage 1 histogram: black outside + bright disk peaks
        fig = plt.figure(figsize=(6, 4)); plt.hist(img.ravel(), bins=64); plt.title("Intensity histogram"); show_figure(fig)
        threshold, _ = cv2.threshold(_otsu_input(img), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        print(f"Threshold for separating the solar disk: {threshold}")


@optional_stage("exploration")
def run_exploration():
    explorer = DatasetExplorer(index, settings, dataset_paths)
    explorer.counts_and_annotators()
    explorer.geometry_distributions()
    explorer.bucket_summary(bucket_scheme)
    explorer.normalisation_statistics()
    explorer.disk_and_polygon_inspection()


with run_log.stage("Exploration"):
    if settings.run_eda:
        run_exploration()
    else:
        print("Exploration disabled (settings.run_eda=False).")

### Annotator ceiling check (A3)
- Take annotator A's filament instances as the predictions.
- Take annotator B's instances for that same physical image as the ground truth.
- Match instances using IoU and Hungarian assignment, accepting matches only when IoU is greater than 0.5.
- Repeat for every ordered annotator pair, then aggregate TP, FP, FN, and matched IoUs to calculate PQ, SQ, and RQ.

Overlap-based split/merge flags are *candidates for visual inspection*, not definitive semantic labels.

In [ ]:
class AnnotatorCeilingAnalysis:
    """Detailed human-vs-human agreement analysis — writes five CSVs to the workspace."""

    IOU_THRESHOLD = 0.5
    DIAGNOSTIC_IOU = 0.1

    def __init__(self, index, settings, workspace):
        self.index, self.settings, self.workspace = index, settings, workspace
        h, w = settings.image_shape
        self.rles_by_record = {}
        for record_id, annotation_ids in index.annotation_ids_by_image.items():
            rles = [mask_util.merge(mask_util.frPyObjects(index.annotations_by_id[a]["segmentation"], h, w))
                    for a in annotation_ids]
            self.rles_by_record[record_id] = rles

    @staticmethod
    def _iou_matrix(pred_rles, gt_rles):
        if not pred_rles or not gt_rles:
            return np.zeros((len(pred_rles), len(gt_rles)), dtype=float)
        return np.asarray(mask_util.iou(pred_rles, gt_rles, [0] * len(gt_rles)), dtype=float)

    def compare(self, pred_rles, gt_rles):
        """Hungarian one-to-one matching; accept only IoU strictly above threshold."""
        n_pred, n_gt = len(pred_rles), len(gt_rles)
        matrix = self._iou_matrix(pred_rles, gt_rles)
        accepted = threshold_aware_hungarian(matrix, self.IOU_THRESHOLD)
        matched_pred, matched_gt = {p for p, _, _ in accepted}, {g for _, g, _ in accepted}
        unmatched_pred = [i for i in range(n_pred) if i not in matched_pred]
        unmatched_gt = [i for i in range(n_gt) if i not in matched_gt]
        tp, fp, fn = len(accepted), len(unmatched_pred), len(unmatched_gt)
        sum_iou = sum(iou for _, _, iou in accepted)
        denominator = tp + 0.5 * fp + 0.5 * fn
        return {
            "PQ": sum_iou / denominator if denominator else 0.0, "SQ": sum_iou / tp if tp else 0.0,
            "RQ": tp / denominator if denominator else 0.0, "TP": tp, "FP": fp, "FN": fn, "sum_iou": sum_iou,
            "n_pred": n_pred, "n_gt": n_gt, "matched_pairs": accepted,
            "unmatched_pred_idx": unmatched_pred, "unmatched_gt_idx": unmatched_gt, "iou_matrix": matrix,
        }

    def run(self):
        pair_results, disagreement_rows, pattern_rows = [], [], []
        for physical_id, record_ids in self.index.file_to_image_ids.items():
            record_ids = list(dict.fromkeys(r for r in record_ids if r in self.rles_by_record))
            if len(record_ids) < 2:
                continue
            for pred_id in record_ids:
                for gt_id in record_ids:
                    if pred_id == gt_id:
                        continue
                    result = self.compare(self.rles_by_record[pred_id], self.rles_by_record[gt_id])
                    key = {"physical_image": physical_id, "pred_record_id": pred_id, "gt_record_id": gt_id,
                           "pred_annotator": pred_id.split("-", 1)[0], "gt_annotator": gt_id.split("-", 1)[0]}
                    pair_results.append({**key, **{k: result[k] for k in
                                         ("PQ", "SQ", "RQ", "TP", "FP", "FN", "sum_iou", "n_pred", "n_gt")}})
                    matrix = result["iou_matrix"]
                    pred_multi = {p for p in result["unmatched_pred_idx"]
                                  if matrix.shape[1] and np.sum(matrix[p, :] >= self.DIAGNOSTIC_IOU) >= 2}
                    gt_multi = {g for g in result["unmatched_gt_idx"]
                                if matrix.shape[0] and np.sum(matrix[:, g] >= self.DIAGNOSTIC_IOU) >= 2}
                    for p, g, iou in result["matched_pairs"]:
                        disagreement_rows.append({**key, "case_type": "matched_shape", "pred_object_index": p,
                                                  "gt_object_index": g, "IoU": iou,
                                                  "diagnostic_note": "Matched instances; inspect lower-IoU pairs for boundary/extent differences."})
                    for p in result["unmatched_pred_idx"]:
                        overlaps = int(np.sum(matrix[p, :] >= self.DIAGNOSTIC_IOU)) if matrix.shape[1] else 0
                        disagreement_rows.append({**key, "case_type": "possible_split_merge_candidate" if p in pred_multi else "unmatched_pred_instance",
                                                  "pred_object_index": p, "gt_object_index": np.nan,
                                                  "IoU": float(np.max(matrix[p, :])) if matrix.shape[1] else 0.0,
                                                  "diagnostic_note": f"Unmatched predicted-side object; weak overlaps with {overlaps} opposite-side objects. Visual inspection needed."})
                    for g in result["unmatched_gt_idx"]:
                        overlaps = int(np.sum(matrix[:, g] >= self.DIAGNOSTIC_IOU)) if matrix.shape[0] else 0
                        disagreement_rows.append({**key, "case_type": "possible_split_merge_candidate" if g in gt_multi else "unmatched_gt_instance",
                                                  "pred_object_index": np.nan, "gt_object_index": g,
                                                  "IoU": float(np.max(matrix[:, g])) if matrix.shape[0] else 0.0,
                                                  "diagnostic_note": f"Unmatched GT-side object; weak overlaps with {overlaps} opposite-side objects. Visual inspection needed."})
                    pattern_rows.append({**key, "n_pred": result["n_pred"], "n_gt": result["n_gt"],
                                         "object_count_difference_pred_minus_gt": result["n_pred"] - result["n_gt"],
                                         "matched_count": result["TP"], "unmatched_pred_count": result["FP"],
                                         "unmatched_gt_count": result["FN"], "matched_mean_IoU": result["SQ"],
                                         "low_shape_match_count": sum(iou < 0.7 for _, _, iou in result["matched_pairs"]),
                                         "possible_split_merge_candidate_count": len(pred_multi) + len(gt_multi)})

        pair_df = pd.DataFrame(pair_results)
        if pair_df.empty:
            print("No multi-annotator image pairs found; annotator ceiling skipped.")
            return None
        tp, fp, fn = int(pair_df["TP"].sum()), int(pair_df["FP"].sum()), int(pair_df["FN"].sum())
        total_iou = float(pair_df["sum_iou"].sum())
        denominator = tp + 0.5 * fp + 0.5 * fn
        pooled = {"PQ": total_iou / denominator if denominator else 0.0, "SQ": total_iou / tp if tp else 0.0,
                  "RQ": tp / denominator if denominator else 0.0, "TP": tp, "FP": fp, "FN": fn,
                  "physical_images": int(pair_df["physical_image"].nunique()), "directed_comparisons": int(len(pair_df))}
        disagreement_df, pattern_df = pd.DataFrame(disagreement_rows), pd.DataFrame(pattern_rows)
        annotator_summary = (pair_df.groupby("pred_annotator", dropna=False)
                             .agg(comparisons=("PQ", "size"), mean_PQ=("PQ", "mean"), median_PQ=("PQ", "median"),
                                  mean_SQ=("SQ", "mean"), mean_RQ=("RQ", "mean"), mean_FP=("FP", "mean"), mean_FN=("FN", "mean"),
                                  mean_objects_pred=("n_pred", "mean"), mean_objects_gt=("n_gt", "mean")).reset_index())

        print("HUMAN-VS-HUMAN ANNOTATOR ANALYSIS\n---------------------------------")
        print(pd.Series(pooled).round(4))
        print("\nPer-comparison PQ distribution:"); print(pair_df["PQ"].describe().round(4))
        print("\nDisagreement categories (directed object cases):"); print(disagreement_df["case_type"].value_counts().to_string())
        print("\nAnnotator-side summary (interpret cautiously; image sets may differ):")
        print(annotator_summary.round(4).to_string(index=False))

        for frame, name in ((pair_df, "a3_annotator_pair_metrics.csv"), (pd.DataFrame([pooled]), "a3_annotator_ceiling_summary.csv"),
                            (disagreement_df, "a3_disagreement_examples.csv"), (pattern_df, "a3_annotator_error_patterns.csv"),
                            (annotator_summary, "a3_annotator_side_summary.csv")):
            workspace.write_csv(frame, name)
        print("\nSaved A3 CSVs to:", workspace.root)
        return pooled


@optional_stage("annotator ceiling (A3)")
def run_annotator_ceiling():
    return AnnotatorCeilingAnalysis(index, settings, workspace).run()


with run_log.stage("Annotator ceiling (A3)"):
    annotator_ceiling = run_annotator_ceiling() if settings.run_annotator_ceiling else None

## 6. Model Factory and lifecycle — load by default

Coarse: one-channel ResNet34 U-Net. Fine: two-channel EfficientNet-B3 U-Net. Strictly load the configured artifacts; never substitute another local checkpoint implicitly. Both default to `load`. Explicit `train` uses the retained legacy trainers; `fine_tune` first loads the configured checkpoint, then saves new weights separately. The optional trainer is not a claim to reproduce the seven-hour campaign. DataParallel wraps models only after loading/training and eval mode is explicit.

In [ ]:
class ModelFactory:
    """Factory: ResNet34 coarse, EfficientNet-B3 fine (independent encoder settings)."""

    ENCODER = "resnet34"

    @classmethod
    def create(cls, in_channels, pretrained, encoder):
        """Returns (model on device, encoder-weights description)."""
        weights_used = "none (weights loaded from checkpoint)"
        if pretrained:
            try:
                model = smp.Unet(encoder_name=encoder, encoder_weights="imagenet", in_channels=in_channels, classes=1)
                weights_used = "imagenet"
            except Exception as exc:  # noqa: BLE001 - offline commit: no weight download possible
                print(f"WARNING: ImageNet weights unavailable ({type(exc).__name__}: {str(exc)[:120]}). "
                      "Falling back to random initialisation.")
                model = smp.Unet(encoder_name=encoder, encoder_weights=None, in_channels=in_channels, classes=1)
                weights_used = "random (imagenet download failed)"
        else:
            model = smp.Unet(encoder_name=encoder, encoder_weights=None, in_channels=in_channels, classes=1)
        return model.to(device), weights_used

    @classmethod
    def create_coarse(cls, pretrained):
        return cls.create(1, pretrained, settings.coarse_encoder)

    @classmethod
    def create_fine(cls, pretrained):
        return cls.create(2, pretrained, settings.fine_encoder)


def load_state_dict_file(model, path):
    """Load a raw state dict or a common wrapper (state_dict / model_state_dict, DataParallel prefix)."""
    if not path or not os.path.isfile(path):
        raise FileNotFoundError(f"Checkpoint not found: {path!r}")
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:                       # torch < 1.13
        checkpoint = torch.load(path, map_location="cpu")
    except Exception:                       # wrapper containing non-tensor objects: trusted, user-supplied file
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(checkpoint, dict):
        for key in ("state_dict", "model_state_dict", "model"):
            if key in checkpoint and isinstance(checkpoint[key], dict):
                checkpoint = checkpoint[key]
                break
    if isinstance(checkpoint, dict) and checkpoint and all(k.startswith("module.") for k in checkpoint):
        checkpoint = {k[len("module."):]: v for k, v in checkpoint.items()}
    model.load_state_dict(checkpoint)
    return os.path.abspath(path)


@dataclass(eq=False)
class ModelHandle:
    name: str
    model: nn.Module
    checkpoint_path: str
    sha256: Optional[str]
    mode: str
    encoder_weights: str


class ModelLifecycle(abc.ABC):
    """Template Method: build → (train | load) → eval → handle."""

    name = "model"

    def __init__(self, settings, workspace, locator):
        self.settings, self.workspace, self.locator = settings, workspace, locator

    @property
    @abc.abstractmethod
    def mode(self):
        ...

    @abc.abstractmethod
    def _build(self):
        """Return (model, encoder_weights_description)."""

    @abc.abstractmethod
    def _train(self, model):
        """Train in place and return the saved checkpoint path."""

    @abc.abstractmethod
    def _load(self, model):
        """Load weights and return the checkpoint path used."""

    def obtain(self):
        model, encoder_weights = self._build()
        if self.mode == "fine_tune":
            initialization_path = self._load(model)
            print(f"[{self.name}] deliberate fine-tune initialization: {initialization_path}")
            checkpoint_path = self._train(model)
        else:
            checkpoint_path = self._train(model) if self.mode == "train" else self._load(model)
        model.eval()
        if self.settings.use_data_parallel and device.type == "cuda" and torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
            print(f"[{self.name}] DataParallel on {torch.cuda.device_count()} visible GPUs")
        for parameter in model.parameters():
            parameter.requires_grad_(False)          # inference-only from here on; frees autograd bookkeeping
        handle = ModelHandle(self.name, model, checkpoint_path, sha256_file(checkpoint_path), self.mode, encoder_weights)
        print(f"[{self.name}] mode={self.mode} | checkpoint={checkpoint_path}\n[{self.name}] SHA-256={handle.sha256}")
        return handle

    def _resolve_or_raise(self, explicit, patterns, what):
        path = self.locator.resolve_file(explicit_paths=explicit, patterns=patterns)
        if path is None:
            raise FileNotFoundError(
                f"No {what} checkpoint found. Looked at explicit paths {list(explicit)} and patterns {list(patterns)} "
                f"under {[str(r) for r in self.locator.roots]}. Attach the checkpoint dataset or set the mode to 'train'."
            )
        return path


def dice_loss(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred).flatten()
    target = target.flatten()
    intersection = (pred * target).sum()
    return 1.0 - (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)


class CoarseSegmentationDataset(Dataset):
    """Full-frame samples resized to coarse_resize² with D4 symmetry + photometric jitter (train only)."""

    def __init__(self, index, split_dict, image_dir, resize, is_train=True):
        self.index, self.split_dict, self.image_dir = index, split_dict, image_dir
        self.image_ids = list(split_dict.keys())
        self.resize, self.is_train = resize, is_train

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.index.images_by_id[image_id]
        masked_img, _ = prepare_image(os.path.join(self.image_dir, info["file_name"]))
        img_h, img_w = masked_img.shape

        mask = np.zeros((img_h, img_w), dtype=np.uint8)
        for ann_id in self.split_dict[image_id]:
            mask = np.maximum(mask, polygon_to_mask(self.index.annotations_by_id[ann_id], img_h, img_w))

        resized_img = cv2.resize(masked_img, (self.resize, self.resize), interpolation=cv2.INTER_AREA)
        resized_mask = cv2.resize(mask, (self.resize, self.resize), interpolation=cv2.INTER_NEAREST)

        if self.is_train:
            if random.random() > 0.5:
                resized_img, resized_mask = np.fliplr(resized_img).copy(), np.fliplr(resized_mask).copy()
            if random.random() > 0.5:
                resized_img, resized_mask = np.flipud(resized_img).copy(), np.flipud(resized_mask).copy()
            k = random.choice([0, 1, 2, 3])
            if k > 0:
                resized_img, resized_mask = np.rot90(resized_img, k).copy(), np.rot90(resized_mask, k).copy()
            gain, bias = random.uniform(0.9, 1.1), random.uniform(-0.1, 0.1)
            resized_img = np.clip(resized_img * gain + bias, -3.0, 3.0)

        return (torch.tensor(resized_img, dtype=torch.float32).unsqueeze(0),
                torch.tensor(resized_mask / 255.0, dtype=torch.float32).unsqueeze(0))


class CoarseLifecycle(ModelLifecycle):
    name = "coarse"

    def __init__(self, settings, workspace, locator, index, split, image_dir):
        super().__init__(settings, workspace, locator)
        self.index, self.split, self.image_dir = index, split, image_dir
        self.checkpoint_name = f"coarse_{settings.coarse_resize}_{settings.experiment_id}.pth"

    @property
    def mode(self):
        return self.settings.coarse_mode

    def _build(self):
        return ModelFactory.create_coarse(pretrained=(self.mode == "train"))

    def _loader(self, dataset, shuffle, seed):
        return DataLoader(dataset, batch_size=self.settings.coarse_batch_size, shuffle=shuffle,
                          num_workers=self.settings.num_workers, worker_init_fn=seed_worker,
                          generator=make_generator(seed), pin_memory=(device.type == "cuda"))

    def _train(self, model):
        s = self.settings
        seed_everything(s.seed + 11)
        train_loader = self._loader(CoarseSegmentationDataset(self.index, self.split.train_split_dict, self.image_dir, s.coarse_resize, True), True, s.seed + 11)
        val_loader = self._loader(CoarseSegmentationDataset(self.index, self.split.validation_split_dict, self.image_dir, s.coarse_resize, False), False, s.seed + 12)

        # Balanced BCE (pos_weight 5.0, reduced from 25.0 to suppress false-positive hallucinations) + Dice
        bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([s.coarse_pos_weight]).to(device))
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=s.coarse_epochs)

        best_val, best_path = float("inf"), None
        print(f"Beginning coarse training ({s.coarse_resize}², pos_weight={s.coarse_pos_weight}, {s.coarse_epochs} epochs)…")
        for epoch in range(s.coarse_epochs):
            model.train()
            train_loss = 0.0
            for imgs, masks in train_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                optimizer.zero_grad()
                preds = model(imgs)
                loss = bce(preds, masks) + dice_loss(preds, masks)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            scheduler.step()
            train_loss /= max(1, len(train_loader))

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for imgs, masks in val_loader:
                    imgs, masks = imgs.to(device), masks.to(device)
                    preds = model(imgs)
                    val_loss += (bce(preds, masks) + dice_loss(preds, masks)).item()
            val_loss /= max(1, len(val_loader))
            print(f"Epoch [{epoch + 1:02d}/{s.coarse_epochs:02d}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

            if val_loss < best_val or best_path is None:      # `or best_path is None` guards NaN losses
                best_val = val_loss
                best_path = self.workspace.save_state_dict(model, "checkpoints", self.checkpoint_name)
                print(f"  -> best model saved to: {best_path}")

        # Inference must use the best-val weights recorded in the manifest, not the last epoch.
        load_state_dict_file(model, best_path)
        print("Restored best coarse checkpoint:", best_path)
        return best_path

    def _load(self, model):
        s = self.settings
        explicit = [s.coarse_load_path]
        patterns = [Path(s.coarse_load_path).name]
        path = self._resolve_or_raise(explicit, patterns, "coarse")
        return load_state_dict_file(model, path)

## 6.1 Coarse segmenter — batched 4-view TTA with a caching proxy

* **One forward pass for four views** (original, H-flip, V-flip, both) under `torch.inference_mode()`; the inverse flips and the mean are done on the GPU, followed by a single device→host copy.
* **OOM back-off:** if the 4-view batch does not fit, the same views are run in chunks of 2, then 1 (identical result, lower peak memory) instead of crashing the commit.
* **Leak control:** no tensors escape the call, nothing keeps autograd state (`requires_grad_(False)` + `inference_mode`), and cached GPU blocks are released every `empty_cache_every` images.
* **Self-test:** the batched ensemble is compared against sequential per-view inference on the first image (`max |Δp|` is reported).
* **`CachingCoarseSegmenter`** (Proxy) memoises the *compact* components per image, so the coarse stage runs **once per image** across fine-sample creation, scorer fitting, validation, test and every appendix diagnostic.

In [ ]:
@dataclass(frozen=True)
class CoarseParams:
    resize: int
    threshold: float
    min_area: int
    closing_kernel: int
    use_tta: bool

    @classmethod
    def from_settings(cls, s):
        return cls(s.coarse_resize, s.coarse_threshold, s.coarse_min_area, s.coarse_closing_kernel, s.coarse_use_tta)


class CoarseSegmenter:
    """Runs the coarse U-Net (optionally 4-view TTA) and extracts connected components at native scale."""

    def __init__(self, model, params, empty_cache_every=40):
        self.model, self.params, self.empty_cache_every = model, params, empty_cache_every
        self.images_seen = 0
        self.oom_fallbacks = 0

    @staticmethod
    def _views(x):
        return [x, torch.flip(x, dims=[3]), torch.flip(x, dims=[2]), torch.flip(x, dims=[2, 3])]

    @staticmethod
    def _undo(probs):
        """probs: [4,1,H,W] in view order → list of [H,W] maps in the original orientation."""
        return [probs[0, 0], torch.flip(probs[1, 0], dims=[1]), torch.flip(probs[2, 0], dims=[0]),
                torch.flip(probs[3, 0], dims=[0, 1])]

    def _forward_chunked(self, views, chunk):
        outputs = []
        for start in range(0, len(views), chunk):
            outputs.append(torch.sigmoid(self.model(torch.cat(views[start:start + chunk], dim=0))))
        return torch.cat(outputs, dim=0)

    @torch.inference_mode()
    def probability_map(self, masked_img, force_chunk=None):
        """Ensemble probability at coarse resolution, float32 numpy [R,R]."""
        r = self.params.resize
        resized = cv2.resize(masked_img, (r, r), interpolation=cv2.INTER_AREA)
        x = torch.from_numpy(np.ascontiguousarray(resized, dtype=np.float32))[None, None].to(device)
        self.model.eval()

        if not self.params.use_tta:
            return torch.sigmoid(self.model(x))[0, 0].float().cpu().numpy()

        views = self._views(x)
        chunks = [force_chunk] if force_chunk else list(dict.fromkeys([settings.coarse_tta_chunk, 1]))
        probs = None
        for chunk in chunks:
            try:
                probs = self._forward_chunked(views, chunk)
                break
            except RuntimeError as exc:                  # torch.cuda.OutOfMemoryError subclasses RuntimeError
                if "out of memory" not in str(exc).lower() or chunk == chunks[-1]:
                    raise
                self.oom_fallbacks += 1
                print(f"[coarse TTA] CUDA OOM with chunk={chunk}; retrying with smaller chunks")
                release_memory()
        ensemble = torch.stack(self._undo(probs), dim=0).mean(dim=0)
        return ensemble.float().cpu().numpy()

    def self_test(self, prepared, tolerance=1e-3):
        """Batched TTA must equal sequential per-view TTA (BatchNorm is in eval mode)."""
        masked = prepared.image * prepared.disk_mask
        batched = self.probability_map(masked)
        sequential = self.probability_map(masked, force_chunk=1)
        diff = float(np.abs(batched - sequential).max())
        print(f"[coarse TTA self-test] max |batched - sequential| = {diff:.2e} (tolerance {tolerance:.0e})")
        if diff > tolerance:
            warnings.warn(f"Batched TTA deviates from sequential TTA by {diff:.2e}")
        return diff

    def extract_components(self, prepared, prob_coarse, threshold=None):
        p = self.params
        image, disk_mask = prepared.image, prepared.disk_mask
        masked_img = image * disk_mask
        original_h, original_w = masked_img.shape

        prob_native = cv2.resize(prob_coarse, (original_w, original_h), interpolation=cv2.INTER_LINEAR)   # continuous upsampling

        coarse_binary = ((prob_native > (p.threshold if threshold is None else threshold)) & disk_mask).astype(np.uint8)                          # disk-bounded threshold
        if p.closing_kernel > 1:                                                                            # preserve continuous spines
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (p.closing_kernel, p.closing_kernel))
            coarse_binary = cv2.morphologyEx(coarse_binary, cv2.MORPH_CLOSE, kernel)

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(coarse_binary, connectivity=8)
        scale_factor = (original_w / p.resize) * (original_h / p.resize)
        native_min_area = int(p.min_area * scale_factor)

        components = []
        for label in range(1, num_labels):
            area = int(stats[label, cv2.CC_STAT_AREA])
            if area < native_min_area:
                continue
            x0, y0 = int(stats[label, cv2.CC_STAT_LEFT]), int(stats[label, cv2.CC_STAT_TOP])
            w, h = int(stats[label, cv2.CC_STAT_WIDTH]), int(stats[label, cv2.CC_STAT_HEIGHT])
            local = np.ascontiguousarray((labels[y0:y0 + h, x0:x0 + w] == label).astype(np.uint8))
            components.append(CoarseComponent(bbox=(x0, y0, w, h), mask=SparseMask(y0, x0, local), area=area, mean_probability=float(prob_native[y0:y0+h, x0:x0+w][local > 0].mean())))

        self.images_seen += 1
        if device.type == "cuda" and self.images_seen % self.empty_cache_every == 0:
            torch.cuda.empty_cache()
        return components

    def components(self, prepared):
        return self.extract_components(prepared, self.probability_map(prepared.image * prepared.disk_mask))


class CachingCoarseSegmenter:
    """Proxy: sparse components in memory, immutable float32 soft maps on disk.

    Residual thresholds reuse the SAME coarse soft map, never a second model pass.
    """
    def __init__(self, inner):
        self.inner = inner
        self._cache, self._map_paths = {}, {}
        self.hits = self.misses = 0
        self.namespace = Path(workspace.path("coarse_cache", hashlib.sha256(repr(inner.params).encode()).hexdigest()[:16]))
        self.namespace.mkdir(parents=True, exist_ok=True)

    @property
    def params(self):
        return self.inner.params

    def components(self, prepared):
        key = (prepared.key, self.params)
        if key in self._cache:
            self.hits += 1
            return self._cache[key]
        self.misses += 1
        probability = self.inner.probability_map(prepared.image * prepared.disk_mask)
        path = self.namespace / (hashlib.sha256(prepared.key.encode()).hexdigest() + ".npy")
        # A new workspace owns this namespace; no external cache is silently reused.
        if not path.exists():
            with path.open("xb") as handle:
                np.save(handle, probability.astype(np.float32), allow_pickle=False)
        self._map_paths[key] = str(path)
        self._cache[key] = self.inner.extract_components(prepared, probability)
        return self._cache[key]

    def components_at(self, prepared, threshold):
        self.components(prepared)
        probability = np.load(self._map_paths[(prepared.key, self.params)], allow_pickle=False)
        return self.inner.extract_components(prepared, probability, float(threshold))

    def peek(self, path):
        return self._cache.get((os.path.abspath(path), self.params))

    def clear(self):
        self._cache.clear()
        self._map_paths.clear()
        release_memory()

    def stats(self):
        return {"coarse_cache_hits": self.hits, "coarse_cache_misses": self.misses,
                "cached_images": len(self._cache), "tta_oom_fallbacks": self.inner.oom_fallbacks}

In [ ]:
with run_log.stage("Coarse U-Net"):
    coarse_lifecycle = CoarseLifecycle(settings, workspace, locator, index, split, dataset_paths.train_images)
    coarse_handle = coarse_lifecycle.obtain()
    coarse_params = CoarseParams.from_settings(settings)
    coarse_segmenter = CachingCoarseSegmenter(CoarseSegmenter(coarse_handle.model, coarse_params, settings.empty_cache_every))

    # Equivalence self-test of the batched 4-view TTA on one real image (non-fatal).
    @optional_stage("coarse TTA self-test")
    def _tta_self_test():
        return coarse_segmenter.inner.self_test(load_prepared(dataset_paths.train_images, train_image_names[0]))
    tta_self_test_diff = _tta_self_test() if settings.run_coarse_tta_self_test and settings.coarse_use_tta else None

## 7. Fine samples and ROI-aware loss — training is opt-in

No samples or training loaders are constructed in `fine_mode="load"`. The retained loss asserts `[B,1,H,W]` ROI shapes and keeps region scaling at 0/1. Singleton bucket/tail batches are guarded for BatchNorm. New training never overwrites the known B3 checkpoint. Keep the default load mode for the final submission run.

In [ ]:
class FineSampleBuilder:
    """Turns coarse proposals of one physical training image into (crop, region, target) samples."""

    def __init__(self, index, split, image_dir, coarse_segmenter, input_builder):
        self.index, self.split, self.image_dir = index, split, image_dir
        self.coarse, self.input_builder = coarse_segmenter, input_builder

    def build_for_file(self, file_id):
        record_ids = [r for r in self.index.file_to_image_ids[file_id] if r in self.split.train_split_dict]
        if not record_ids:
            return []
        prepared = load_prepared(self.image_dir, self.index.images_by_id[record_ids[0]]["file_name"])
        h_img, w_img = prepared.shape
        components = self.coarse.components(prepared)

        gt_masks = {}
        for record_id in record_ids:
            mask = np.zeros((h_img, w_img), dtype=np.uint8)
            for annotation_id in self.split.train_split_dict[record_id]:
                mask = np.maximum(mask, polygon_to_mask(self.index.annotations_by_id[annotation_id], h_img, w_img))
            gt_masks[record_id] = mask

        samples = []
        for component_index, component in enumerate(components):
            fine_input = self.input_builder.build(prepared.image, component)
            x0, y0, x1, y1 = fine_input.crop_coords
            bucket_w, bucket_h = fine_input.bucket
            image_canvas = fine_input.image_canvas.astype(np.float32)
            region_canvas = fine_input.region_canvas * 255
            for record_id in record_ids:
                target_canvas = center_in_canvas(gt_masks[record_id][y0:y1, x0:x1], bucket_w, bucket_h)
                samples.append({"image_id": record_id, "component_index": component_index, "bucket": fine_input.bucket,
                                "bbox": component.bbox, "crop_coords": fine_input.crop_coords, "image": image_canvas,
                                "dilated_mask": region_canvas, "target": target_canvas})
        return samples

    def build_all(self, reporter=None):
        samples = []
        for file_id in self.split.train_file_ids:
            samples.extend(self.build_for_file(file_id))
            if reporter:
                reporter.step(f"| samples: {len(samples)}")
        return samples


class FilamentDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image, dilated_mask, target = sample["image"], sample["dilated_mask"] / 255.0, sample["target"]
        x = torch.tensor(np.stack([image, dilated_mask], axis=0), dtype=torch.float32)
        region = torch.tensor(dilated_mask, dtype=torch.float32).unsqueeze(0)            # [1,H,W]
        target = torch.tensor(target / 255.0, dtype=torch.float32).unsqueeze(0)          # (H,W) → (1,H,W)
        return x, target, region


def fine_criterion(predictions, targets, regions):
    """BCE + batch-global Dice, both restricted to each sample's ROI mask."""
    if regions.ndim == 3:
        regions = regions.unsqueeze(1)
    if predictions.ndim != 4 or targets.ndim != 4:
        raise ValueError(f"Expected predictions/targets [B,1,H,W]; got {tuple(predictions.shape)} and {tuple(targets.shape)}")
    if targets.shape != predictions.shape:
        raise ValueError(f"Target shape {tuple(targets.shape)} does not match predictions {tuple(predictions.shape)}")
    if regions.shape != predictions.shape:
        raise ValueError(f"ROI shape {tuple(regions.shape)} does not match predictions {tuple(predictions.shape)}. "
                         "Refusing unintended batch broadcasting.")
    regions, targets = regions.to(dtype=predictions.dtype), targets.to(dtype=predictions.dtype)
    bce_pixels = F.binary_cross_entropy_with_logits(predictions, targets, reduction="none")
    bce = (bce_pixels * regions).sum() / regions.sum().clamp_min(1.0)
    probs = torch.sigmoid(predictions)
    intersection = (probs * targets * regions).sum()
    denominator = (probs * regions).sum() + (targets * regions).sum()
    return bce + (1 - (2 * intersection + 1e-6) / (denominator + 1e-6))


class FineLifecycle(ModelLifecycle):
    name = "fine"

    def __init__(self, settings, workspace, locator, sample_builder, sample_hooks=()):
        super().__init__(settings, workspace, locator)
        self.sample_builder = sample_builder
        self.sample_hooks = list(sample_hooks)          # Observer hooks: called with the finished sample list
        self.checkpoint_name = f"fine_{settings.experiment_id}.pth"

    @property
    def mode(self):
        return self.settings.fine_mode

    def _build(self):
        return ModelFactory.create_fine(pretrained=(self.mode == "train"))

    def _train(self, model):
        s = self.settings
        seed_everything(s.seed + 21)
        samples = self.sample_builder.build_all(
            ProgressReporter("Fine samples (train images)", len(self.sample_builder.split.train_file_ids), every=50)
        )
        empty = sum(1 for smp_ in samples if not ((smp_["target"] > 0) & (smp_["dilated_mask"] > 0)).any())
        print(f"total samples: {len(samples)}\nsamples with empty target inside the region: {empty}")
        if not samples:
            raise RuntimeError("No fine-stage training samples were produced (coarse stage returned no components).")
        for hook in self.sample_hooks:
            hook(samples)

        bucket_samples = defaultdict(list)
        for sample in samples:
            bucket_samples[sample["bucket"]].append(sample)
        if s.fine_batch_size < 2:
            raise ValueError("Fine training needs batch_size >= 2 for BatchNorm")
        bucket_samples = {bucket: items * 2 if len(items) == 1 else items for bucket, items in bucket_samples.items()}
        loaders = {
            bucket: DataLoader(
                FilamentDataset(items),
                batch_size=s.fine_batch_size,
                shuffle=True,
                drop_last=(len(items) % s.fine_batch_size == 1),
                generator=make_generator(s.seed + 22 + i),
            )
            for i, (bucket, items) in enumerate(bucket_samples.items())
        }
        print("Number of buckets:", len(loaders), "| bucket sizes:", {k: len(v.dataset) for k, v in loaders.items()})

        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=s.fine_epochs, eta_min=1e-6)

        for epoch in range(s.fine_epochs):
            model.train()
            epoch_loss, epoch_steps = 0.0, 0
            for bucket, loader in loaders.items():
                bucket_loss, bucket_steps = 0.0, 0
                for step, (images, masks, regions) in enumerate(loader):
                    images, masks, regions = images.to(device), masks.to(device), regions.to(device)
                    optimizer.zero_grad()
                    loss = fine_criterion(model(images), masks, regions)
                    loss.backward()
                    optimizer.step()
                    bucket_loss += loss.item()
                    epoch_loss += loss.item()
                    bucket_steps += 1
                    epoch_steps += 1
                    if (step + 1) % 25 == 0 or (step + 1) == len(loader):
                        print(
                            f"Epoch [{epoch + 1}/{s.fine_epochs}] Bucket {bucket} Step [{step + 1}/{len(loader)}] "
                            f"- Loss: {loss.item():.4f} - Avg: {bucket_loss / bucket_steps:.4f}"
                        )
            if epoch_steps == 0:
                raise RuntimeError("No fine-stage training batches were produced.")
            scheduler.step()
            print(
                f"Epoch [{epoch + 1}/{s.fine_epochs}] - Loss: {epoch_loss / epoch_steps:.4f} "
                f"| LR: {scheduler.get_last_lr()[0]:.2e}"
            )

        path = self.workspace.save_state_dict(model, "checkpoints", self.checkpoint_name)
        print("Saved new fine checkpoint:", path)
        del samples, bucket_samples, loaders, optimizer, scheduler
        release_memory()
        return path

    def _load(self, model):
        s = self.settings
        explicit = [s.fine_load_path]
        path = self._resolve_or_raise(explicit, [Path(s.fine_load_path).name], "fine")
        return load_state_dict_file(model, path)


def inspect_fine_samples(samples, n_samples=3, rng_seed=0):
    """Original cell 41: full image + coarse/crop boxes, bucket input + dilated region, and target."""
    chosen = random.Random(rng_seed).sample(samples, min(n_samples, len(samples)))
    fig, axes = plt.subplots(len(chosen), 3, figsize=(18, 6 * len(chosen)), squeeze=False)
    for row, sample in enumerate(chosen):
        info = index.images_by_id[sample["image_id"]]
        img = normalise_image(os.path.join(dataset_paths.train_images, info["file_name"]))
        bx, by, b_w, b_h = sample["bbox"]
        x0, y0, x1, y1 = sample["crop_coords"]
        ax = axes[row, 0]
        ax.imshow(img, cmap="gray")
        ax.add_patch(patches.Rectangle((bx, by), b_w, b_h, fill=False, linewidth=2, edgecolor="red", label="Coarse bbox"))
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, linewidth=2, edgecolor="cyan", label="Crop box"))
        ax.set_title(f"Full image\n{sample['image_id']}"); ax.legend(); ax.axis("off")

        ax = axes[row, 1]
        region = sample["dilated_mask"]
        ax.imshow(sample["image"], cmap="gray")
        ax.imshow(np.ma.masked_where(region == 0, region), cmap="jet", alpha=0.35)
        bucket_w, bucket_h = sample["bucket"]
        ax.add_patch(patches.Rectangle((0, 0), bucket_w, bucket_h, fill=False, linewidth=2, edgecolor="cyan"))
        ax.set_title(f"Bucket input\nbucket = {sample['bucket']}"); ax.axis("off")

        ax = axes[row, 2]
        ax.imshow(sample["target"], cmap="gray")
        contours, _ = cv2.findContours((region > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            contour = contour.reshape(-1, 2)
            if len(contour) > 2:
                ax.plot(contour[:, 0], contour[:, 1], color="red", linewidth=1.5)
        ax.set_title("Target filament\nred = loss-mask boundary"); ax.axis("off")
    plt.tight_layout()
    show_figure(fig)


inspect_fine_samples_safe = optional_stage("fine input inspection")(inspect_fine_samples)

with run_log.stage("Fine U-Net"):
    sample_builder = FineSampleBuilder(index, split, dataset_paths.train_images, coarse_segmenter, fine_input_builder)
    fine_lifecycle = FineLifecycle(
        settings, workspace, locator, sample_builder,
        sample_hooks=[inspect_fine_samples_safe] if settings.run_fine_input_inspection else [],
    )
    if settings.fine_mode == "load":
        print("fine_mode='load': skipping fine-training crop generation and DataLoader construction.")
    fine_handle = fine_lifecycle.obtain()
    release_memory()

## 8. Final inference Strategies and population-level Pipeline Facade

The candidate-generation floor is **20 native pixels**; final greedy cleanup uses **75**. Features use explicit global coordinates and ordered eight-feature schema. N4 ranks four-view uncertainty over the *whole supplied split*, then upgrades the top 20% primary crops. It must not be reconstructed one image at a time. Only routed maps receive the measured float16 rounding. Disk caches retain crop-local maps/support, not all full-image canvases.

Residual rescue is a bounded challenger: 0.22 coarse threshold, mask-IoU<0.30 against all primary components, at most four added proposals, ordinary four-view fine inference. Main and rescue candidates are resolved jointly by the same scorer. N4+rescue is new and must win its validation comparison before export.

In [ ]:
import csv
import tempfile
import warnings


FEATURE_NAMES = (
    "log_area", "mean_prob", "max_prob", "std_prob", "aspect_ratio", "extent",
    "norm_radius", "tortuosity",
)
RECIPE_NAMES = ("n1_standard", "n4_midpoint", "n3_rescue", "n4_rescue")
_ROUTED_RECIPES = ("n4_midpoint", "n4_rescue")
_RESIDUAL_RECIPES = ("n3_rescue", "n4_rescue")


def extract_candidate_features(mask, prob, x0=0, y0=0, shape=None):
    """Source eight-feature schema; shape is the full image, or mask.shape for legacy calls."""
    ys, xs = np.where(mask > 0)
    if not len(xs):
        return [0.0] * len(FEATURE_NAMES)
    values = np.asarray(prob[mask > 0], dtype=np.float32)
    area = float(len(xs))
    h = max(1, int(ys.max() - ys.min() + 1))
    w = max(1, int(xs.max() - xs.min() + 1))
    image_h, image_w = mask.shape if shape is None else shape
    cx, cy = float(x0 + xs.mean()), float(y0 + ys.mean())
    norm_radius = math.hypot(cx - image_w / 2, cy - image_h / 2) / (image_w / 2)
    contours, _ = cv2.findContours(
        (mask > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE,
    )
    perimeter = float(sum(cv2.arcLength(contour, True) for contour in contours))
    return [
        float(math.log(max(area, 1.0))), float(values.mean()), float(values.max()),
        float(values.std()), float(max(w, h) / min(w, h)), float(area / (w * h)),
        float(norm_radius), float(perimeter / (math.sqrt(area) + 1e-6)),
    ]


_FINE_VIEWS = (
    "identity", "hflip", "vflip", "rot180", "rot90", "rot270", "transpose", "anti_transpose",
)


def _fine_transform(array, view):
    """D4 on the last two axes; quarter turns swap rectangular canvas dimensions."""
    if view == "identity":
        return array
    if view == "hflip":
        return np.flip(array, axis=-1)
    if view == "vflip":
        return np.flip(array, axis=-2)
    if view == "rot180":
        return np.rot90(array, 2, axes=(-2, -1))
    if view == "rot90":
        return np.rot90(array, 1, axes=(-2, -1))
    if view == "rot270":
        return np.rot90(array, 3, axes=(-2, -1))
    if view == "transpose":
        return np.swapaxes(array, -2, -1)
    if view == "anti_transpose":
        return np.rot90(np.swapaxes(array, -2, -1), 2, axes=(-2, -1))
    raise ValueError(f"Unknown fine TTA view: {view}")


def _fine_inverse(array, view):
    if view == "rot90":
        return _fine_transform(array, "rot270")
    if view == "rot270":
        return _fine_transform(array, "rot90")
    return _fine_transform(array, view)


@dataclass(frozen=True, eq=False)
class FineCrop:
    crop_coords: tuple
    prob_crop: np.ndarray        # float32 [crop_h, crop_w], never the padded canvas
    region_crop: np.ndarray      # bool [crop_h, crop_w]
    uncertainty: float = 0.0     # mean view variance over ALL unpadded pixels


class FineSegmenter:
    """Bucketed four-view mean, or selected eight-view midpoint-median inference."""

    def __init__(self, model, input_builder, batch_size):
        self.model, self.input_builder, self.batch_size = model, input_builder, int(batch_size)
        if self.batch_size < 1:
            raise ValueError("Fine inference batch_size must be positive")

    def _forward(self, batch_np):
        x = torch.from_numpy(np.ascontiguousarray(batch_np, dtype=np.float32)).to(device)
        return torch.sigmoid(self.model(x))[:, 0].float().cpu().numpy()

    def _forward_safe(self, batch_np):
        try:
            return self._forward(batch_np)
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower() or len(batch_np) == 1:
                raise
        # Leave the exception scope before releasing memory, so its traceback drops tensors.
        release_memory()
        half = len(batch_np) // 2
        return np.concatenate([
            self._forward_safe(batch_np[:half]), self._forward_safe(batch_np[half:]),
        ], axis=0)

    def _view_stack(self, batch_np, mode):
        views = _FINE_VIEWS[:4] if mode == "4view" else _FINE_VIEWS
        return np.stack([
            _fine_inverse(self._forward_safe(_fine_transform(batch_np, view)), view)
            for view in views
        ], axis=0).astype(np.float32, copy=False)

    @torch.inference_mode()
    def infer(self, prepared, components, indices=None, mode="4view", aggregation="mean"):
        """Return a component-length list; unselected indices are None.

        Ordinary calls ALWAYS use four-view mean, including scorer training and residuals.
        Routed calls explicitly request mode='8view', aggregation='midpoint_median'.
        Only routed probabilities are rounded float32 -> float16 -> float32, as in N4.
        """
        if (mode, aggregation) not in (("4view", "mean"), ("8view", "midpoint_median")):
            raise ValueError(f"Unsupported fine inference combination: {mode}/{aggregation}")
        selected = list(range(len(components))) if indices is None else list(indices)
        if any(not isinstance(i, (int, np.integer)) for i in selected):
            raise TypeError("Fine component indices must be integers")
        if len(set(selected)) != len(selected):
            raise ValueError("Duplicate fine component indices")
        if any(i < 0 or i >= len(components) for i in selected):
            raise IndexError("Fine component index out of range")
        selected = sorted(map(int, selected))
        results = [None] * len(components)
        if not selected:
            return results
        self.model.eval()
        inputs = {i: self.input_builder.build(prepared.image, components[i]) for i in selected}
        groups = defaultdict(list)
        for position, fine_input in inputs.items():
            groups[tuple(fine_input.bucket)].append(position)
        for positions in groups.values():
            for start in range(0, len(positions), self.batch_size):
                chunk = positions[start:start + self.batch_size]
                batch = np.stack([
                    np.stack([inputs[p].image_canvas, (inputs[p].region_canvas > 0).astype(np.float32)])
                    for p in chunk
                ]).astype(np.float32)
                stack = self._view_stack(batch, mode)
                for j, position in enumerate(chunk):
                    fine_input = inputs[position]
                    x0, y0, x1, y1 = fine_input.crop_coords
                    oy, ox = fine_input.offsets
                    crop_stack = stack[:, j, oy:oy + y1 - y0, ox:ox + x1 - x0]
                    uncertainty = float(np.var(crop_stack.astype(np.float32), axis=0).mean())
                    if mode == "4view":
                        probability = crop_stack.mean(axis=0, dtype=np.float32)
                    else:
                        ordered = np.sort(crop_stack, axis=0)
                        probability = ((ordered[3] + ordered[4]) / 2.0).astype(np.float16).astype(np.float32)
                    results[position] = FineCrop(
                        tuple(fine_input.crop_coords), probability.copy(),
                        (fine_input.region_canvas[oy:oy + y1 - y0, ox:ox + x1 - x0] > 0).copy(),
                        uncertainty,
                    )
                del batch, stack
        return results


@dataclass(frozen=True, eq=False)
class Candidate:
    mask: SparseMask
    features: np.ndarray        # (8,) float32
    confidence: float
    component_index: int

    def __post_init__(self):
        object.__setattr__(self, "features", np.asarray(self.features, dtype=np.float32))

    @property
    def area(self):
        return self.mask.area


class CandidateBatch:
    """All candidates of one image (compact) + lazily cached scorer utilities."""

    _uids = itertools.count(1)

    def __init__(self, name, shape, candidates):
        self.name, self.shape, self.candidates = name, tuple(shape), list(candidates)
        self.uid = next(self._uids)
        self._utility = {}

    def __len__(self):
        return len(self.candidates)

    @property
    def features(self):
        if not self.candidates:
            return np.zeros((0, len(FEATURE_NAMES)), dtype=np.float32)
        return np.vstack([c.features for c in self.candidates])

    @property
    def confidences(self):
        return np.array([c.confidence for c in self.candidates], dtype=float)

    def utility(self, scorer):
        """Expected-IoU utility of every candidate, computed once per explicit scorer."""
        if scorer is None:
            raise ValueError("Candidate utility requires an explicit scorer; no unscored fallback")
        if scorer.uid not in self._utility:
            self._utility[scorer.uid] = scorer.predict(self.features) if self.candidates else np.zeros(0)
        return self._utility[scorer.uid]

    @classmethod
    def from_components(cls, name, shape, components):
        """Coarse-only view: every coarse component is one candidate."""
        return cls(name, shape, [
            Candidate(c.mask, np.zeros(len(FEATURE_NAMES), dtype=np.float32), 1.0, i)
            for i, c in enumerate(components)
        ])


class CandidateGenerator:
    """Coarse components -> ordinary fine inference -> CV-label-ordered instances."""

    def __init__(self, coarse, fine, settings):
        self.coarse, self.fine, self.settings = coarse, fine, settings

    def generate(self, prepared):
        """ScorerBuilder's invariant baseline: ordinary four-view mean, NEVER routed."""
        components = self.coarse.components(prepared)
        crops = self.fine.infer(prepared, components) if components else []
        return self.from_crops(prepared, crops)

    def from_crops(self, prepared, crops, component_offset=0):
        """Build a batch from aligned crop slots; None slots keep their component indices."""
        candidates = []
        for component_index, crop in enumerate(crops):
            if crop is None:
                continue
            x0, y0, _, _ = crop.crop_coords
            probability = np.asarray(crop.prob_crop, dtype=np.float32)
            binary = ((probability > self.settings.fine_threshold) & crop.region_crop).astype(np.uint8)
            num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
            # Preserve OpenCV label order, matching the source experiment's stable tie ordering.
            for label in range(1, num_labels):
                if int(stats[label, cv2.CC_STAT_AREA]) < self.settings.min_candidate_area:
                    continue
                x, y = int(stats[label, cv2.CC_STAT_LEFT]), int(stats[label, cv2.CC_STAT_TOP])
                w, h = int(stats[label, cv2.CC_STAT_WIDTH]), int(stats[label, cv2.CC_STAT_HEIGHT])
                local = np.ascontiguousarray((labels[y:y + h, x:x + w] == label).astype(np.uint8))
                prob_local = probability[y:y + h, x:x + w]
                candidates.append(Candidate(
                    mask=SparseMask(y0 + y, x0 + x, local),
                    features=np.asarray(extract_candidate_features(
                        local, prob_local, x0 + x, y0 + y, prepared.shape,
                    ), dtype=np.float32),
                    confidence=float(prob_local[local > 0].mean()),
                    component_index=component_offset + component_index,
                ))
        return CandidateBatch(prepared.name, prepared.shape, candidates)


# ------------------------------------------------------------------
# Selection policies (Strategy)
# ------------------------------------------------------------------
class SelectionPolicy(abc.ABC):
    label = "policy"

    @abc.abstractmethod
    def rank(self, batch):
        """Indices into batch.candidates, best first, already filtered."""


class ConfidencePolicy(SelectionPolicy):
    """Explicit no-scorer baseline, never selected automatically."""
    label = "confidence (no scorer)"

    def rank(self, batch):
        return sorted(range(len(batch)), key=lambda i: -batch.candidates[i].confidence)


class OrderPreservingPolicy(SelectionPolicy):
    label = "as-proposed"

    def rank(self, batch):
        return list(range(len(batch)))


class UtilityThresholdPolicy(SelectionPolicy):
    """Keep utility >= tau, then rank stably by utility (the PQ/2 rule)."""

    def __init__(self, scorer, tau):
        if scorer is None:
            raise ValueError("UtilityThresholdPolicy requires an explicit scorer")
        self.scorer, self.tau = scorer, float(tau)
        self.label = f"scorer τ={self.tau:.3f}"

    def rank(self, batch):
        utility = batch.utility(self.scorer)
        kept = [i for i in range(len(batch)) if np.isfinite(utility[i]) and utility[i] >= self.tau]
        return sorted(kept, key=lambda i: -utility[i])


# ------------------------------------------------------------------
# Overlap removal + RLE
# ------------------------------------------------------------------
@dataclass(frozen=True, eq=False)
class ResolvedMask:
    candidate_index: int
    mask: SparseMask
    unchanged: bool


class OverlapResolver:
    """Greedy policy-ordered overlap resolution with final minimum-area pruning."""

    def __init__(self, min_area):
        self.min_area = min_area

    def resolve(self, batch, ranked_indices):
        occupied = np.zeros(batch.shape, dtype=bool)
        resolved = []
        for i in ranked_indices:
            candidate = batch.candidates[i].mask
            h, w = candidate.data.shape
            window = occupied[candidate.y0:candidate.y0 + h, candidate.x0:candidate.x0 + w]
            keep = candidate.data.astype(bool) & ~window
            area = int(keep.sum())
            if area >= self.min_area:
                window |= keep
                resolved.append(ResolvedMask(
                    i, SparseMask(candidate.y0, candidate.x0, keep.astype(np.uint8)),
                    unchanged=(area == candidate.area),
                ))
        return resolved


class RleEncoder:
    """COCO-RLE encoder with the original (batch uid, candidate index) memo contract."""

    def __init__(self, shape):
        self.shape = tuple(shape)
        self._memo = {}

    def encode(self, sparse, memo_key=None):
        if memo_key is not None and memo_key in self._memo:
            return self._memo[memo_key]
        from pycocotools import mask as mask_util
        full = np.zeros(self.shape, dtype=np.uint8, order="F")
        h, w = sparse.data.shape
        full[sparse.y0:sparse.y0 + h, sparse.x0:sparse.x0 + w] = sparse.data
        counts = mask_util.encode(full)["counts"].decode("utf-8")
        if memo_key is not None:
            self._memo[memo_key] = counts
        return counts


def mask_to_rle(mask):
    """RLE string of a full-size binary mask."""
    from pycocotools import mask as mask_util
    return mask_util.encode(np.asfortranarray(np.asarray(mask).astype(np.uint8)))["counts"].decode("utf-8")


PREDICTION_COLUMNS = ["filament_id", "segmentation_rle"]


class PredictionFrameBuilder:
    """Builder: population batches + one explicit policy -> submission frame."""

    def __init__(self, resolver, encoder):
        self.resolver, self.encoder = resolver, encoder

    def build(self, image_names, batches, policy, reporter=None):
        import pandas as pd
        rows = []
        for name in image_names:
            batch = batches[name]
            resolved = self.resolver.resolve(batch, policy.rank(batch))
            stem = Path(name).stem
            for k, item in enumerate(resolved, 1):
                key = (batch.uid, item.candidate_index) if item.unchanged else None
                rows.append({"filament_id": f"{stem}_{k}", "segmentation_rle": self.encoder.encode(item.mask, key)})
            if reporter:
                reporter.step()
        return pd.DataFrame(rows, columns=PREDICTION_COLUMNS)


# ------------------------------------------------------------------
# Immutable, crop-local recipe caches (no image/support input canvases)
# ------------------------------------------------------------------
def _save_recipe_crops(path, name, shape, crops, component_offset=0):
    payload, entries = {}, []
    for i, crop in enumerate(crops):
        if crop is None:
            continue
        payload[f"prob_{i}"] = np.asarray(crop.prob_crop, dtype=np.float32)
        payload[f"region_{i}"] = np.asarray(crop.region_crop, dtype=bool)
        entries.append({
            "index": i, "crop_coords": list(map(int, crop.crop_coords)),
            "uncertainty": float(crop.uncertainty),
        })
    payload["metadata_json"] = np.asarray(json.dumps({
        "name": name, "shape": list(map(int, shape)), "component_count": len(crops),
        "component_offset": int(component_offset), "entries": entries,
    }, allow_nan=False))
    with Path(path).open("xb") as handle:
        np.savez_compressed(handle, **payload)


def _load_recipe_crops(path):
    with np.load(path, allow_pickle=False) as data:
        metadata = json.loads(str(data["metadata_json"].item()))
        crops = [None] * metadata["component_count"]
        for entry in metadata["entries"]:
            i = entry["index"]
            crops[i] = FineCrop(
                tuple(entry["crop_coords"]), data[f"prob_{i}"].astype(np.float32),
                data[f"region_{i}"].astype(bool), float(entry["uncertainty"]),
            )
    return crops


def _recipe_mask_iou(a, b):
    x0, y0 = max(a.x0, b.x0), max(a.y0, b.y0)
    x1 = min(a.x0 + a.data.shape[1], b.x0 + b.data.shape[1])
    y1 = min(a.y0 + a.data.shape[0], b.y0 + b.data.shape[0])
    intersection = 0
    if x1 > x0 and y1 > y0:
        aa = a.data[y0 - a.y0:y1 - a.y0, x0 - a.x0:x1 - a.x0] > 0
        bb = b.data[y0 - b.y0:y1 - b.y0, x0 - b.x0:x1 - b.x0] > 0
        intersection = int(np.logical_and(aa, bb).sum())
    union = a.area + b.area - intersection
    return float(intersection / union) if union else 0.0


def _append_recipe_batch(primary, extra):
    if not len(extra):
        return primary
    return CandidateBatch(primary.name, primary.shape, primary.candidates + extra.candidates)


# ------------------------------------------------------------------
# Facade + whole-population recipe orchestration
# ------------------------------------------------------------------
class FilamentPipeline:
    """Population extraction shares one primary coarse/four-mean pass across recipes.

    CoarseProxy must implement components, cached-softmap components_at, peek, clear,
    and stats. It remains owned by the parent; no model loading/training occurs here.
    recipe_metadata[label] records the latest extraction for that label, while every
    extraction's immutable disk artifacts live in a fresh workspace subdirectory.
    """

    def __init__(self, settings, coarse, generator):
        self.settings, self.coarse, self.generator = settings, coarse, generator
        self.recipe_metadata = {}
        self._validate_recipe_settings()
        self.select_recipe(settings.final_recipe)

    def _validate_recipe_settings(self):
        if self.settings.fine_base_tta != "4view":
            raise ValueError("The ordinary/scorer baseline must use four-view mean")
        if self.settings.routed_aggregation != "midpoint_median":
            raise ValueError("N4 requires midpoint_median aggregation")
        if np.dtype(self.settings.routed_probability_dtype) != np.dtype("float16"):
            raise ValueError("Measured N4 requires float16 probability rounding")
        fraction = float(self.settings.routed_fraction)
        if not math.isfinite(fraction) or not 0.0 <= fraction <= 1.0:
            raise ValueError("routed_fraction must be finite and in [0, 1]")

    def _check_recipe(self, name):
        if name not in RECIPE_NAMES:
            raise ValueError(f"Unknown recipe {name!r}; supported recipes: {RECIPE_NAMES}")
        if name in _RESIDUAL_RECIPES and not self.settings.enable_residual_challenger:
            raise ValueError(f"Recipe {name!r} requires enable_residual_challenger=True")
        return name

    def select_recipe(self, name):
        """Set the active recipe after explicit scored validation (or a configured choice)."""
        self.active_recipe = self._check_recipe(name)
        return self

    def candidates(self, image_dir, name):
        """Compatibility single-image extraction; N4 routing depends on this singleton population."""
        return self.extract(image_dir, [name], "single_image_candidates")[name]

    def extract(self, image_dir, names, label, recipe=None):
        """Extract the active/explicit recipe over ALL supplied names before any selection.

        Validation and test use the identical algorithm. N4's cutoff is derived from
        the supplied split's own unlabeled crops, never validation/test ground truth.
        """
        chosen = self._check_recipe(self.active_recipe if recipe is None else recipe)
        return self.extract_recipes(image_dir, names, label, recipes=(chosen,))[chosen]

    def _residual_components(self, prepared, primary):
        secondary = self.coarse.components_at(prepared, threshold=self.settings.residual_threshold)
        residual = [
            component for component in secondary
            if all(_recipe_mask_iou(component.mask, p.mask) < self.settings.residual_iou_limit for p in primary)
        ]
        residual.sort(key=lambda c: (-float(c.mean_probability), -c.mask.area))
        return residual[:max(0, int(self.settings.residual_max_added))]

    def _release_periodically(self, image_index):
        every = int(self.settings.empty_cache_every)
        if every > 0 and image_index % every == 0:
            release_memory()

    def extract_recipes(self, image_dir, names, label, recipes=RECIPE_NAMES):
        """Return recipe -> name -> CandidateBatch; score/resolve each JOINT batch later.

        n1_standard: primary four-view mean.
        n4_midpoint: global top-ceil(fraction*n_primary_crops) get eight-view midpoint.
        n3_rescue: N1 plus ranked residual proposals, all using ordinary four-view mean.
        n4_rescue: N4 plus those same residuals; NEW, UNMEASURED validation challenger.
        No recipe selects a scorer, changes tau, resolves an arm separately, or falls back.
        """
        self._validate_recipe_settings()
        recipes = tuple(recipes)
        if not recipes or len(set(recipes)) != len(recipes):
            raise ValueError("Provide at least one recipe, without duplicates")
        for recipe in recipes:
            self._check_recipe(recipe)
        names = [str(name) for name in names]
        if len(set(names)) != len(names):
            raise ValueError("Population image names must be unique")
        wants_routing = any(recipe in _ROUTED_RECIPES for recipe in recipes)
        wants_residual = any(recipe in _RESIDUAL_RECIPES for recipe in recipes)
        if wants_routing and len(names) == 1:
            warnings.warn(
                "N4 extraction received a one-image population: routing is population-dependent. "
                "For validation/test export, call extract once with the entire split and reuse its batches.",
                RuntimeWarning, stacklevel=2,
            )

        safe_label = "".join(c if c.isalnum() or c in "-_" else "_" for c in str(label)) or "population"
        cache_root = Path(workspace.path("recipe_cache", safe_label))
        cache_root.mkdir(parents=True, exist_ok=True)
        cache_dir = Path(tempfile.mkdtemp(prefix="population_", dir=str(cache_root)))
        for subdir in ("primary", "routed", "residual"):
            (cache_dir / subdir).mkdir()

        base_batches, extra_batches, records, uncertainty_rows = {}, {}, {}, []
        reporter = ProgressReporter(f"Candidates [{label}] primary/4mean", len(names), every=20)
        candidate_count = 0
        for image_index, name in enumerate(names):
            prepared = load_prepared(image_dir, name)
            primary = list(self.coarse.components(prepared))
            crops = self.generator.fine.infer(prepared, primary) if primary else []
            if len(crops) != len(primary) or any(crop is None for crop in crops):
                raise RuntimeError(f"Incomplete primary fine inference for {name}")
            base_batches[name] = self.generator.from_crops(prepared, crops)
            primary_path = cache_dir / "primary" / f"{image_index:06d}.npz"
            _save_recipe_crops(primary_path, name, prepared.shape, crops)
            for component_index, crop in enumerate(crops):
                if not math.isfinite(crop.uncertainty):
                    raise ValueError(f"Non-finite crop uncertainty: {name}/{component_index}")
                uncertainty_rows.append({
                    "name": name, "component_index": component_index, "uncertainty": float(crop.uncertainty),
                })
            del crops

            residual = self._residual_components(prepared, primary) if wants_residual else []
            extra_crops = self.generator.fine.infer(prepared, residual) if residual else []
            extra_batches[name] = self.generator.from_crops(prepared, extra_crops, component_offset=len(primary))
            if wants_residual:
                _save_recipe_crops(
                    cache_dir / "residual" / f"{image_index:06d}.npz", name, prepared.shape,
                    extra_crops, component_offset=len(primary),
                )
            records[name] = {
                "primary": primary, "primary_path": primary_path, "shape": tuple(prepared.shape),
                "image_index": image_index, "added_proposals": len(residual),
            }
            candidate_count += len(base_batches[name])
            reporter.step(f"| base candidates: {candidate_count}")
            del prepared, extra_crops, residual
            self._release_periodically(image_index + 1)

        ranked = sorted(uncertainty_rows, key=lambda row: (-row["uncertainty"], row["name"], row["component_index"]))
        route_count = int(math.ceil(float(self.settings.routed_fraction) * len(ranked))) if wants_routing else 0
        routed_keys = {(row["name"], row["component_index"]) for row in ranked[:route_count]}
        route_by_name = defaultdict(list)
        for name, component_index in sorted(routed_keys):
            route_by_name[name].append(component_index)
        ranks = {(row["name"], row["component_index"]): i + 1 for i, row in enumerate(ranked)}
        with (cache_dir / "uncertainty_routing.csv").open("x", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=("name", "component_index", "uncertainty", "routing_rank", "routed"))
            writer.writeheader()
            for row in uncertainty_rows:
                key = (row["name"], row["component_index"])
                writer.writerow(dict(row, routing_rank=ranks[key], routed=int(key in routed_keys)))

        routed_batches = dict(base_batches)
        if route_by_name:
            routed_reporter = ProgressReporter(f"Candidates [{label}] routed/8view", len(route_by_name), every=20)
            for routed_image_index, (name, indices) in enumerate(sorted(route_by_name.items()), 1):
                record = records[name]
                # Retained sparse primary components guarantee identical geometry; never rerun coarse.
                prepared = load_prepared(image_dir, name)
                if tuple(prepared.shape) != record["shape"]:
                    raise ValueError(f"Prepared image shape changed between recipe passes: {name}")
                replacements = self.generator.fine.infer(
                    prepared, record["primary"], indices=indices, mode="8view",
                    aggregation=self.settings.routed_aggregation,
                )
                if len(replacements) != len(record["primary"]) or any(replacements[i] is None for i in indices):
                    raise RuntimeError(f"Incomplete routed fine inference for {name}")
                _save_recipe_crops(
                    cache_dir / "routed" / f"{record['image_index']:06d}.npz", name, prepared.shape, replacements,
                )
                crops = _load_recipe_crops(record["primary_path"])
                for component_index in indices:
                    replacement = replacements[component_index]
                    if replacement.crop_coords != crops[component_index].crop_coords:
                        raise ValueError(f"Fine crop geometry changed between recipe passes: {name}/{component_index}")
                    crops[component_index] = replacement
                routed_batches[name] = self.generator.from_crops(prepared, crops)
                routed_reporter.step(f"| upgraded crops: {len(indices)}")
                del prepared, replacements, crops
                self._release_periodically(routed_image_index)

        result = {}
        for recipe in recipes:
            primary_batches = routed_batches if recipe in _ROUTED_RECIPES else base_batches
            result[recipe] = {
                name: _append_recipe_batch(primary_batches[name], extra_batches[name])
                if recipe in _RESIDUAL_RECIPES else primary_batches[name]
                for name in names
            }
        added_by_name = {name: records[name]["added_proposals"] for name in names}
        metadata = {
            "label": str(label), "cache_dir": str(cache_dir), "population_size": len(names),
            "population_names": names, "select_recipe_on_validation": bool(self.settings.select_recipe_on_validation),
            "selection": "Explicit scored validation in the parent; no automatic recipe/scorer fallback",
            "routing": {
                "population": "This extraction's unlabeled primary crops only; residual crops excluded",
                "enabled": wants_routing, "total_crops": len(ranked), "routed_crops": route_count,
                "requested_fraction": float(self.settings.routed_fraction),
                "actual_fraction": route_count / len(ranked) if ranked else 0.0,
                "uncertainty_cutoff": float(ranked[route_count - 1]["uncertainty"]) if route_count else None,
                "ordering": "(-uncertainty, name, component_index)",
                "uncertainty": "mean(var(float32 aligned four-view probabilities), all unpadded pixels)",
                "base_tta": "4view/mean", "routed_tta": "8view/midpoint_median",
                "routed_probability_rounding": "float16 then float32 before features",
            },
            "residual": {
                "enabled": wants_residual, "threshold": float(self.settings.residual_threshold),
                "primary_mask_iou_strictly_below": float(self.settings.residual_iou_limit),
                "max_added_per_image": int(self.settings.residual_max_added),
                "ordering": "mean_probability descending, area descending, stable ties",
                "added_proposals": sum(added_by_name.values()), "added_proposals_by_name": added_by_name,
                "added_candidates": sum(len(batch) for batch in extra_batches.values()),
                "resolution": "Append to primary candidates; resolve jointly with the SAME explicit scorer/policy",
            },
            "recipes": {
                recipe: {
                    "candidate_count": sum(len(batch) for batch in batches.values()),
                    "routed_crops": route_count if recipe in _ROUTED_RECIPES else 0,
                    "added_proposals": sum(added_by_name.values()) if recipe in _RESIDUAL_RECIPES else 0,
                    "status": "NEW UNMEASURED validation challenger; not a measured N4 result"
                    if recipe == "n4_rescue" else "Source recipe arm; no performance claim for this run",
                }
                for recipe, batches in result.items()
            },
        }
        with (cache_dir / "summary.json").open("x") as handle:
            json.dump(metadata, handle, indent=2, allow_nan=False)
        self.recipe_metadata[str(label)] = metadata
        return result

    def coarse_components(self, image_dir, name):
        cached = self.coarse.peek(str(Path(image_dir) / name))
        return cached if cached is not None else self.coarse.components(load_prepared(image_dir, name))

    def predict_masks(self, image_dir, name, policy, batch=None):
        """Ranked, overlap-free uint8 masks, preferably from a cached population batch.

        With batch=None this calls extract([name]); N4 emits a warning because its
        routing population is then just one image. Core validation/test export must
        instead pass batches from extract(image_dir, ALL_NAMES, label) to the frame builder.
        """
        if batch is None:
            batch = self.extract(image_dir, [name], "single_image_prediction")[name]
        resolved = OverlapResolver(self.settings.min_prediction_area).resolve(batch, policy.rank(batch))
        return [r.mask.to_full(batch.shape) for r in resolved]

In [ ]:
def run_final_contract_tests():
    # D4 must round-trip rectangular canvases: square-only tests miss axis bugs.
    grid = np.arange(2 * 3 * 5, dtype=np.float32).reshape(2, 3, 5)
    for view in _FINE_VIEWS:
        np.testing.assert_array_equal(_fine_inverse(_fine_transform(grid, view), view), grid)
    # The fine loss must not broadcast [B,H,W] into [B,B,H,W].
    z = torch.zeros((2, 1, 5, 7))
    y = torch.zeros_like(z)
    region = torch.zeros((2, 5, 7)); region[0, :2] = 1; region[1, 3:] = 1
    assert torch.isfinite(fine_criterion(z, y, region))
    try:
        fine_criterion(z, y, torch.ones((2, 2, 5, 7)))
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid fine ROI shape was accepted")
    # Overlap ownership and RLE must describe the SAME final masks.
    a = np.zeros((12, 14), np.uint8); a[2:8, 2:8] = 1
    b = np.zeros_like(a); b[5:11, 5:11] = 1
    candidates = [Candidate(SparseMask.from_full(m), np.zeros(8), 1.0, i) for i,m in enumerate((a,b))]
    batch = CandidateBatch("synthetic.jpeg", a.shape, candidates)
    resolved = OverlapResolver(1).resolve(batch, [0,1])
    full = [item.mask.to_full(a.shape) for item in resolved]
    assert len(full) == 2 and not np.any(full[0] & full[1])
    encoder = RleEncoder(a.shape)
    for item, expected in zip(resolved, full):
        text = encoder.encode(item.mask)
        decoded = mask_util.decode({"size": list(a.shape), "counts": text.encode("utf-8")})
        np.testing.assert_array_equal(decoded, expected)
    assert extract_candidate_features(a, a.astype(np.float32), shape=a.shape).__len__() == 8
    overlap = pd.DataFrame([{"n_gt":2, "n_pred":2, "iou_matrix":np.array([[.8,.7],[.7,0.]])}])
    assert pq_counts(overlap)[1:] == (2,0,0)
    print("Final synthetic contracts passed: rectangular D4, ROI shapes, one-to-one PQ, overlaps and RLE.")

if settings.run_synthetic_tests:
    run_final_contract_tests()


## 9. Reusable eight-feature hurdle scorer

The measured best arm predicts `P(any annotator IoU>0.5) × E(max annotator IoU | eligible)` with an unweighted classifier/regressor pair. It is **not annotator consensus** and is trained on ordinary four-view candidates, as in the measured run. Default target geometry is all-polygon rounded OpenCV, matching the research bank; COCO targets are an explicit separate setting.

The first run regenerates training-side candidates and fits only small CPU estimators (historically ~19 minutes of candidate inference). Later runs reuse a matching `candidate_scorer_solarcrow10.pkl` bundle. The bundle contains plain sklearn components, schema, hashes and provenance—no fragile custom-class pickle. Old six-feature scorers cannot silently replace it. Fixed-checkpoint training-side candidates are not model-level OOF.

In [ ]:
# Replacement for JSON cell 27. Parent Settings defaults: scorer_mode="auto",
# scorer_target="hurdle_max", scorer_rasterizer="opencv_round", scorer_max_train_images=None,
# seed=42, default_tau=0.325;
# scorer_load_paths must contain only the new candidate_scorer_solarcrow10.pkl bundle.
# The parent generator emits ordinary four-view-mean candidates, never rescue candidates.
import inspect
import types
from importlib.metadata import PackageNotFoundError, version as package_version

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor


_SCORER_FORMAT = "solarcrow10_scorer_bundle_v1"
_SCORER_FEATURE_NAMES = (
    "log_area", "mean_prob", "max_prob", "std_prob", "aspect_ratio", "extent",
    "norm_radius", "tortuosity",
)
_SCORER_FEATURE_DEFINITIONS = {
    "log_area": "natural log(max(foreground pixel count, 1))",
    "mean_prob": "float32 foreground probabilities: numpy mean",
    "max_prob": "float32 foreground probabilities: numpy max",
    "std_prob": "float32 foreground probabilities: numpy std(ddof=0)",
    "aspect_ratio": "max(tight foreground bbox width,height)/min(width,height)",
    "extent": "foreground pixel count/(tight foreground bbox width*height)",
    "norm_radius": "hypot(global foreground centroid x-image_width/2, y-image_height/2)/(image_width/2)",
    "tortuosity": "sum(cv2.arcLength(external contours, closed=True))/(sqrt(foreground pixel count)+1e-6)",
}
_SCORER_PARAMS = dict(
    max_iter=220, max_leaf_nodes=31, min_samples_leaf=25,
    l2_regularization=1.0, random_state=42,
)
_SCORER_TARGETS = {"hurdle_max", "mean_eligible", "max_iou"}
_SCORER_PREPROCESSING_VERSION = "solarcrow10_otsu_disk_median_unscaled_mad_v1"
_SCORER_GT_VERSIONS = {
    "opencv_round": "opencv_all_polygons_float32_np_round_fillPoly_v1",
    "coco": "coco_full_annotation_frPyObjects_merge_v1",
}


def _scorer_rasterizer(settings):
    rasterizer = getattr(settings, "scorer_rasterizer", "opencv_round")
    if rasterizer not in _SCORER_GT_VERSIONS:
        raise ValueError(f"Unknown scorer_rasterizer={rasterizer!r}; expected {sorted(_SCORER_GT_VERSIONS)}")
    return rasterizer


def _scorer_gt_version(settings):
    return _SCORER_GT_VERSIONS[_scorer_rasterizer(settings)]


def _scorer_digest(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _scorer_code_spec(value):
    """Fingerprint actual implementations without notebook cell names or line numbers."""
    if isinstance(value, types.CodeType):
        return {
            "bytecode": value.co_code.hex(), "names": list(value.co_names),
            "varnames": list(value.co_varnames), "freevars": list(value.co_freevars),
            "cellvars": list(value.co_cellvars), "flags": value.co_flags,
            "args": [value.co_argcount, value.co_posonlyargcount, value.co_kwonlyargcount],
            "constants": [_scorer_code_spec(item) for item in value.co_consts],
        }
    if isinstance(value, (tuple, list)):
        return [_scorer_code_spec(item) for item in value]
    if isinstance(value, frozenset):
        return {"frozenset": sorted((_scorer_code_spec(item) for item in value), key=repr)}
    if isinstance(value, bytes):
        return {"bytes": value.hex()}
    if value is Ellipsis:
        return {"ellipsis": True}
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    raise TypeError(f"Unsupported implementation constant in scorer fingerprint: {type(value).__name__}")


def _scorer_callable_digest(function):
    function = inspect.unwrap(function)
    return _scorer_digest({
        "code": _scorer_code_spec(function.__code__),
        "defaults": _scorer_code_spec(function.__defaults__),
        "kwdefaults": {key: _scorer_code_spec(value)
                       for key, value in (function.__kwdefaults__ or {}).items()},
    })


def _scorer_class_digest(cls):
    methods = {}
    for name, item in sorted(vars(cls).items()):
        if isinstance(item, (staticmethod, classmethod)):
            item = item.__func__
        if inspect.isfunction(item):
            methods[name] = _scorer_callable_digest(item)
    return _scorer_digest(methods)


def _scorer_versions():
    versions = {
        "python": sys.version.split()[0], "numpy": np.__version__, "opencv": cv2.__version__,
        "torch": str(torch.__version__), "pandas": pd.__version__, "joblib": joblib.__version__,
    }
    for package in ("scikit-learn", "scipy", "pycocotools", "segmentation-models-pytorch"):
        try:
            versions[package] = package_version(package)
        except PackageNotFoundError:
            versions[package] = "unavailable"
    return versions


def _scorer_training_ids(settings, data_split):
    file_ids = list(data_split.train_file_ids)
    if len(file_ids) != len(set(file_ids)):
        raise ValueError("Scorer training split contains duplicate physical images")
    if set(file_ids) & set(data_split.validation_file_ids):
        raise ValueError("Scorer training split overlaps validation physical images")
    limit = getattr(settings, "scorer_max_train_images", None)
    if limit is not None:
        if isinstance(limit, bool) or not isinstance(limit, (int, np.integer)) or limit <= 0:
            raise ValueError("scorer_max_train_images must be None or a positive integer")
        file_ids = file_ids[:int(limit)]
    if not file_ids:
        raise ValueError("Scorer training split contains no physical images")
    return file_ids


def _scorer_annotation_mask(annotation, height, width, scorer_settings=None):
    """Default to the exact research cell-4 rasterizer; COCO is an explicit alternative.

    OpenCV rounds float32 vertices before filling every polygon. COCO polygon conversion
    can produce different boundary pixels/targets; neither setting guarantees historical PQ.
    """
    rasterizer = _scorer_rasterizer(settings if scorer_settings is None else scorer_settings)
    if rasterizer == "opencv_round":
        seg = annotation.get("segmentation", [])
        if isinstance(seg, dict):
            seg = dict(seg)
            if isinstance(seg.get("counts"), str):
                seg["counts"] = seg["counts"].encode("utf-8")
            decoded = mask_util.decode(seg)
            if decoded.ndim == 3:
                decoded = decoded.any(axis=2)
            if decoded.shape != (height, width):
                decoded = cv2.resize(decoded.astype(np.uint8), (width, height), interpolation=cv2.INTER_NEAREST)
            return decoded.astype(np.uint8)
        out = np.zeros((height, width), dtype=np.uint8)
        if isinstance(seg, list):
            polys = seg if seg and isinstance(seg[0], (list, tuple)) else [seg]
            for poly in polys:
                arr = np.asarray(poly, dtype=np.float32).reshape(-1, 2) if len(poly) else np.empty((0, 2))
                if len(arr) >= 3:
                    cv2.fillPoly(out, [np.round(arr).astype(np.int32)], 1)
        return out

    segmentation = annotation.get("segmentation", [])
    if isinstance(segmentation, dict):
        rle = dict(segmentation)
        if isinstance(rle.get("counts"), list):
            rle = mask_util.frPyObjects(rle, height, width)
        elif isinstance(rle.get("counts"), str):
            rle["counts"] = rle["counts"].encode("utf-8")
    elif isinstance(segmentation, list):
        if not segmentation:
            return np.zeros((height, width), dtype=np.uint8)
        polygons = segmentation if isinstance(segmentation[0], (list, tuple)) else [segmentation]
        rle = mask_util.merge(mask_util.frPyObjects(polygons, height, width))
    else:
        raise ValueError(f"Unsupported segmentation for annotation {annotation.get('id')}")
    mask = mask_util.decode(rle)
    if mask.ndim == 3:
        mask = mask.any(axis=2)
    if mask.shape != (height, width):
        raise ValueError(f"GT geometry mismatch: {mask.shape} != {(height, width)}")
    return np.ascontiguousarray(mask > 0, dtype=np.uint8)


def build_fingerprint(settings, coarse_handle, fine_handle, scheme):
    """Exact model, label, training-membership and ordinary-candidate provenance."""
    if tuple(FEATURE_NAMES) != _SCORER_FEATURE_NAMES:
        raise ValueError(f"Scorer requires these eight ordered features: {_SCORER_FEATURE_NAMES}")
    target = getattr(settings, "scorer_target", "hurdle_max")
    if target not in _SCORER_TARGETS:
        raise ValueError(f"Unknown scorer_target={target!r}; expected {sorted(_SCORER_TARGETS)}")
    training_ids = _scorer_training_ids(settings, split)
    records = []
    for file_id in training_ids:
        record_ids = list(index.file_to_image_ids[file_id])
        if not record_ids:
            raise ValueError(f"Physical training image {file_id} has no annotator records")
        records.append([str(file_id), sorted(map(str, record_ids))])
    hashes = {}
    for name, handle in (("coarse", coarse_handle), ("fine", fine_handle)):
        digest = handle.sha256
        if not digest:
            digest = sha256_file(handle.checkpoint_path)
        if not isinstance(digest, str) or re.fullmatch(r"[0-9a-fA-F]{64}", digest) is None:
            raise ValueError(f"Missing/invalid {name} checkpoint SHA-256; cannot verify scorer reuse")
        hashes[f"{name}_sha256"] = digest.lower()
    fingerprint = {
        "format": _SCORER_FORMAT, **hashes,
        "image_shape": list(settings.image_shape),
        "coarse_resize": settings.coarse_resize, "coarse_threshold": settings.coarse_threshold,
        "coarse_min_area": settings.coarse_min_area, "coarse_closing_kernel": settings.coarse_closing_kernel,
        "coarse_use_tta": settings.coarse_use_tta, "fine_threshold": settings.fine_threshold,
        "min_candidate_area": getattr(settings, "min_candidate_area", 20),
        "min_final_area": settings.min_prediction_area,
        "expansion": scheme.expansion, "dilation_size": scheme.dilation_size,
        "width_buckets": list(scheme.width_buckets), "height_buckets": list(scheme.height_buckets),
        "feature_names": list(FEATURE_NAMES), "feature_dtype": "float32",
        "feature_definitions": _SCORER_FEATURE_DEFINITIONS,
        "feature_implementation_sha256": _scorer_callable_digest(extract_candidate_features),
        "preprocessing_version": _SCORER_PREPROCESSING_VERSION,
        "preprocessing_implementations": {
            name: _scorer_callable_digest(function)
            for name, function in (("prepare_image", prepare_image), ("normalise_array", normalise_array),
                                   ("get_disk_mask", get_disk_mask), ("otsu_input", _otsu_input))
        },
        "fine_input_implementation_sha256": _scorer_class_digest(FineInputBuilder),
        "fine_segmenter_implementation_sha256": _scorer_class_digest(FineSegmenter),
        "candidate_generator_implementation_sha256": _scorer_class_digest(CandidateGenerator),
        "training_map": {
            "name": "ordinary_4mean", "views": ["identity", "hflip", "vflip", "rot180"],
            "aggregation": "arithmetic_mean_of_aligned_sigmoid_probabilities",
            "probability_dtype": "float32", "rescue": False, "uncertainty_routing": False,
        },
        "scorer_target": target, "scorer_params": _SCORER_PARAMS, "sample_weight": None,
        "scorer_fit_implementation_sha256": _scorer_callable_digest(CandidateScorer.fit.__func__),
        "target_builder_implementation_sha256": _scorer_callable_digest(ScorerTrainingSetBuilder.build),
        "target_iou_threshold": 0.5, "target_eligibility": "strict_greater_than",
        "scorer_rasterizer": _scorer_rasterizer(settings),
        "gt_rasterizer": _scorer_gt_version(settings),
        "gt_implementation_sha256": _scorer_callable_digest(_scorer_annotation_mask),
        "annotations_sha256": sha256_file(dataset_paths.annotations),
        "train_split_membership_sha256": _scorer_digest(sorted(map(str, split.train_file_ids))),
        "validation_split_membership_sha256": _scorer_digest(sorted(map(str, split.validation_file_ids))),
        "fit_membership_sha256": _scorer_digest(sorted(map(str, training_ids))),
        "fit_file_order_sha256": _scorer_digest(list(map(str, training_ids))),
        "fit_record_list_sha256": _scorer_digest(sorted(records)),
        "fit_physical_images": len(training_ids), "fit_annotator_records": sum(len(row[1]) for row in records),
        "scorer_max_train_images": getattr(settings, "scorer_max_train_images", None),
        "model_level_oof": False, "versions": _scorer_versions(),
    }
    return json.loads(json.dumps(fingerprint, allow_nan=False))


class CandidateScorer:
    """Shape-checked utility wrapper; only sklearn components/numeric constants are pickled."""

    FORMAT = _SCORER_FORMAT
    _uids = itertools.count(1)

    def __init__(self, estimator, fingerprint=None, meta=None, source=None):
        if not isinstance(fingerprint, dict) or fingerprint.get("format") != self.FORMAT:
            raise ValueError("A verified solarcrow10 fingerprint is required; legacy scorers are unsupported")
        if fingerprint.get("feature_names") != list(_SCORER_FEATURE_NAMES):
            raise ValueError("Scorer fingerprint does not declare the required eight-feature order")
        self.target = fingerprint.get("scorer_target")
        if self.target not in _SCORER_TARGETS:
            raise ValueError(f"Unsupported scorer target: {self.target!r}")
        if not isinstance(estimator, dict):
            raise ValueError("Scorer components must be a plain dict, not a custom estimator wrapper")
        self.estimator = self.components = dict(estimator)
        self.fingerprint, self.meta, self.source = fingerprint, dict(meta or {}), source
        self.uid = next(self._uids)
        self._validate_components()

    def _validate_components(self):
        expected = ({"classifier", "classifier_constant", "conditional_regressor", "conditional_constant"}
                    if self.target == "hurdle_max" else {"regressor"})
        if set(self.components) != expected:
            raise ValueError(f"Malformed {self.target} component keys: {sorted(self.components)}")
        if self.target == "hurdle_max":
            for name, constant_name, cls in (
                ("classifier", "classifier_constant", HistGradientBoostingClassifier),
                ("conditional_regressor", "conditional_constant", HistGradientBoostingRegressor),
            ):
                model, constant = self.components[name], self.components[constant_name]
                if model is None:
                    if not isinstance(constant, (int, float)) or not np.isfinite(constant) or not 0 <= constant <= 1:
                        raise ValueError(f"Invalid numeric fallback for {name}")
                else:
                    if constant is not None or not isinstance(model, cls):
                        raise ValueError(f"Invalid sklearn component {name}")
                    if getattr(model, "n_features_in_", None) != len(_SCORER_FEATURE_NAMES):
                        raise ValueError(f"{name} was not fitted on all eight features")
            classifier = self.components["classifier"]
            if classifier is not None and not np.array_equal(classifier.classes_, [0, 1]):
                raise ValueError("Hurdle classifier must have classes [0, 1]")
        else:
            model = self.components["regressor"]
            if not isinstance(model, HistGradientBoostingRegressor) or getattr(model, "n_features_in_", None) != 8:
                raise ValueError("Regressor must be a fitted eight-feature HistGradientBoostingRegressor")

    @staticmethod
    def _matrix(features):
        matrix = np.asarray(features, dtype=np.float32)
        if matrix.ndim != 2 or matrix.shape[1] != len(_SCORER_FEATURE_NAMES):
            raise ValueError(f"Expected features shaped (n,8), received {matrix.shape}")
        if not np.isfinite(matrix).all():
            raise ValueError("Scorer features contain NaN or Inf")
        return matrix

    @staticmethod
    def _prediction(values, count, name):
        values = np.asarray(values, dtype=np.float64)
        if values.shape != (count,) or not np.isfinite(values).all():
            raise ValueError(f"{name} produced invalid shape/nonfinite predictions: {values.shape}")
        return np.clip(values, 0.0, 1.0)

    def predict(self, features):
        matrix = self._matrix(features)
        count = len(matrix)
        if not count:
            return np.empty(0, dtype=np.float32)
        if self.target != "hurdle_max":
            utility = self._prediction(self.components["regressor"].predict(matrix), count, "regressor")
        else:
            classifier = self.components["classifier"]
            if classifier is None:
                probability = np.full(count, self.components["classifier_constant"], dtype=np.float64)
            else:
                probabilities = np.asarray(classifier.predict_proba(matrix), dtype=np.float64)
                if probabilities.shape != (count, 2) or not np.isfinite(probabilities).all():
                    raise ValueError("Hurdle classifier returned invalid predict_proba output")
                probability = probabilities[:, 1]
            conditional = self.components["conditional_regressor"]
            conditional = (conditional.predict(matrix) if conditional is not None else
                           np.full(count, self.components["conditional_constant"], dtype=np.float64))
            utility = (self._prediction(probability, count, "classifier") *
                       self._prediction(conditional, count, "conditional regressor"))
        return self._prediction(utility, count, "utility").astype(np.float32)

    def mismatch_with(self, fingerprint):
        # Lifecycle provenance does not change candidates once weights and input distribution match.
        keys = (set(self.fingerprint) | set(fingerprint)) - {"coarse_mode", "fine_mode"}
        return sorted(key for key in keys if self.fingerprint.get(key) != fingerprint.get(key))

    def to_bundle(self):
        return {
            "format": self.FORMAT, "feature_names": list(_SCORER_FEATURE_NAMES), "target": self.target,
            "components": self.components, "fingerprint": self.fingerprint, "meta": self.meta,
        }

    @classmethod
    def from_file(cls, path):
        bundle = joblib.load(path)
        if not isinstance(bundle, dict) or bundle.get("format") != cls.FORMAT:
            raise ValueError("Not a solarcrow10 scorer bundle; old six-feature/unverified scorers cannot be reused")
        if bundle.get("feature_names") != list(_SCORER_FEATURE_NAMES):
            raise ValueError("Bundle feature order is not solarcrow10's eight-feature schema")
        fingerprint = bundle.get("fingerprint")
        if not isinstance(fingerprint, dict) or bundle.get("target") != fingerprint.get("scorer_target"):
            raise ValueError("Bundle target and fingerprint disagree")
        return cls(bundle["components"], fingerprint, bundle.get("meta"), source=str(path))

    @classmethod
    def fit(cls, X, y, fingerprint, meta):
        matrix = cls._matrix(X)
        if not len(matrix):
            raise RuntimeError("No training candidates; cannot fit the scorer or continue unscored")
        maximum = np.asarray(y["max_iou"], dtype=np.float32)
        consensus = np.asarray(y["mean_eligible_iou"], dtype=np.float32)
        for name, values in (("max_iou", maximum), ("mean_eligible_iou", consensus)):
            if values.shape != (len(matrix),) or not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
                raise ValueError(f"Invalid training target {name}")
        positive = maximum > 0.5
        target = fingerprint["scorer_target"]
        if target == "hurdle_max":
            labels = positive.astype(np.int32)
            components = {
                "classifier": None, "classifier_constant": None,
                "conditional_regressor": None, "conditional_constant": None,
            }
            if len(np.unique(labels)) == 1:
                components["classifier_constant"] = float(labels[0])
            else:
                components["classifier"] = HistGradientBoostingClassifier(**_SCORER_PARAMS).fit(matrix, labels)
            n_positive = int(positive.sum())
            if n_positive < 2:
                components["conditional_constant"] = float(maximum[positive][0]) if n_positive else 0.0
            else:
                components["conditional_regressor"] = HistGradientBoostingRegressor(**_SCORER_PARAMS).fit(
                    matrix[positive], maximum[positive],
                )
        elif target in {"mean_eligible", "max_iou"}:
            utility = consensus if target == "mean_eligible" else np.where(positive, maximum, 0.0)
            components = {"regressor": HistGradientBoostingRegressor(**_SCORER_PARAMS).fit(matrix, utility)}
        else:
            raise ValueError(f"Unknown scorer_target={target!r}")
        return cls(components, fingerprint, meta)


class ScorerTrainingSetBuilder:
    """Ordinary candidates on training physical images, with full per-record multi-polygon GT.

    Empty annotator records contribute zero to mean_eligible_iou. No candidate, image,
    class or annotator weighting is applied. These active-model predictions are not OOF.
    """

    FILENAME = "candidate_bank_training.csv"

    def __init__(self, pipeline, index, split, settings, image_dir):
        self.pipeline, self.index, self.split, self.settings, self.image_dir = pipeline, index, split, settings, image_dir
        self.bank = None
        self.meta = {}

    def build(self):
        file_ids = _scorer_training_ids(self.settings, self.split)
        reporter = ProgressReporter("Scorer fit — ordinary 4mean training images", len(file_ids), every=50)
        rows = []
        for file_id in file_ids:
            # train_split_dict omits empty records; physical membership is authoritative.
            records = list(self.index.file_to_image_ids[file_id])
            if not records:
                raise ValueError(f"No annotator records for physical training image {file_id}")
            filenames = {self.index.images_by_id[record]["file_name"] for record in records}
            if len(filenames) != 1:
                raise ValueError(f"Conflicting physical filenames for {file_id}: {sorted(filenames)}")
            filename = next(iter(filenames))
            prepared = load_prepared(self.image_dir, filename)
            batch = self.pipeline.generator.generate(prepared)
            if len(batch):
                height, width = prepared.shape
                gt_by_record = [
                    [SparseMask.from_full(_scorer_annotation_mask(
                        self.index.annotations_by_id[aid], height, width, self.settings,
                    ))
                     for aid in self.index.annotation_ids_by_image.get(record, [])]
                    for record in records
                ]
                for candidate in batch.candidates:
                    features = np.asarray(candidate.features, dtype=np.float32)
                    if features.shape != (8,) or not np.isfinite(features).all():
                        raise ValueError(f"Invalid eight-feature candidate for {file_id}: {features.shape}")
                    best_by_record = [max((candidate.mask.iou(gt) for gt in annotations), default=0.0)
                                      for annotations in gt_by_record]
                    maximum = float(max(best_by_record, default=0.0))
                    eligible = [value if value > 0.5 else 0.0 for value in best_by_record]
                    rows.append({
                        "physical_image": str(file_id), "filename": filename,
                        "target_max_iou": maximum if maximum > 0.5 else 0.0,
                        "mean_eligible_iou": float(np.mean(eligible)), "max_iou": maximum,
                        "is_tp": int(maximum > 0.5),
                        **dict(zip(_SCORER_FEATURE_NAMES, map(float, features))),
                        "n_annotator_records": len(records),
                        "n_annotators_with_eligible_match": sum(value > 0.5 for value in best_by_record),
                    })
            reporter.step(f"| candidate rows: {len(rows)}")
        if not rows:
            raise RuntimeError("No candidates from the training split's ordinary 4mean pipeline; scorer fit aborted")
        self.bank = pd.DataFrame(rows)
        X = self.bank[list(_SCORER_FEATURE_NAMES)].to_numpy(dtype=np.float32)
        y = {name: self.bank[name].to_numpy(dtype=np.float64)
             for name in ("max_iou", "target_max_iou", "mean_eligible_iou", "is_tp")}
        self.meta = {
            "physical_images_processed": len(file_ids), "n_train_candidates": len(rows),
            "physical_images_with_candidates": int(self.bank["physical_image"].nunique()),
            "positive_fraction": float(self.bank["is_tp"].mean()),
            "mean_target_max_iou": float(self.bank["target_max_iou"].mean()),
            "mean_eligible_iou": float(self.bank["mean_eligible_iou"].mean()),
        }
        return X, y


class ScorerProvider:
    """Reuse only matching versioned bundles, including in load mode; retrain is explicit."""

    FILENAME = "candidate_scorer_solarcrow10.pkl"

    def __init__(self, settings, workspace, locator, fingerprint, training_set_builder):
        self.settings, self.workspace, self.locator = settings, workspace, locator
        self.fingerprint, self.training_set_builder = fingerprint, training_set_builder

    def _candidate_files(self):
        explicit = [self.workspace.path(self.FILENAME), *getattr(self.settings, "scorer_load_paths", ())]
        discovered = self.locator.find_files(["candidate_scorer_solarcrow10*.pkl"])
        found = []
        for path in [*explicit, *discovered]:
            if path and os.path.isfile(path):
                path = os.path.abspath(str(path))
                if path not in found:
                    found.append(path)
        return found

    def _reuse(self):
        for path in self._candidate_files():
            try:
                scorer = CandidateScorer.from_file(path)
                differences = scorer.mismatch_with(self.fingerprint)
            except Exception as exc:
                print(f"  scorer {path}: REJECTED ({type(exc).__name__}: {str(exc)[:180]})")
                continue
            if not differences:
                print(f"  scorer {path}: complete fingerprint MATCH; reuse {scorer.target}")
                return scorer
            print(f"  scorer {path}: REJECTED; fingerprint differs in {differences}")
        return None

    def obtain(self):
        settings = self.settings
        if settings.scorer_mode not in {"auto", "load", "retrain"}:
            raise ValueError("scorer_mode must be auto, load or retrain")
        if not settings.use_scorer:
            print("use_scorer=False explicitly disables utility scoring; no scorer is fitted.")
            return None
        modes = {name: getattr(settings, f"{name}_mode") for name in ("coarse", "fine")}
        if any(mode in {"train", "fine_tune"} for mode in modes.values()):
            warnings.warn(
                f"Segmentation modes {modes}: scorer candidates may be in-sample predictions of trained/fine-tuned "
                "models, not model-level OOF. Validation-used checkpoint/tau selection is not an unbiased estimate."
            )
        else:
            print("Segmentation checkpoints are loaded; their external training/validation membership is unknown.")
        print("Scorer targets use training physical images only; model_level_oof=False. "
              "The historical default tau=0.325 is validation-tuned, not an independent test calibration.")
        rasterizer = _scorer_rasterizer(settings)
        print(f"Scorer GT rasterization: {rasterizer} ({self.fingerprint['gt_rasterizer']})")
        if rasterizer == "opencv_round":
            print("Using the research cell-4 rasterization semantics; historical validation PQ is not guaranteed.")
        else:
            print("Using optional COCO rasterization: polygon boundary pixels/targets may differ from the "
                  "research cell-4 rounded OpenCV masks; historical validation PQ is not guaranteed.")
        scorer = None if settings.scorer_mode == "retrain" else self._reuse()
        if scorer is not None:
            # Reuse never regenerates candidates. Copy an accompanying bank only if its hash verifies.
            original_source = scorer.source
            bank_name = ScorerTrainingSetBuilder.FILENAME
            bank_destination = self.workspace.path(bank_name)
            expected_hash = scorer.meta.get("candidate_bank_sha256")
            bank_sources = [bank_destination, str(Path(original_source).parent / bank_name)]
            for candidate_bank in dict.fromkeys(bank_sources):
                if expected_hash and os.path.isfile(candidate_bank) and sha256_file(candidate_bank) == expected_hash:
                    if os.path.abspath(candidate_bank) != os.path.abspath(bank_destination):
                        self.workspace._atomic(bank_destination, lambda temp: shutil.copyfile(candidate_bank, temp))
                    print(f"Verified reused training bank: {bank_destination}")
                    break
            else:
                print("Original candidate-bank CSV is unavailable or mismatched; its saved hash/provenance is retained. "
                      "No replacement bank is fabricated on scorer reuse.")
            scorer.source = self.workspace.dump_joblib(scorer.to_bundle(), self.FILENAME)
            print(f"Reused scorer from {original_source} -> {scorer.source}; no candidate inference/refit.")
            return scorer
        if settings.scorer_mode == "load":
            raise FileNotFoundError("scorer_mode='load' requires a solarcrow10 bundle with an exact fingerprint match")

        print(f"Fitting unweighted {self.fingerprint['scorer_target']} scorer on ordinary 4mean candidates …")

        began = time.time()
        X, y = self.training_set_builder.build()
        if not len(X):
            raise RuntimeError("No training candidates; refusing a silent unscored fallback")
        bank_path = self.workspace.write_csv(self.training_set_builder.bank, ScorerTrainingSetBuilder.FILENAME)
        source_metadata = {
            "reference": "MAGFiLO_2026_zero_training_research_submission.ipynb cells 10/11; progress7_append.md 15.5",
            "reference_rasterizer": "all polygons, np.round vertices, cv2.fillPoly (research cell 4)",
            "scorer_rasterizer": rasterizer,
            "active_rasterizer": self.fingerprint["gt_rasterizer"],
            "rasterizer_matches_reference": rasterizer == "opencv_round",
            "rasterizer_semantic_difference": (
                "opencv_round uses the research cell-4 float32/np.round/cv2.fillPoly union of all polygons; "
                "coco uses frPyObjects/merge and may produce different boundary pixels and utility targets"
            ),
            "reference_validation_result_not_guaranteed": True,
            "target_definitions": {
                "hurdle_max": "P(any annotator best IoU>0.5)*E(max annotator IoU | max IoU>0.5), not consensus",
                "max_iou": "max annotator IoU if strictly >0.5, else zero",
                "mean_eligible": "mean over all annotator records of their best IoU if >0.5, else zero; includes empty records",
            },
            "model_level_oof": False, "validation_targets_used_for_scorer_fit": False,
            "segmentation_modes": modes,
            "external_weight_training_membership": {name: "unknown" for name in modes},
            "in_run_segmentation_fitting": {
                name: mode in {"train", "fine_tune"} for name, mode in modes.items()
            },
            "coarse_checkpoint": str(coarse_handle.checkpoint_path),
            "fine_checkpoint": str(fine_handle.checkpoint_path),
            "train_split_membership_sha256": self.fingerprint["train_split_membership_sha256"],
            "fit_record_list_sha256": self.fingerprint["fit_record_list_sha256"],
            "versions": self.fingerprint["versions"], "scorer_params": dict(_SCORER_PARAMS),
            "sample_weight": None, "fitted_target": self.fingerprint["scorer_target"],
            "default_tau": float(getattr(settings, "default_tau", 0.325)),
            "default_tau_source": "historical validation-selected N1 hurdle tau; no calibration fitted here",
            "candidate_bank_filename": ScorerTrainingSetBuilder.FILENAME,
            "candidate_bank_sha256": sha256_file(bank_path),
            **self.training_set_builder.meta,
            "created_utc": datetime.now(timezone.utc).isoformat(),
        }
        scorer = CandidateScorer.fit(X, y, self.fingerprint, source_metadata)
        scorer.meta["fit_seconds"] = round(time.time() - began, 3)
        sample = X[:min(32, len(X))]
        before = scorer.predict(sample)
        path = self.workspace.dump_joblib(scorer.to_bundle(), self.FILENAME)
        reloaded = CandidateScorer.from_file(path)  # CPU sklearn components only; no segmentation models reloaded.
        if reloaded.mismatch_with(self.fingerprint):
            raise RuntimeError("Saved scorer fingerprint changed during serialization")
        np.testing.assert_allclose(reloaded.predict(sample), before, rtol=0.0, atol=0.0)
        print(f"Saved candidate bank: {bank_path} ({len(X)} rows)")
        print(f"Saved scorer: {path}; CPU reload predictions match exactly on {len(sample)} rows")
        return reloaded


with run_log.stage("Inference engine + scorer"):
    fine_segmenter = FineSegmenter(fine_handle.model, fine_input_builder, settings.fine_infer_batch_size)
    candidate_generator = CandidateGenerator(coarse_segmenter, fine_segmenter, settings)
    pipeline = FilamentPipeline(settings, coarse_segmenter, candidate_generator)

    fingerprint = build_fingerprint(settings, coarse_handle, fine_handle, bucket_scheme)
    scorer_provider = ScorerProvider(
        settings, workspace, locator, fingerprint,
        ScorerTrainingSetBuilder(pipeline, index, split, settings, dataset_paths.train_images),
    )
    try:
        scorer = scorer_provider.obtain()
    finally:
        coarse_segmenter.clear()  # Training-image proposals must not remain resident after scorer fitting.
    print("Active scorer:", None if scorer is None else {"uid": scorer.uid, "source": scorer.source, **scorer.meta})

## 10. Final development shortlist and explicit recipe selection

Default: compare N1, N4, N3 and the new N4+N3 combination at the **fixed historical hurdle tau 0.325**, reusing primary inference. Select the highest final pooled PQ; preserve every result. An optional small tau sweep is off by default. No claim of independent validation: this 141-image split has already guided many decisions. Test inference receives the selected recipe and policy unchanged.

In [ ]:
def build_gt_df(index, annotator_image_ids, shape):
    """Ground truth frame — full multi-polygon RLE per annotation, ids '<annotator-image>_<k>'."""
    wanted, counters, rows = set(annotator_image_ids), defaultdict(int), []
    h, w = shape
    for annotation in index.raw["annotations"]:
        image_id = annotation["image_id"]
        if image_id not in wanted:
            continue
        counters[image_id] += 1
        rle = mask_util.merge(mask_util.frPyObjects(annotation["segmentation"], h, w))
        rows.append({"filament_id": f"{image_id}_{counters[image_id]}", "segmentation_rle": rle["counts"].decode("utf-8")})
    return pd.DataFrame(rows, columns=PREDICTION_COLUMNS)


def _stem_of(filament_ids):
    return filament_ids.astype(str).str.rsplit("_", n=1).str[0]


class OverlapTableBuilder:
    """GT-vs-prediction IoU / Dice tables per annotator record, computed on RLEs (no dense decoding)."""

    def __init__(self, gt_df, shape, annotator_image_ids=()):
        self.shape = tuple(shape)
        self.gt_rles = {record: [] for record in annotator_image_ids}
        for annotator_image, group in gt_df.groupby(_stem_of(gt_df["filament_id"]), sort=False):
            self.gt_rles[annotator_image] = [self._rle(r) for r in group["segmentation_rle"]]

    def _rle(self, counts):
        return {"size": list(self.shape), "counts": str(counts).encode("utf-8")}

    def build(self, pred_df):
        pred_by_image = defaultdict(list)
        for stem, counts in zip(_stem_of(pred_df["filament_id"]), pred_df["segmentation_rle"]):
            pred_by_image[stem].append(self._rle(counts))
        rows = []
        for annotator_image, gt_rles in self.gt_rles.items():
            pred_rles = pred_by_image.get(annotator_image.split("-", maxsplit=1)[1], [])
            if gt_rles and pred_rles:
                iou = np.asarray(mask_util.iou(pred_rles, gt_rles, [0] * len(gt_rles)), dtype=np.float32).T   # [n_gt, n_pred]
            else:
                iou = np.zeros((len(gt_rles), len(pred_rles)), dtype=np.float32)
            dice = 2.0 * iou / (1.0 + iou)                      # Dice = 2·IoU / (1 + IoU)
            rows.append({"annotator_image": annotator_image, "iou_matrix": torch.from_numpy(iou),
                         "dice_matrix": torch.from_numpy(dice.astype(np.float32)),
                         "n_gt": len(gt_rles), "n_pred": len(pred_rles)})
        return pd.DataFrame(rows)


class ValidationHarness:
    """Scores any SelectionPolicy against the held-out ground truth using cached candidates."""

    def __init__(self, index, split, settings, frame_builder, image_names):
        self.image_names, self.frame_builder = image_names, frame_builder
        self.gt_df = build_gt_df(index, split.validation_split_dict.keys(), settings.image_shape)
        self.tables = OverlapTableBuilder(self.gt_df, settings.image_shape, split.validation_split_dict)

    def evaluate(self, batches, policy, keep_frames=False):
        pred_df = self.frame_builder.build(self.image_names, batches, policy)
        overlap_df = self.tables.build(pred_df)
        tp_iou, tp, fp, fn = pq_counts(overlap_df)
        denominator = tp + 0.5 * fp + 0.5 * fn
        result = {"pq": tp_iou / denominator if denominator > 0 else 0.0,
                  "sq": tp_iou / tp if tp else 0.0, "rq": tp / denominator if denominator else 0.0,
                  "tp": tp, "fp": fp, "fn": fn, "sum_matched_iou": tp_iou,
                  "precision": tp / (tp + fp) if tp + fp else 0.0,
                  "recall": tp / (tp + fn) if tp + fn else 0.0, "emitted": len(pred_df)}
        if keep_frames:
            result.update(pred_df=pred_df, overlap_df=overlap_df)
        return result


@dataclass
class SweepResult:
    table: pd.DataFrame
    best_tau: float
    best_pq: float
    baseline_pq: float            # same candidates, ranked by mean probability, no utility filtering


class TauOptimizer:
    """Strategy: grid sweep → optional local refinement → (optionally smoothed) arg-max."""

    def __init__(self, harness, settings):
        self.harness, self.settings = harness, settings

    def _select(self, rows):
        taus = np.array(sorted(rows))
        pq = np.array([rows[t]["pq"] for t in taus])
        window = self.settings.tau_smoothing_window
        score = pq
        if window > 1:
            padded = np.pad(pq, window // 2, mode="edge")
            score = np.convolve(padded, np.ones(window) / window, mode="valid")
        tied = taus[np.isclose(score, score.max(), rtol=0, atol=1e-12)]
        return float(tied[np.argsort(np.abs(tied - np.median(tied)))[0]])        # median of ties

    def sweep(self, scorer, batches):
        rows = {}

        def evaluate(tau):
            tau = round(float(tau), 4)
            if tau < 0 or tau in rows:
                return
            rows[tau] = self.harness.evaluate(batches, UtilityThresholdPolicy(scorer, tau))
            r = rows[tau]
            print(f"  τ={tau:.3f} | emitted={r['emitted']:5d} | TP={r['tp']:4d} FP={r['fp']:5d} FN={r['fn']:4d} | PQ={r['pq']:.4f}")

        print(f"τ sweep over {len(self.settings.tau_grid)} grid points …")
        for tau in self.settings.tau_grid:
            evaluate(tau)
        best = self._select(rows)
        step = self.settings.tau_refine_step
        if step and step > 0:
            print(f"Refining around τ={best:.3f} (step {step}) …")
            for k in range(-4, 5):
                evaluate(best + k * step)
            best = self._select(rows)

        baseline = self.harness.evaluate(batches, ConfidencePolicy())
        table = pd.DataFrame([{"tau": t, **rows[t]} for t in sorted(rows)])
        table["selected"] = np.isclose(table["tau"], best)
        print(f"Baseline (no scorer, rank by mean probability): PQ={baseline['pq']:.4f} | emitted={baseline['emitted']}")
        return SweepResult(table, best, float(rows[best]["pq"]), float(baseline["pq"]))


@optional_stage("τ sweep plot")
def plot_tau_sweep(result, workspace):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(result.table["tau"], result.table["pq"], marker="o", ms=3, label="validation PQ")
    ax.axvline(result.best_tau, color="crimson", linestyle="--", label=f"selected τ = {result.best_tau:.3f}")
    ax.axhline(result.baseline_pq, color="gray", linestyle=":", label=f"no-scorer baseline = {result.baseline_pq:.4f}")
    ax.set_xlabel("candidate rejection threshold τ"); ax.set_ylabel("PQ"); ax.grid(True, linestyle="--", alpha=0.5)
    ax.set_title("Utility-threshold sweep on held-out physical images"); ax.legend()
    workspace.save_figure(fig, "tau_sweep.png")
    show_figure(fig)


def validate_prediction_frame(pred_df, image_names, shape, decode_masks=False, check_non_overlap=True):
    """Schema / id / (optionally) decoded-mask validation. Raises ValueError on any violation."""
    if list(pred_df.columns) != PREDICTION_COLUMNS:
        raise ValueError(f"Expected columns {PREDICTION_COLUMNS}; got {list(pred_df.columns)}")
    if pred_df["filament_id"].isna().any():
        raise ValueError("Prediction table contains missing filament_id values")
    if pred_df["segmentation_rle"].isna().any():
        raise ValueError("Prediction table contains missing RLE values")
    if pred_df["filament_id"].duplicated().any():
        raise ValueError(f"Duplicate filament_id: {pred_df.loc[pred_df['filament_id'].duplicated(), 'filament_id'].iloc[0]}")
    if pred_df["segmentation_rle"].astype(str).str.len().eq(0).any():
        raise ValueError("Prediction table contains an empty RLE string")

    for filament_id in pred_df["filament_id"].astype(str):
        stem, separator, suffix = filament_id.rpartition("_")
        if not separator or not stem or not suffix.isdigit() or int(suffix) < 1:
            raise ValueError(f"Invalid filament ID suffix: {filament_id}")
    expected = {os.path.splitext(os.path.basename(name))[0] for name in image_names}
    stems = list(_stem_of(pred_df["filament_id"])) if not pred_df.empty else []
    produced = set(stems)
    unknown, missing = produced - expected, expected - produced
    if unknown:
        raise ValueError(f"Predictions contain unknown image stems: {sorted(unknown)[:5]}")
    print(f"Prediction images represented: {len(produced)}/{len(expected)} | images with zero emitted masks: {len(missing)}")

    if decode_masks and not pred_df.empty:
        work = pred_df.assign(_stem=stems)
        for stem, group in work.groupby("_stem", sort=False):
            occupied = np.zeros(shape, dtype=bool)
            for row in group.itertuples(index=False):
                decoded = mask_util.decode({"size": list(shape), "counts": str(row.segmentation_rle).encode("utf-8")})
                if decoded.shape != tuple(shape):
                    raise ValueError(f"Decoded mask {row.filament_id} has shape {decoded.shape}")
                mask = decoded.astype(bool)
                if not mask.any():
                    raise ValueError(f"Decoded mask is empty: {row.filament_id}")
                if check_non_overlap and np.logical_and(occupied, mask).any():
                    raise ValueError(f"Overlapping masks remain after cleanup for image {stem}")
                occupied |= mask
    return {"expected_images": len(expected), "predicted_images": len(produced), "images_with_zero_masks": len(missing), "zero_prediction_image_stems": sorted(missing)}

In [ ]:
with run_log.stage("Validation: shared recipe extraction"):
    validation_image_names = [index.file_name_of_physical(fid) for fid in split.validation_file_ids]
    recipes_to_check = (("n1_standard", "n4_midpoint", "n3_rescue", "n4_rescue")
                        if settings.enable_residual_challenger else ("n1_standard", "n4_midpoint"))
    if not settings.select_recipe_on_validation:
        recipes_to_check = (settings.final_recipe,)
    validation_recipe_batches = pipeline.extract_recipes(dataset_paths.train_images, validation_image_names,
                                                        "validation", recipes_to_check)
    frame_builder = PredictionFrameBuilder(OverlapResolver(settings.min_prediction_area), RleEncoder(settings.image_shape))
    harness = ValidationHarness(index, split, settings, frame_builder, validation_image_names)

with run_log.stage("Validation: final shortlist"):
    best_tau = settings.default_tau
    selection_policy = UtilityThresholdPolicy(scorer, best_tau) if scorer is not None else ConfidencePolicy()
    recipe_rows = []
    for recipe, batches in validation_recipe_batches.items():
        metrics = harness.evaluate(batches, selection_policy)
        recipe_rows.append({"recipe": recipe, "tau": best_tau, **metrics})
        print(f"{recipe:15s} | PQ={metrics['pq']:.6f} SQ={metrics['sq']:.6f} RQ={metrics['rq']:.6f} "
              f"TP={metrics['tp']} FP={metrics['fp']} FN={metrics['fn']}")
    # Preserve explicit priority on exact metric ties; select on development PQ, not row count.
    recipe_table = pd.DataFrame(recipe_rows)
    selected_recipe = str(max(recipe_rows, key=lambda row: row["pq"])["recipe"])
    pipeline.select_recipe(selected_recipe)
    validation_batches = validation_recipe_batches[selected_recipe]
    recipe_table["selected"] = recipe_table["recipe"] == selected_recipe
    workspace.write_csv(recipe_table, "final_recipe_comparison.csv")
    print(f"Selected final recipe: {selected_recipe}. This is validation-selected, NOT unbiased OOF.")

    sweep = None
    if scorer is not None and settings.auto_tune_tau:
        sweep = TauOptimizer(harness, settings).sweep(scorer, validation_batches)
        best_tau = sweep.best_tau
        workspace.write_csv(sweep.table, "tau_sweep_validation.csv")
        selection_policy = UtilityThresholdPolicy(scorer, best_tau)
        plot_tau_sweep(sweep, workspace)
    print("Exact policy handed to test:", selected_recipe, selection_policy.label)

with run_log.stage("Validation: selected final masks"):
    final = harness.evaluate(validation_batches, selection_policy, keep_frames=True)
    self_eval_gt_df, self_eval_pred_df, self_eval_overlap_df = harness.gt_df, final["pred_df"], final["overlap_df"]
    self_eval_pq = get_pq_score(self_eval_overlap_df)
    validation_frame_check = validate_prediction_frame(self_eval_pred_df, validation_image_names, settings.image_shape)
    print(f"Final development PQ={self_eval_pq:.6f}; official evaluator parity unverified.")
    workspace.write_csv(self_eval_gt_df, "self_eval_gt.csv")
    workspace.write_csv(self_eval_pred_df, "self_eval_pred.csv")
    workspace.write_pickle(self_eval_overlap_df, "self_eval_overlap.pkl")
    workspace.write_csv(pd.DataFrame([{k: v for k,v in final.items() if k not in ("pred_df", "overlap_df")}]), "self_eval_summary.csv")
    # Sparse selected candidates remain available for reports; discard other candidate sets before test.
    del validation_recipe_batches
    coarse_segmenter.clear()
    release_memory()

## 10.1 Detailed evaluation and diagnostic plots *(optional — failure-isolated, no model inference)*

Pooled PQ / SQ / RQ, per-station and per-image breakdowns, IoU and Dice distributions and the many-to-one / one-to-many relations — all computed from the in-memory overlap table.

In [ ]:
def plot_distribution(matrices, title, x_label):
    """Distribution of non-zero pairwise IoU or Dice scores."""
    scores = np.concatenate([_as_numpy(m).astype(float).ravel() for m in matrices] or [np.zeros(0)])
    scores = scores[scores > 0]
    fig = plt.figure(figsize=(8, 5))
    counts, edges = np.histogram(scores, bins=np.linspace(0.0, 1.0, 51))
    probabilities = counts / counts.sum() if counts.sum() > 0 else counts
    plt.bar((edges[:-1] + edges[1:]) / 2, probabilities, width=0.018, edgecolor="black", alpha=0.7)
    plt.grid(axis="y", linestyle="--")
    if scores.size > 0:
        plt.axvline(scores.mean(), linestyle="--", linewidth=1.5, label=f"Mean = {scores.mean():.3f}")
        plt.legend()
    plt.xlabel(x_label); plt.ylabel("Probability"); plt.title(title)
    plt.tight_layout()
    show_figure(fig)


def plot_m2n_counts(overlap_df, title="Many-to-One and One-to-Many Distribution"):
    """How many GT filaments and predictions overlap each other (any positive overlap)."""
    gt_counts, pred_counts = [], []
    for row in overlap_df.itertuples(index=False):
        n_gt, n_pred = int(row.n_gt), int(row.n_pred)
        if n_gt == 0:
            pred_counts.extend([0] * n_pred); continue
        if n_pred == 0:
            gt_counts.extend([0] * n_gt); continue
        hits = _as_numpy(row.iou_matrix) > 0
        gt_counts.extend(hits.sum(axis=1).tolist())
        pred_counts.extend(hits.sum(axis=0).tolist())

    gt_deg, pred_deg = Counter(gt_counts), Counter(pred_counts)
    offset = 0.5
    gt_bars = {d + offset: c for d, c in gt_deg.items()}
    pred_bars = {-(d + offset): c for d, c in pred_deg.items()}
    gt_miss, pred_miss = gt_deg.get(0, 0), pred_deg.get(0, 0)
    gt_hit = sum(c for d, c in gt_deg.items() if d > 0)
    pred_hit = sum(c for d, c in pred_deg.items() if d > 0)

    fig = plt.figure(figsize=(12, 6))
    for pos in sorted(gt_bars):
        plt.bar(pos, gt_bars[pos], width=0.8, color="crimson" if int(round(pos - offset)) == 0 else "forestgreen", edgecolor="black", alpha=0.7)
    for pos in sorted(pred_bars):
        plt.bar(pos, pred_bars[pos], width=0.8, color="darkorange" if int(round(-pos - offset)) == 0 else "steelblue", edgecolor="black", alpha=0.7)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.grid(axis="y", linestyle="--"); plt.xlabel("Matching Case"); plt.ylabel("Frequency")
    positions = sorted(list(gt_bars) + list(pred_bars))
    labels = [f"{abs(int(round(v + offset)))}:1" if v < 0 else f"1:{int(round(v - offset))}" for v in positions]
    plt.xticks(positions, labels, rotation=45, fontsize=9)
    plt.title(title)
    max_pos = max([abs(k) for k in positions], default=offset)
    plt.xlim(-max_pos - 1, max_pos + 1)
    plt.legend(handles=[Patch(facecolor="forestgreen", label=f"GT with overlaps: {gt_hit}"),
                        Patch(facecolor="crimson", label=f"GT with no overlap: {gt_miss}"),
                        Patch(facecolor="steelblue", label=f"Predictions with overlaps: {pred_hit}"),
                        Patch(facecolor="darkorange", label=f"Predictions with no overlap: {pred_miss}")], fontsize=9)
    plt.tight_layout()
    show_figure(fig)


@optional_stage("validation report")
def show_validation_report(overlap_df):
    print("=" * 70 + "\nTIER A1: COMPREHENSIVE METRIC READOUT\n" + "=" * 70)
    pooled, station_df, per_image_df = compute_detailed_metrics(overlap_df)
    print("\n--- [1] GLOBAL POOLED METRICS ---");              display(pd.DataFrame([pooled]).round(4))
    print("\n--- [2] PERFORMANCE BY OBSERVATORY STATION ---"); display(station_df.round(4))
    print("\n--- [3] PER-IMAGE PQ DISTRIBUTION ---")
    display(pd.DataFrame([{"Mean PQ": per_image_df["PQ"].mean(), "Median PQ": per_image_df["PQ"].median(),
                           "Std PQ": per_image_df["PQ"].std(), "Min PQ": per_image_df["PQ"].min(),
                           "Max PQ": per_image_df["PQ"].max(), "Zero-PQ Images": (per_image_df["PQ"] == 0.0).sum(),
                           "Total Images": len(per_image_df)}]).round(4))
    print("\n--- [4] WORST 5 PHYSICAL OBSERVATIONS (LOWEST PQ) ---")
    display(per_image_df.head(5)[["physical_image", "station", "PQ", "SQ", "RQ", "TP", "FP", "FN"]].round(4))
    plot_distribution(overlap_df["iou_matrix"], "IoU Distribution", "IoU")
    plot_distribution(overlap_df["dice_matrix"], "Dice Distribution", "Dice")
    plot_m2n_counts(overlap_df, "Many-to-One and One-to-Many Distribution")
    return pooled, station_df, per_image_df


with run_log.stage("Validation report"):
    report = show_validation_report(self_eval_overlap_df) if settings.run_validation_report else None
    validation_pooled, validation_station_df, validation_per_image_df = report if report else (None, None, None)

## 11. Selected-recipe test inference and exact serialized CSV audit

This cell uses `pipeline.active_recipe`, the same scorer and the same selected tau as validation. It does not fall back to the old research export. The CSV is written in this run's new directory, reloaded, decoded at 2048×2048 and checked for unique IDs, valid nonempty masks, known stems and zero within-image overlap. Zero-prediction images are reported, not forced to contain objects.

In [ ]:
class SubmissionExporter:
    """Builds, writes (atomically, output root) and re-validates submission.csv."""

    FILENAME = "submission.csv"

    def __init__(self, workspace, frame_builder, settings):
        self.workspace, self.frame_builder, self.settings = workspace, frame_builder, settings

    def export(self, image_names, batches, policy):
        reporter = ProgressReporter("Encoding test predictions", len(image_names), every=45)
        frame = self.frame_builder.build(image_names, batches, policy, reporter)
        policy_used = policy.label
        if frame.empty:
            raise RuntimeError("Selected recipe emitted zero masks across the entire test set. No silent unscored fallback; inspect validation/artifacts first.")

        shape = self.settings.image_shape
        in_memory_check = validate_prediction_frame(frame, image_names, shape, decode_masks=False)
        path = self.workspace.write_csv(frame, self.FILENAME)
        print("Saved:", path, "| rows:", len(frame), "| bytes:", os.path.getsize(path))

        reloaded = pd.read_csv(path, dtype={"filament_id": str, "segmentation_rle": str})
        if reloaded.values.tolist() != frame.astype(str).values.tolist():
            raise ValueError("submission.csv does not round-trip losslessly through the filesystem.")
        serialized_check = validate_prediction_frame(reloaded, image_names, shape, decode_masks=True, check_non_overlap=True)
        self.workspace.write_json(serialized_check, "submission_integrity_audit.json")

        root_sub = os.path.join("/kaggle/working", self.FILENAME)
        if os.path.isdir("/kaggle/working") and os.path.abspath(path) != os.path.abspath(root_sub):
            shutil.copyfile(path, root_sub + ".tmp")
            os.replace(root_sub + ".tmp", root_sub)
            print("Mirrored root submission for Kaggle UI:", root_sub)
            
        print("Serialized submission checks passed:", serialized_check)
        return {"path": path, "rows": int(len(frame)), "sha256": sha256_file(path), "policy": policy_used,
                "in_memory_check": in_memory_check, "serialized_check": serialized_check}


with run_log.stage("Test inference"):
    test_batches = pipeline.extract(dataset_paths.test_images, test_image_names, "test")
    print("Coarse stage cache:", coarse_segmenter.stats())

with run_log.stage("Submission export"):
    submission_exporter = SubmissionExporter(workspace, frame_builder, settings)
    submission_info = submission_exporter.export(test_image_names, test_batches, selection_policy)
    pred_df = pd.read_csv(submission_info["path"], dtype=str)
    print(pred_df.head())
    print("Number of test images:", len(test_image_names), "| submission rows:", submission_info["rows"])

In [ ]:
# ============================================================
# 12. RUN MANIFEST — immutable model config + machine-readable run summary
# ============================================================
with run_log.stage("Run manifest"):
    model_config = {
        "experiment_id": settings.experiment_id,
        "settings": dataclasses.asdict(settings), "selected_recipe": selected_recipe,
        "feature_names": list(FEATURE_NAMES), "scorer_bundle_sha256": sha256_file(getattr(scorer, "source", None)),
        "recipe_generation": pipeline.recipe_metadata,
        "scorer_target": settings.scorer_target, "scorer_rasterizer": settings.scorer_rasterizer,
        "training_bank_is_model_level_oof": False,
        "metric_version": "strict_eligible_one_to_one_pooled_v2",
        "external_checkpoint_training_membership": "unknown; reported provenance does not prove membership",
        "notebook_derived_from": {"path": "solarcrow9.ipynb", "sha256": "716755489ddf8aaaa88077177dc62ade487406644a6ca0fe0faabb02dbd3d248"},
        "coarse_mode": settings.coarse_mode, "fine_mode": settings.fine_mode,
        "coarse_checkpoint_path": coarse_handle.checkpoint_path, "coarse_checkpoint_sha256": coarse_handle.sha256,
        "fine_checkpoint_path": fine_handle.checkpoint_path, "fine_checkpoint_sha256": fine_handle.sha256,
        "fine_encoder_weights": fine_handle.encoder_weights, "coarse_encoder_weights": coarse_handle.encoder_weights,
        "split_path": os.path.abspath(split.path), "split_sha256": split.sha256, "split_origin": split.origin,
        "proposal_provenance": ("in-sample coarse proposals" if settings.coarse_mode == "train"
                                else "loaded coarse checkpoint; training-membership unknown unless separately documented"),
        "seed": settings.seed, "resize_multiplier": settings.coarse_resize,
        "expansion": settings.expansion, "dilation_size": settings.dilation_size,
        "coarse_threshold": settings.coarse_threshold, "coarse_min_area": settings.coarse_min_area,
        "coarse_closing_kernel": settings.coarse_closing_kernel, "coarse_use_tta": settings.coarse_use_tta,
        "fine_threshold": settings.fine_threshold, "min_prediction_area": settings.min_prediction_area,
        "fine_num_epochs": settings.fine_epochs, "coarse_num_epochs": settings.coarse_epochs,
        "width_buckets": list(bucket_scheme.width_buckets), "height_buckets": list(bucket_scheme.height_buckets),
        "scorer_source": getattr(scorer, "source", None), "scorer_fingerprint": getattr(scorer, "fingerprint", None),
        "scorer_meta": getattr(scorer, "meta", None), "tau_selected": best_tau if scorer is not None else None,
        "tau_auto_tuned": bool(sweep is not None), "image_height": settings.image_height, "image_width": settings.image_width,
    }
    config_path = workspace.write_json(model_config, f"model_config_{settings.experiment_id}.json", unique=True)

    manifest = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "selected_recipe": selected_recipe, "recipe_comparison": recipe_rows,
        "validation_selected": True, "validation_metrics": {k:v for k,v in final.items() if k not in ("pred_df", "overlap_df")},
        "versions": _scorer_versions(),
        "validation_pq": self_eval_pq,
        "validation_pq_no_scorer_baseline": None if sweep is None else sweep.baseline_pq,
        "selected_tau": best_tau if scorer is not None else None,
        "submission": submission_info,
        "coarse_stage_cache": coarse_segmenter.stats(),
        "tta_self_test_max_abs_diff": tta_self_test_diff,
        "stage_seconds": run_log.stage_seconds, "optional_stage_failures": run_log.optional_failures,
        "optional_stages_skipped_for_budget": run_log.optional_skipped,
        "elapsed_seconds": round(run_log.elapsed(), 1), "device": str(device),
        "torch": torch.__version__, "config_path": config_path,
    }
    manifest_path = workspace.write_json(manifest, "run_manifest.json")

    print("Run artifact inventory:")
    for path in [coarse_handle.checkpoint_path, fine_handle.checkpoint_path, config_path, split.path,
                 workspace.path("candidate_scorer_solarcrow10.pkl"), workspace.path("tau_sweep_validation.csv"),
                 workspace.path("self_eval_gt.csv"), workspace.path("self_eval_pred.csv"),
                 workspace.path("self_eval_overlap.pkl"), workspace.path("self_eval_summary.csv"),
                 submission_info["path"], manifest_path]:
        print(f"  {'OK ' if os.path.exists(path) else '-- '} {path} ({os.path.getsize(path) if os.path.exists(path) else 0} bytes)")
    print(f"\nVALIDATION PQ = {self_eval_pq:.4f} | τ = {manifest['selected_tau']} | SUBMISSION = {submission_info['path']}")

## Appendix — diagnostics *(optional, failure-isolated, no duplicate inference)*

Everything below runs **after** `submission.csv` has been written and verified, so a failure here can never cost the submission. All diagnostics re-use the cached coarse components and candidate batches of the held-out and test images; the standalone cell that used to re-run an older coarse-only pipeline over all 141 validation + 180 test images (and needed a previous run's artifacts on disk) is now a cache-backed comparison.

* **A. Coarse-only coverage** (A2) and **full-vs-coarse comparison** (+ separate `submission_coarse_only.csv`; the real `submission.csv` is never touched)
* **B. Coarse model diagnostic** on one training image, **Q4 audits** (rasterisation, disk support, crop support), **A5 failure autopsy**, **test preview**
* **C. Annotator-level PQ simulation**, **metric audit**, **artifact reload check**, **dense-vs-RLE IoU equivalence check**

In [ ]:
# ---------------------------------------------------------------- shared helpers
def coarse_only_batches(image_dir, names):
    """Each coarse component is one candidate; components come from the proxy cache (no new inference)."""
    return {name: CandidateBatch.from_components(name, settings.image_shape, pipeline.coarse_components(image_dir, name))
            for name in names}


def fresh_frame_builder():
    return PredictionFrameBuilder(OverlapResolver(settings.min_prediction_area), RleEncoder(settings.image_shape))


def draw_contours(ax, mask, **kwargs):
    contours, _ = cv2.findContours(mask.astype("uint8"), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        if len(contour) >= 3:
            contour = contour.reshape(-1, 2)
            ax.plot(contour[:, 0], contour[:, 1], linewidth=1, **kwargs)


# ---------------------------------------------------------------- A. coarse-only
@optional_stage("coarse-only coverage (A2)")
def coarse_only_coverage():
    builder = fresh_frame_builder()
    batches = coarse_only_batches(dataset_paths.train_images, validation_image_names)
    pred_df = builder.build(validation_image_names, batches, OrderPreservingPolicy())
    overlap_df = harness.tables.build(pred_df)
    pooled, _, _ = compute_detailed_metrics(overlap_df)
    print("Coarse-only metrics:"); display(pd.DataFrame([pooled]).round(4))

    total_gt = covered_any = covered_01 = covered_05 = 0
    for row in overlap_df.itertuples(index=False):
        total_gt += int(row.n_gt)
        if row.n_gt == 0 or row.n_pred == 0:
            continue
        iou = _as_numpy(row.iou_matrix)
        covered_any += int((iou > 0).any(axis=1).sum())          # GT covered by ANY coarse prediction
        covered_01 += int((iou >= 0.1).any(axis=1).sum())
        covered_05 += int((iou >= 0.5).any(axis=1).sum())
    pct = lambda n: 100 * n / max(total_gt, 1)
    print("=" * 60 + "\nCOARSE-STAGE GT INSTANCE COVERAGE\n" + "=" * 60)
    print(f"Total GT instances: {total_gt}")
    print(f"Any overlap: {covered_any}/{total_gt} ({pct(covered_any):.2f}%)")
    print(f"IoU >= 0.1: {covered_01}/{total_gt} ({pct(covered_01):.2f}%)")
    print(f"IoU >= 0.5: {covered_05}/{total_gt} ({pct(covered_05):.2f}%)")
    print(f"Completely missed: {total_gt - covered_any}/{total_gt} ({pct(total_gt - covered_any):.2f}%)")
    return overlap_df


def _by_image(df):
    grouped = defaultdict(list)
    for fid, rle in zip(df["filament_id"], df["segmentation_rle"]):
        grouped[str(fid).rsplit("_", 1)[0]].append(str(rle))
    return grouped


def _rle(text):
    return {"size": list(settings.image_shape), "counts": str(text).encode("utf-8")}


def compare_full_and_coarse(full_df, coarse_df, image_names, label):
    full_by, coarse_by = _by_image(full_df), _by_image(coarse_df)
    rows = []
    for image_name in image_names:
        stem = os.path.splitext(os.path.basename(image_name))[0]
        full_rles, coarse_rles = full_by.get(stem, []), coarse_by.get(stem, [])
        best = np.nan
        if full_rles and coarse_rles:
            ious = mask_util.iou([_rle(r) for r in full_rles], [_rle(r) for r in coarse_rles], [0] * len(coarse_rles))
            best = float(np.asarray(ious).max(axis=1).mean())
        rows.append({"image": stem, "n_full": len(full_rles), "n_coarse": len(coarse_rles), "best_iou": best,
                     "area_full": float(sum(np.asarray(mask_util.area(_rle(r))).item() for r in full_rles)),
                     "area_coarse": float(sum(np.asarray(mask_util.area(_rle(r))).item() for r in coarse_rles)),
                     "rle_ordered_equal": full_rles == coarse_rles})
    result = pd.DataFrame(rows)
    area_c = result["area_coarse"].sum()
    summary = {"dataset": label, "images": len(result), "full_rows": int(result["n_full"].sum()),
               "coarse_rows": int(result["n_coarse"].sum()), "full_masks_per_image": float(result["n_full"].mean()),
               "coarse_masks_per_image": float(result["n_coarse"].mean()),
               "area_ratio_full_over_coarse": float(result["area_full"].sum() / area_c) if area_c > 0 else np.nan,
               "mean_best_iou_full_to_coarse": float(result["best_iou"].mean()),
               "same_count_and_best_iou_gt_095": float(((result["n_full"] == result["n_coarse"]) & (result["best_iou"] > 0.95)).mean()),
               "exact_ordered_rle_equal_fraction": float(result["rle_ordered_equal"].mean()),
               "full_zero_prediction_images": int((result["n_full"] == 0).sum()),
               "coarse_zero_prediction_images": int((result["n_coarse"] == 0).sum())}
    print(f"\n{'=' * 20} {label.upper()} {'=' * 20}")
    for key, value in summary.items():
        if key != "dataset":
            print(f"{key:42s}: {value:.6f}" if isinstance(value, (float, np.floating)) else f"{key:42s}: {value}")
    print("\n10 images with the lowest full-to-coarse best IoU:")
    display(result.sort_values("best_iou", na_position="first").head(10).reset_index(drop=True))
    return result, summary


def audit_prediction_csv(df, image_names, label):
    expected = {os.path.splitext(os.path.basename(n))[0] for n in image_names}
    frame = df.copy()
    frame["_stem"] = _stem_of(frame["filament_id"])
    unknown, represented = sorted(set(frame["_stem"]) - expected), set(frame["_stem"])
    errors, empty_masks, overlapping, decoded_count = [], 0, set(), 0
    for stem, group in frame.groupby("_stem", sort=False):
        occupied = np.zeros(settings.image_shape, dtype=bool)
        for row in group.itertuples(index=False):
            try:
                decoded = mask_util.decode(_rle(row.segmentation_rle))
                decoded_count += 1
                if decoded.shape != tuple(settings.image_shape):
                    errors.append(f"{row.filament_id}: shape={decoded.shape}"); continue
                mask = decoded.astype(bool)
                empty_masks += int(not mask.any())
                if np.logical_and(occupied, mask).any():
                    overlapping.add(stem)
                occupied |= mask
            except Exception as exc:  # noqa: BLE001
                errors.append(f"{row.filament_id}: {type(exc).__name__}: {exc}")
    audit = {"label": label, "rows": len(df), "unique_images": len(represented), "expected_images": len(expected),
             "zero_prediction_images": len(expected - represented), "duplicate_ids": int(frame["filament_id"].duplicated().sum()),
             "missing_fields": int(frame["filament_id"].isna().sum() + frame["segmentation_rle"].isna().sum()),
             "unknown_image_stems": len(unknown), "decoded_masks": decoded_count, "empty_masks": empty_masks,
             "overlap_images": len(overlapping), "decode_errors": len(errors)}
    print(f"\n{'=' * 16} {label} CSV AUDIT {'=' * 16}")
    for key, value in audit.items():
        if key != "label":
            print(f"{key}: {value}")
    return audit


@optional_stage("full-vs-coarse comparison")
def full_vs_coarse_comparison():
    builder = fresh_frame_builder()
    coarse_val = builder.build(validation_image_names, coarse_only_batches(dataset_paths.train_images, validation_image_names), OrderPreservingPolicy())
    coarse_test = builder.build(test_image_names, coarse_only_batches(dataset_paths.test_images, test_image_names), OrderPreservingPolicy())
    coarse_csv = workspace.write_csv(coarse_test, "submission_coarse_only.csv")
    print("Saved separate coarse-only CSV:", coarse_csv, "| the real submission.csv is untouched.")

    full_test = pd.read_csv(submission_info["path"], dtype=str)
    val_cmp, val_summary = compare_full_and_coarse(self_eval_pred_df, coarse_val, validation_image_names, "validation")
    test_cmp, test_summary = compare_full_and_coarse(full_test, coarse_test, test_image_names, "test")
    print(f"\nImage counts — validation: {len(validation_image_names)} | test: {len(test_image_names)}")
    audits = [audit_prediction_csv(full_test, test_image_names, "FULL PIPELINE"),
              audit_prediction_csv(coarse_test, test_image_names, "COARSE-ONLY")]
    summary = pd.DataFrame([{**val_summary, "csv_sha256": submission_info["sha256"], "checkpoint_sha256": coarse_handle.sha256},
                            {**test_summary, "csv_sha256": submission_info["sha256"], "checkpoint_sha256": coarse_handle.sha256}])
    workspace.write_csv(summary, "diagnostic_summary.csv")
    print("\nCOMPACT SUMMARY"); display(summary)
    print("\nManual provenance still needed from Kaggle: leaderboard score per submission, notebook version, "
          "and which submission file each score corresponds to.")
    return summary

In [ ]:
# ---------------------------------------------------------------- B. model / data diagnostics
@optional_stage("coarse model diagnostic")
def coarse_model_diagnostic(sample_idx=0):
    dataset = CoarseSegmentationDataset(index, split.train_split_dict, dataset_paths.train_images, settings.coarse_resize, is_train=False)
    image, true_mask = dataset[sample_idx]
    image_id = dataset.image_ids[sample_idx]
    image_name = index.images_by_id[image_id]["file_name"]
    print("=" * 70 + "\nBASIC INFORMATION\n" + "=" * 70)
    print("Image ID:", image_id, "| name:", image_name, "| shapes:", tuple(image.shape), tuple(true_mask.shape))
    print("Image min/max:", image.min().item(), image.max().item(), "| unique mask values:", torch.unique(true_mask).tolist())

    with torch.inference_mode():
        logits = coarse_handle.model(image.unsqueeze(0).to(device))
        probabilities = torch.sigmoid(logits)
    prob, gt = probabilities[0, 0].cpu().numpy(), true_mask[0].numpy()
    print("\n" + "=" * 70 + "\nMODEL OUTPUT (single view, no TTA)\n" + "=" * 70)
    print("Logits min/max:", logits.min().item(), logits.max().item())
    print("Probability min/max/mean/median/std:", probabilities.min().item(), probabilities.max().item(),
          probabilities.mean().item(), probabilities.median().item(), probabilities.std().item())

    gt_binary = gt > 0.5
    print("\n" + "=" * 70 + "\nGROUND TRUTH\n" + "=" * 70)
    print("GT positive pixels:", int(gt_binary.sum()), "| fraction:", float(gt_binary.mean()))
    print("\n" + "=" * 70 + "\nPREDICTION AT DIFFERENT THRESHOLDS\n" + "=" * 70)
    for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        pred = prob > t
        print(f"Threshold {t:.1f}: {int(pred.sum()):7d} pixels ({pred.mean() * 100:.4f}%)")
    if gt_binary.any():
        print("\nInside GT  : mean", prob[gt_binary].mean(), "max", prob[gt_binary].max(), "median", np.median(prob[gt_binary]))
        print("Outside GT : mean", prob[~gt_binary].mean(), "max", prob[~gt_binary].max(), "median", np.median(prob[~gt_binary]))

    pred_binary = (prob > 0.5).astype(np.uint8)
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(pred_binary, connectivity=8)
    comps = sorted(((int(stats[i, 4]), int(stats[i, 0]), int(stats[i, 1]), int(stats[i, 2]), int(stats[i, 3])) for i in range(1, num_labels)), reverse=True)
    print("\n" + "=" * 70 + f"\nCONNECTED COMPONENTS @0.5: {len(comps)}\n" + "=" * 70)
    for i, (area, x, y, w, h) in enumerate(comps[:20]):
        print(f"{i + 1:2d}: area={area:6d}, bbox=({x},{y},{w},{h})")

    print("\n" + "=" * 70 + "\nPER-FILAMENT DETECTION\n" + "=" * 70)
    h0, w0 = plt.imread(os.path.join(dataset_paths.train_images, image_name)).shape[:2]
    for i, annotation_id in enumerate(split.train_split_dict[image_id]):
        resized = cv2.resize(polygon_to_mask(index.annotations_by_id[annotation_id], h0, w0), (settings.coarse_resize,) * 2, interpolation=cv2.INTER_NEAREST) > 0
        if resized.sum() == 0:
            print(f"Filament {i + 1}: DISAPPEARED DURING RESIZE"); continue
        fp = prob[resized]
        print(f"Filament {i + 1}: area={int(resized.sum()):6d}, mean_prob={fp.mean():.4f}, max_prob={fp.max():.4f}, "
              f"median_prob={np.median(fp):.4f}, >0.5={(fp > 0.5).mean() * 100:.2f}%")

    inter, union = np.logical_and(pred_binary, gt_binary).sum(), np.logical_or(pred_binary, gt_binary).sum()
    ps, gs = pred_binary.sum(), gt_binary.sum()
    print("\n" + "=" * 70 + "\nCOARSE SEGMENTATION METRICS (@0.5)\n" + "=" * 70)
    print("IoU:", inter / union if union else 0, "| Dice:", 2 * inter / (ps + gs) if ps + gs else 0,
          "| Precision:", inter / ps if ps else 0, "| Recall:", inter / gs if gs else 0)

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, (img, title) in zip(axes.flat, [(image[0], "Input"), (gt, "Ground Truth"), (prob, "Raw Probability"),
                                            (prob > 0.1, "Threshold 0.1"), (prob > 0.3, "Threshold 0.3"), (prob > 0.5, "Threshold 0.5")]):
        ax.imshow(img, cmap="gray"); ax.set_title(title); ax.axis("off")
    plt.tight_layout(); show_figure(fig)

    fig = plt.figure(figsize=(8, 8))
    plt.imshow(image[0], cmap="gray")
    draw_contours(plt.gca(), gt_binary); draw_contours(plt.gca(), pred_binary)
    plt.title("GT + Prediction Contours"); plt.axis("off"); show_figure(fig)


@optional_stage("Q4 rasterisation audit")
def audit_rasterization():
    """Production OpenCV masks (first polygon) vs full-polygon pycocotools masks."""
    rows = []
    for ann in index.raw["annotations"][:settings.q4_raster_annotation_sample]:
        info = index.images_by_id[ann["image_id"]]
        h, w = int(info.get("height", settings.image_height)), int(info.get("width", settings.image_width))
        cv_mask = polygon_to_mask(ann, h, w) > 0
        coco_mask = mask_util.decode(mask_util.merge(mask_util.frPyObjects(ann["segmentation"], h, w))).astype(bool)
        if coco_mask.ndim == 3:
            coco_mask = np.any(coco_mask, axis=2)
        inter, union = int(np.logical_and(cv_mask, coco_mask).sum()), int(np.logical_or(cv_mask, coco_mask).sum())
        rows.append({"annotation_id": ann["id"], "image_id": ann["image_id"], "polygon_count": len(ann.get("segmentation", [])),
                     "is_multipolygon": len(ann.get("segmentation", [])) > 1, "opencv_area": int(cv_mask.sum()),
                     "pycocotools_area": int(coco_mask.sum()), "area_abs_difference": abs(int(cv_mask.sum()) - int(coco_mask.sum())),
                     "different_pixels": int(np.logical_xor(cv_mask, coco_mask).sum()), "IoU": inter / union if union else 1.0})
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    print("Rasterization audit (diagnostic only):", {
        "annotations_checked": len(result), "multipolygon_records": int(result["is_multipolygon"].sum()),
        "median_IoU": float(result["IoU"].median()), "p05_IoU": float(result["IoU"].quantile(0.05)),
        "records_with_area_difference": int((result["area_abs_difference"] > 0).sum()),
        "records_with_different_pixels": int((result["different_pixels"] > 0).sum())})
    display(result.sort_values(["IoU", "different_pixels"]).head(15))
    return result


@optional_stage("Q4 disk-support audit")
def audit_disk_support():
    rows, seen = [], set()
    for info in index.raw["images"]:
        stem = info["id"].split("-", 1)[1]
        if stem in seen:
            continue
        seen.add(stem)
        image = plt.imread(os.path.join(dataset_paths.train_images, info["file_name"]))
        threshold, binary = cv2.threshold(_otsu_input(image), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        otsu_support, disk = binary > 0, get_disk_mask(image).astype(bool)
        outside = int(np.logical_and(otsu_support, ~disk).sum())
        rows.append({"physical_image": stem, "filename": info["file_name"], "otsu_threshold": float(threshold),
                     "otsu_positive_fraction": float(otsu_support.mean()), "disk_fraction": float(disk.mean()),
                     "disk_is_empty": not bool(disk.any()), "otsu_pixels_outside_selected_disk": outside,
                     "otsu_support_outside_disk_fraction": outside / max(int(otsu_support.sum()), 1)})
        if len(rows) >= settings.q4_disk_image_sample:
            break
    result = pd.DataFrame(rows)
    print("Disk-support audit (large outside-support fractions merit visual review):")
    display(result.sort_values("otsu_support_outside_disk_fraction", ascending=False).head(15))
    print("Empty/implausibly small or large disk count:",
          int((result["disk_is_empty"] | (result["disk_fraction"] < 0.2) | (result["disk_fraction"] > 0.98)).sum()))
    return result


@optional_stage("Q4 fine-support audit")
def audit_fine_support():
    """Retained crop support per GT instance on validation images (uses cached coarse components)."""
    rows = []
    for file_id in split.validation_file_ids[:settings.q4_support_image_sample]:
        record_ids = index.file_to_image_ids[file_id]
        name = index.file_name_of_physical(file_id)
        prepared = load_prepared(dataset_paths.train_images, name)
        h_img, w_img = prepared.shape
        gt = [(rid, aid, polygon_to_mask(index.annotations_by_id[aid], h_img, w_img) > 0)
              for rid in record_ids for aid in split.validation_split_dict.get(rid, [])]
        components = pipeline.coarse_components(dataset_paths.train_images, name)
        union_support = np.zeros((h_img, w_img), dtype=bool)
        per_gt_max, oversized = {aid: 0.0 for _, aid, _ in gt}, 0
        for component in components:
            x, y, bw, bh = component.bbox
            req_w = min(w_img, x + bw + settings.expansion) - max(0, x - settings.expansion)
            req_h = min(h_img, y + bh + settings.expansion) - max(0, y - settings.expansion)
            bucket_w, bucket_h = bucket_scheme.choose(req_w, req_h)
            oversized += int(req_w > bucket_w or req_h > bucket_h)
            fine_input = fine_input_builder.build(prepared.image, component)
            x0, y0, x1, y1 = fine_input.crop_coords
            oy, ox = fine_input.offsets
            support = fine_input.region_canvas[oy:oy + (y1 - y0), ox:ox + (x1 - x0)] > 0
            union_support[y0:y1, x0:x1] |= support
            for _, aid, gt_mask in gt:
                area = int(gt_mask.sum())
                if area:
                    per_gt_max[aid] = max(per_gt_max[aid], int(np.logical_and(gt_mask[y0:y1, x0:x1], support).sum()) / area)
        for rid, aid, gt_mask in gt:
            area = int(gt_mask.sum())
            if area:
                rows.append({"physical_image": file_id, "annotator_image": rid, "annotation_id": aid, "gt_area_pixels": area,
                             "max_single_crop_support_fraction": per_gt_max[aid],
                             "union_of_crop_support_fraction": int(np.logical_and(gt_mask, union_support).sum()) / area,
                             "coarse_component_count": len(components), "oversized_component_crops": oversized})
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    print("Fine support audit (diagnostic bound, not an attainable PQ oracle):", {
        "GT_instances": len(result),
        "median_max_single_crop_support": float(result["max_single_crop_support_fraction"].median()),
        "fraction_with_under_50pct_single_crop_support": float((result["max_single_crop_support_fraction"] < 0.5).mean()),
        "median_union_crop_support": float(result["union_of_crop_support_fraction"].median()),
        "rows_with_oversized_component_crop": int((result["oversized_component_crops"] > 0).sum())})
    display(result.sort_values("max_single_crop_support_fraction").head(20))
    return result


KNOWN_WORST_STEMS = ["20170323041730Lh", "20170310173450Bh", "20180318020350Uh", "20190412025430Lh", "20210810121630Ch"]


@optional_stage("A5 failure autopsy")
def failure_autopsy():
    """Worst (legacy-known + current lowest-PQ) and random validation images: GT | coarse boxes | final output."""
    current_worst = [] if validation_per_image_df is None else list(validation_per_image_df["physical_image"].head(settings.autopsy_max_worst))
    worst = [s for s in dict.fromkeys(KNOWN_WORST_STEMS + current_worst) if s in index.file_to_image_ids][:settings.autopsy_max_worst + 3]
    rng = random.Random(settings.seed)
    pool = [s for s in split.validation_file_ids if s not in worst]
    targets = [(s, "Worst case") for s in worst] + [(s, "Representative sample") for s in rng.sample(pool, min(settings.autopsy_random_samples, len(pool)))]
    if not targets:
        print("No images available for the autopsy."); return
    fig, axes = plt.subplots(len(targets), 3, figsize=(18, 6 * len(targets)), squeeze=False)
    for row, (stem, category) in enumerate(targets):
        name = index.file_name_of_physical(stem)
        prepared = load_prepared(dataset_paths.train_images, name)
        norm_img = prepared.image
        batch = validation_batches.get(name) or pipeline.candidates(dataset_paths.train_images, name)
        final_preds = pipeline.predict_masks(dataset_paths.train_images, name, selection_policy, batch=batch)
        coarse_comps = pipeline.coarse_components(dataset_paths.train_images, name)
        polys = []
        for rid in index.file_to_image_ids.get(stem, []):
            for aid in split.train_split_dict.get(rid, []) + split.validation_split_dict.get(rid, []):
                pts = index.annotations_by_id[aid]["segmentation"][0]
                polys.append(list(zip(pts[::2], pts[1::2])))
        axes[row, 0].imshow(norm_img, cmap="gray")
        for poly in polys:
            axes[row, 0].add_patch(Polygon(poly, closed=True, fill=False, edgecolor="lime", linewidth=1.5))
        axes[row, 0].set_title(f"[{category}]\nGT (lime) — {stem}")
        axes[row, 1].imshow(norm_img, cmap="gray")
        for comp in coarse_comps:
            bx, by, bw, bh = comp.bbox
            axes[row, 1].add_patch(patches.Rectangle((bx, by), bw, bh, fill=False, edgecolor="red", linewidth=1.2))
        axes[row, 1].set_title(f"Coarse proposals: {len(coarse_comps)} boxes (red)")
        axes[row, 2].imshow(norm_img, cmap="gray")
        overlay = np.zeros((*norm_img.shape, 3), dtype=np.float32)
        for mask in final_preds:
            overlay[mask > 0] = [0, 1, 1]
        axes[row, 2].imshow(overlay, alpha=0.45)
        for poly in polys:
            axes[row, 2].add_patch(Polygon(poly, closed=True, fill=False, edgecolor="lime", linewidth=1.2, linestyle="--"))
        axes[row, 2].set_title(f"Emitted: {len(final_preds)} predictions (cyan) vs GT (dashed lime)")
        for ax in axes[row]:
            ax.axis("off")
    plt.tight_layout(); show_figure(fig)


@optional_stage("test preview")
def test_preview():
    for name in test_image_names[:settings.preview_test_images]:
        batch = test_batches[name]
        masks = pipeline.predict_masks(dataset_paths.test_images, name, selection_policy, batch=batch)
        prepared = load_prepared(dataset_paths.test_images, name)
        fig = plt.figure(figsize=(8, 8))
        plt.imshow(prepared.image, cmap="gray")
        for mask in masks:
            draw_contours(plt.gca(), mask)
        plt.title(f"{name} — {len(masks)} predictions"); plt.axis("off")
        print(name, "predicted filaments:", len(masks))
        show_figure(fig)


# ---------------------------------------------------------------- C. metric / artifact diagnostics
@optional_stage("annotator-level PQ")
def annotator_level_pq():
    rows = []
    for r in self_eval_overlap_df.itertuples(index=False):
        annotator, img = r.annotator_image.split("-", 1)
        matches = threshold_aware_hungarian(_as_numpy(r.iou_matrix), 0.5) if r.n_gt and r.n_pred else []
        rows.append({"annotator": annotator, "img": img, "s": sum(m[2] for m in matches), "tp": len(matches),
                     "fp": r.n_pred - len(matches), "fn": r.n_gt - len(matches)})
    d = pd.DataFrame(rows)

    def pq(g):
        den = g["tp"].sum() + 0.5 * g["fp"].sum() + 0.5 * g["fn"].sum()
        return g["s"].sum() / den if den else 0.0

    print("\n=== MODEL PQ BY ANNOTATOR ===")
    print(pd.Series({a: pq(g) for a, g in d.groupby("annotator")}).sort_values().to_string())
    rng = np.random.default_rng(0)
    sims = [pq(d.groupby("img", group_keys=False).sample(n=1, random_state=int(rng.integers(0, 2 ** 32 - 1)))) for _ in range(200)]
    print("\n=== ONE RANDOM ANNOTATOR PER PHYSICAL IMAGE ===")
    print("Mean PQ:", round(float(np.mean(sims)), 4), "| 95% simulation interval:", np.round(np.percentile(sims, [2.5, 97.5]), 4))
    print("Physical images:", d["img"].nunique(), "| Annotators:", d["annotator"].nunique(), "| Annotator-image records:", len(d))


@optional_stage("metric audit")
def metric_audit():
    def match(matrix, inclusive):
        if matrix.size == 0:
            return []
        eligible = matrix >= 0.5 if inclusive else matrix > 0.5
        weights = np.where(eligible, min(matrix.shape) + 1.0 + matrix, 0.0)
        rows_, cols_ = linear_sum_assignment(weights, maximize=True)
        return [(int(r), int(c), float(matrix[r, c])) for r, c in zip(rows_, cols_) if eligible[r, c]]

    pq_of = lambda s, tp, fp, fn: s / (tp + 0.5 * fp + 0.5 * fn) if (tp + 0.5 * fp + 0.5 * fn) else 0.0
    acc = {k: {"s": 0.0, "tp": 0, "fp": 0, "fn": 0} for k in ("strict", "inclusive", "all_hits")}
    per_image, multirow, multicol, extra = [], 0, 0, 0
    for row in self_eval_overlap_df.itertuples(index=False):
        m, n_gt, n_pred = _as_numpy(row.iou_matrix).astype(float), int(row.n_gt), int(row.n_pred)
        for key, inclusive in (("strict", False), ("inclusive", True)):
            matches = match(m, inclusive)
            acc[key]["s"] += sum(x[2] for x in matches); acc[key]["tp"] += len(matches)
            acc[key]["fp"] += n_pred - len(matches); acc[key]["fn"] += n_gt - len(matches)
        hits = m > 0.5
        acc["all_hits"]["s"] += float(m[hits].sum()); acc["all_hits"]["tp"] += int(hits.sum())
        acc["all_hits"]["fp"] += int((~hits.any(axis=0)).sum()) if n_pred else 0
        acc["all_hits"]["fn"] += int((~hits.any(axis=1)).sum()) if n_gt else 0
        multirow += int((hits.sum(axis=1) > 1).sum()) if hits.size else 0
        multicol += int((hits.sum(axis=0) > 1).sum()) if hits.size else 0
        tp_strict = len(match(m, False))
        extra += int(hits.sum()) - tp_strict
        phys = row.annotator_image.split("-", 1)[1].split(".")[0]
        per_image.append({"physical_image": phys, "station": phys[-2:], "n_gt": n_gt, "n_pred": n_pred, "TP": tp_strict,
                          "FP": n_pred - tp_strict, "FN": n_gt - tp_strict, "sum_matched_iou": sum(x[2] for x in match(m, False))})
    summary = pd.DataFrame([{"scorer": name, "PQ": pq_of(v["s"], v["tp"], v["fp"], v["fn"]), "TP": v["tp"], "FP": v["fp"],
                             "FN": v["fn"], "sum_matched_iou": v["s"]}
                            for name, v in (("One-to-one, IoU > 0.5", acc["strict"]), ("One-to-one, IoU >= 0.5", acc["inclusive"]),
                                            ("All eligible pairs, IoU > 0.5", acc["all_hits"]))])
    print("=== METRIC AUDIT ==="); display(summary.round(6))
    print(f"GT instances with multiple eligible predictions: {multirow} | predictions with multiple eligible GT: {multicol} | "
          f"extra eligible pairs beyond one-to-one: {extra}")
    img = pd.DataFrame(per_image)
    report_df = img.groupby(["physical_image", "station"], as_index=False).agg(
        GT=("n_gt", "sum"), predictions=("n_pred", "sum"), TP=("TP", "sum"), FP=("FP", "sum"), FN=("FN", "sum"),
        sum_matched_iou=("sum_matched_iou", "sum"))
    report_df["PQ"] = report_df.apply(lambda r: pq_of(r["sum_matched_iou"], r["TP"], r["FP"], r["FN"]), axis=1)
    report_df = report_df.sort_values("PQ")
    workspace.write_csv(summary, "metric_audit_summary.csv")
    workspace.write_csv(report_df, "metric_audit_per_image.csv")
    print("\n=== 10 WORST PHYSICAL IMAGES ==="); display(report_df.head(10).round(4))


@optional_stage("artifact reload check")
def artifact_reload_check():
    overlap = pd.read_pickle(workspace.path("self_eval_overlap.pkl"))
    gt, pred = pd.read_csv(workspace.path("self_eval_gt.csv")), pd.read_csv(workspace.path("self_eval_pred.csv"))
    summary = pd.read_csv(workspace.path("self_eval_summary.csv"))
    print("=== Reloaded validation artifacts ===")
    print("Overlap table:", overlap.shape, "| Ground truth:", gt.shape, "| Predictions:", pred.shape)
    display(summary)
    if len(overlap):
        row = overlap.iloc[0]
        print("Example overlap record:", row["annotator_image"], "| GT:", row["n_gt"], "| pred:", row["n_pred"], "| IoU matrix:", tuple(row["iou_matrix"].shape))
    assert abs(get_pq_score(overlap) - self_eval_pq) < 1e-9, "reloaded overlap table disagrees with the in-memory PQ"


def rles_to_layers(rles, shape):
    """Reference implementation (dense float32 layers) — used ONLY by the equivalence check below."""
    if not rles:
        return np.zeros((0, *shape), dtype=np.float32)
    masks = mask_util.decode([{"size": list(shape), "counts": str(r).encode("utf-8")} for r in rles])
    return (masks[:, :, None] if masks.ndim == 2 else masks).transpose(2, 0, 1).astype(np.float32)


@optional_stage("IoU backend equivalence")
def iou_backend_equivalence(max_masks=60):
    """Dense torch.matmul IoU (original method) vs RLE IoU (current method) on the smallest validation record."""
    gt_ids = _stem_of(harness.gt_df["filament_id"])
    pred_by_img = _stem_of(self_eval_pred_df["filament_id"])
    sizes = []
    for r in self_eval_overlap_df.itertuples(index=False):
        if r.n_gt and r.n_pred and r.n_gt + r.n_pred <= max_masks:
            sizes.append((r.n_gt + r.n_pred, r.annotator_image))
    if not sizes:
        print("No small-enough record for the dense reference; skipped."); return
    _, annotator_image = min(sizes)
    gt = rles_to_layers(harness.gt_df.loc[gt_ids == annotator_image, "segmentation_rle"].tolist(), settings.image_shape)
    pred = rles_to_layers(self_eval_pred_df.loc[pred_by_img == annotator_image.split("-", 1)[1], "segmentation_rle"].tolist(), settings.image_shape)
    g, p = torch.from_numpy(gt).reshape(len(gt), -1), torch.from_numpy(pred).reshape(len(pred), -1)
    inter = g @ p.T
    union = g.sum(1).view(-1, 1) + p.sum(1).view(1, -1) - inter
    dense = torch.where(union == 0, torch.zeros(()), inter / union)
    current = self_eval_overlap_df.set_index("annotator_image").loc[annotator_image, "iou_matrix"]
    diff = float((dense - current).abs().max())
    print(f"[IoU backend check] {annotator_image}: max |dense matmul − RLE| = {diff:.2e} ({len(gt)} GT × {len(pred)} pred)")
    assert diff < 1e-5, "RLE IoU backend disagrees with the dense reference"

In [ ]:
with run_log.stage("Appendix diagnostics"):
    if settings.run_diagnostics:
        coarse_only_overlap_df = coarse_only_coverage()
        full_vs_coarse_summary = full_vs_coarse_comparison()
        coarse_model_diagnostic()
        if settings.run_q4_diagnostics:
            rasterization_audit_df, disk_support_audit_df, fine_support_audit_df = audit_rasterization(), audit_disk_support(), audit_fine_support()
        failure_autopsy()
        test_preview()
        annotator_level_pq()
        metric_audit()
        artifact_reload_check()
        iou_backend_equivalence()
    else:
        print("Appendix diagnostics disabled (settings.run_diagnostics=False).")

    # Refresh the manifest with optional-stage outcomes (the submission was finalised long before this point).
    manifest.update({"optional_stage_failures": run_log.optional_failures,
                     "optional_stages_skipped_for_budget": run_log.optional_skipped,
                     "stage_seconds": run_log.stage_seconds, "elapsed_seconds": round(run_log.elapsed(), 1),
                     "coarse_stage_cache": coarse_segmenter.stats()})
    workspace.write_json(manifest, "run_manifest.json")
    print(f"\nTotal wall-clock: {run_log.elapsed() / 60:.1f} min | optional-stage failures: {run_log.optional_failures or 'none'}")

## Final handoff and testing checklist

### Implemented in this notebook
- SolarCrow 9 Factory/Template Method, Builder, Strategy, Proxy, Facade and Observer structure retained.
- Correct R34/B3 loading, fixed input geometry, measured eight-feature hurdle, four-view baseline and routed midpoint-median D4.
- Short validation recipe comparison, explicit chosen-recipe test handoff, immutable run directory, reusable scorer bundle and final-mask CSV audit.
- One-to-one metric consistency and empty annotation-record accounting.

### Prepared checks versus verified runtime
Synthetic metric, transform, ROI-shape, mask-conflict and RLE checks run in the notebook. Scorer save/reload prediction equivalence and serialized CSV audit run when their artifacts are created. **The authoring process did not execute the competition pipeline, load real weights or establish a new PQ/LB score.** Read this run's actual logs rather than treating historical results as current outputs.

### First Kaggle run
1. Attach competition data and the `solarcrow8d` weights/split; enable GPU.
2. Leave both neural modes `load`, scorer `auto`, hurdle target, fixed tau and validation shortlist enabled.
3. Save & Run All. Inspect checkpoint/split hashes, synthetic checks, candidate-bank summary and `final_recipe_comparison.csv`.
4. Confirm `selected_recipe` in the config equals the test recipe and `submission_integrity_audit.json` passes.
5. Submit **the `submission.csv` inside the printed new run directory**, not an older root-level file.
6. Save the entire run output. Attach it as input on later runs to reuse its verified scorer. No neural retraining is required for inference changes.

### Optional deliberate changes
`select_recipe_on_validation=False` deploys `final_recipe` directly. `enable_residual_challenger=False` keeps only N1/N4. Explicit `train`/`fine_tune` is required to update weights; the legacy trainer saves separately and may not match the campaign's optimized schedule. COCO scorer labels, other scorer targets, new checkpoints and changed preprocessing are new conditions, not guaranteed improvements.

Known limits: repeated validation selection; external weight membership not independently proved; inference precision/library differences; N4+rescue unmeasured until run; no official-evaluator parity or guaranteed leaderboard gain. All old artifacts remain intact.